# Medallion Extraction Pipeline

Colab-first paper ETL: **Bronze** captures the PDF with Docling, **Silver** cleans it, gates out non-data assets and normalises what survives, **Gold** assembles validated records, and a local SQLite mirrors the production schema. Gold's one model call goes to a local Ollama server.

In [ ]:
import gc
import hashlib
import json
import math
import os
import queue
import random
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import unicodedata
import urllib.error
import urllib.request
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, as_completed, wait
from contextlib import contextmanager
from dataclasses import dataclass, field
from functools import cached_property, lru_cache
from itertools import chain, zip_longest
from pathlib import Path
from typing import Any, NamedTuple, Optional, Sequence


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


def env_text(name, default):
    return os.getenv(name, default)


def env_int(name, default):
    return int(os.getenv(name, default))


def env_float(name, default):
    return float(os.getenv(name, default))


def env_flag(name, default="1"):
    return os.getenv(name, default) == "1"


os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    CUDA_VERSION = torch.version.cuda
except ImportError:
    torch = None
    CUDA_AVAILABLE = False
    CUDA_VERSION = None

DEVICE = "gpu" if CUDA_AVAILABLE else "cpu"

def free_memory() -> None:
    """Collect first: `empty_cache()` only returns unreferenced blocks, and a failed
    forward pass is still holding its tensors through the traceback.

    Called at shard boundaries and on the out-of-memory recovery path only: a full
    generational collection after every document and every image chunk is measurable
    once the corpus is hundreds of papers rather than three.
    """
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()


print(f"CUDA Available : {CUDA_AVAILABLE} ({DEVICE})")
print(f"CUDA Version   : {CUDA_VERSION}")

PIP_INSTALL = [sys.executable, "-m", "pip", "install", "-q"]
run(PIP_INSTALL + ["docling", "pymupdf", "polars", "pandas", "numpy", "pillow",
                   "pydantic>=2", "sqlalchemy", "pyyaml", "tqdm"])
try:
    run(PIP_INSTALL + ["paddleocr", "paddlepaddle-gpu" if CUDA_AVAILABLE else "paddlepaddle"])
except subprocess.CalledProcessError:
    run(PIP_INSTALL + ["paddleocr", "paddlepaddle"])

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_ROOT = WORKDIR / "medallion_extraction"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
BRONZE_ROOT = ARTIFACT_ROOT / "bronze"
SILVER_ROOT = ARTIFACT_ROOT / "silver"
CACHE_ROOT = ARTIFACT_ROOT / "cache"
CHART_CACHE_ROOT = CACHE_ROOT / "chart"
GOLD_CACHE_ROOT = CACHE_ROOT / "gold"
RUN_ROOT = ARTIFACT_ROOT / "runs"
for _root in (PROJECT_ROOT, BRONZE_ROOT, SILVER_ROOT,
              CHART_CACHE_ROOT, GOLD_CACHE_ROOT, RUN_ROOT):
    _root.mkdir(parents=True, exist_ok=True)

LOCAL_DB_PATH = PROJECT_ROOT / "extraction_local.sqlite"
DATABASE_URL = env_text("DATABASE_URL", f"sqlite:///{LOCAL_DB_PATH}")

PIPELINE_MODULES = ("gold_schema.py", "vocabulary.py", "vocabulary.yaml")
PIPELINE_DIR = next((path for path in (Path.cwd(), *Path.cwd().parents, WORKDIR)
                     if all((path / name).exists() for name in PIPELINE_MODULES)), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError(
        f"{' and '.join(PIPELINE_MODULES)} are in neither {Path.cwd()} nor any parent of "
        "it. Copy them from misc/extraction_test/ next to this notebook.")
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

PAPER_INPUT_DIR = Path(env_text("PAPER_INPUT_DIR", str(PIPELINE_DIR / "content")))


OLLAMA_HOST = env_text("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = env_text("OLLAMA_MODEL", "deepseek-r1:14b")
OLLAMA_NUM_CTX = env_int("OLLAMA_NUM_CTX", "16384")
OLLAMA_NUM_PREDICT = env_int("OLLAMA_NUM_PREDICT", "0")
OLLAMA_PARALLEL_SLOTS = env_int("OLLAMA_PARALLEL_SLOTS", "2")


def env_keep_alive(name, default):
    """Ollama reads `keep_alive` as a duration: a JSON *number* is seconds, and negative
    means never unload; a JSON *string* has to carry a unit, so "10m" parses and "-1"
    does not — sending the bare string is a 400 on the whole request. Numeric text is
    therefore handed over as a number and everything else as it was written."""
    value = os.getenv(name, default).strip() or default
    try:
        return int(value)
    except ValueError:
        pass
    try:
        return float(value)
    except ValueError:
        return value


OLLAMA_KEEP_ALIVE = env_keep_alive("OLLAMA_KEEP_ALIVE", "-1")
OLLAMA_RETRIES = env_int("OLLAMA_RETRIES", "3")
OLLAMA_RETRY_BACKOFF = env_float("OLLAMA_RETRY_BACKOFF", "2.0")
OLLAMA_JSON_SCHEMA = env_flag("OLLAMA_JSON_SCHEMA", "0")


def env_think(name, default):
    """true | false for deepseek-r1, or low | medium | high for gpt-oss."""
    value = os.getenv(name, default).strip()
    return value.lower() == "true" if value.lower() in ("true", "false") else value


OLLAMA_THINK = env_think("OLLAMA_THINK", "true")
OLLAMA_TIMEOUT = env_int("OLLAMA_TIMEOUT", "600")
OLLAMA_PROBE_TIMEOUT = env_int("OLLAMA_PROBE_TIMEOUT", "10")


NATIVE_TEXT_THRESHOLD = env_int("NATIVE_TEXT_THRESHOLD", "150")
TABLE_STRUCTURE = env_flag("TABLE_STRUCTURE")
TABLE_STRUCTURE_MODE = env_text("TABLE_STRUCTURE_MODE", "fast")
IMAGES_SCALE = env_float("IMAGES_SCALE", "1.5")
DOCLING_THREADS = env_int("DOCLING_THREADS", str(min(8, os.cpu_count() or 4)))
DOCLING_PAGE_BATCH_SIZE = env_int("DOCLING_PAGE_BATCH_SIZE", "8")
DOCLING_DOC_BATCH_SIZE = env_int("DOCLING_DOC_BATCH_SIZE", "2")
DOCLING_DOC_BATCH_CONCURRENCY = env_int("DOCLING_DOC_BATCH_CONCURRENCY", "2")
DOCLING_PAGE_BATCH_CONCURRENCY = env_int("DOCLING_PAGE_BATCH_CONCURRENCY", "2")
DOCLING_ARTIFACTS_PATH = env_text("DOCLING_ARTIFACTS_PATH", "")


REQUIRE_FIGURE_DATA = env_flag("REQUIRE_FIGURE_DATA")
CHART_BATCH_SIZE = env_int("CHART_BATCH_SIZE", "8")
CHART_MAX_PIXELS = env_int("CHART_MAX_PIXELS", "1024")
CHART_MIN_PIXELS = env_int("CHART_MIN_PIXELS", "512")
CHART_MAX_NEW_TOKENS = env_int("CHART_MAX_NEW_TOKENS", "1024")
FIGURE_EDGE_FILTER = env_flag("FIGURE_EDGE_FILTER", "0")
FIGURE_CAPTION_FILTER = env_flag("FIGURE_CAPTION_FILTER", "0")


FIGURE_CONTEXT_CHARS = env_int("FIGURE_CONTEXT_CHARS", "1200")
MAX_PROMPT_CHARS = env_int("MAX_PROMPT_CHARS", "24000")


SHARD_SIZE = env_int("SHARD_SIZE", "16")
VISION_QUEUE_DEPTH = env_int("VISION_QUEUE_DEPTH", "4")
GOLD_WORKERS = env_int("GOLD_WORKERS", str(OLLAMA_PARALLEL_SLOTS))
RELEASE_MODELS_BETWEEN_STAGES = env_flag("RELEASE_MODELS_BETWEEN_STAGES", "0")
CACHE_ENABLED = env_flag("CACHE_ENABLED")
RESUME_COMPLETED = env_flag("RESUME_COMPLETED")

CHARS_PER_TOKEN = 3.5

GATE_VERSION = "gate-2"
CHART_MODEL_VERSION = "pp-chart2table-1"


def content_key(*parts) -> str:
    """The cache key every stage uses: what went in, hashed, so the answer is reusable
    exactly when the inputs and the code version behind it are the same."""
    digest = hashlib.sha256()
    for part in parts:
        digest.update(str(part).encode("utf-8"))
        digest.update(b"\x00")
    return digest.hexdigest()


STAGE_TIMINGS: dict = {}
_TIMINGS_LOCK = threading.Lock()


def record_metric(stage: str, paper: Optional[str] = None, **fields) -> None:
    with _TIMINGS_LOCK:
        STAGE_TIMINGS.setdefault(stage, []).append({"paper": paper, **fields})


@contextmanager
def stage_timer(stage: str, paper: Optional[str] = None, **fields):
    """Time one stage of one paper. The yielded dict is the body's channel for saying
    what it did — figures converted, tokens read — beside how long it took."""
    extra: dict = {}
    started = time.perf_counter()
    try:
        yield extra
    finally:
        record_metric(stage, paper,
                      **{"seconds": round(time.perf_counter() - started, 3),
                         **fields, **extra})


def reset_timings() -> None:
    with _TIMINGS_LOCK:
        STAGE_TIMINGS.clear()


def stage_timing_frame() -> pd.DataFrame:
    """Every recorded stage, one row each."""
    with _TIMINGS_LOCK:
        rows = [{"stage": stage, **record}
                for stage, records in STAGE_TIMINGS.items() for record in records]
    return pd.DataFrame(rows)


def stage_totals() -> pd.DataFrame:
    """Wall clock by stage, which is the table to read before changing anything."""
    frame = stage_timing_frame()
    if frame.empty or "seconds" not in frame:
        return pd.DataFrame(columns=["stage", "calls", "total_s", "mean_s", "share"])
    grouped = frame.groupby("stage")["seconds"].agg(["count", "sum", "mean"])
    grouped.columns = ["calls", "total_s", "mean_s"]
    grouped["share"] = grouped["total_s"] / grouped["total_s"].sum()
    return (grouped.reset_index()
            .sort_values("total_s", ascending=False, ignore_index=True)
            .round({"total_s": 2, "mean_s": 2, "share": 3}))


def show_stage_totals() -> pd.DataFrame:
    totals = stage_totals()
    display(totals)
    return totals


def save_timings(path) -> Path:
    """Write the run's timings so the next change can be diffed against them."""
    path = Path(path)
    with _TIMINGS_LOCK:
        path.write_text(json.dumps(STAGE_TIMINGS, indent=2, default=str), encoding="utf-8")
    print(f"Timings written to {path}")
    return path


def compare_timings(baseline_path) -> pd.DataFrame:
    """This run against a saved one, per stage. A change that does not move its target
    stage has not earned its complexity."""
    baseline = json.loads(Path(baseline_path).read_text(encoding="utf-8"))
    before = pd.DataFrame([{"stage": stage, "seconds": record.get("seconds", 0.0)}
                           for stage, records in baseline.items() for record in records])
    before = (before.groupby("stage")["seconds"].sum().rename("baseline_s")
              if not before.empty else pd.Series(dtype=float, name="baseline_s"))
    after = stage_totals().set_index("stage")["total_s"].rename("current_s")
    frame = pd.concat([before, after], axis=1).fillna(0.0)
    frame["delta_s"] = (frame["current_s"] - frame["baseline_s"]).round(2)
    frame["speedup"] = (frame["baseline_s"] / frame["current_s"].replace(0, np.nan)).round(2)
    return frame.reset_index(names="stage").sort_values("delta_s", ignore_index=True)


MIN_GENERATION_TOKENS = env_int("MIN_GENERATION_TOKENS", "6000")

_prompt_token_estimate = int(MAX_PROMPT_CHARS / CHARS_PER_TOKEN)
_generation_room = OLLAMA_NUM_CTX - _prompt_token_estimate
if _generation_room < MIN_GENERATION_TOKENS:
    print(f"WARNING: MAX_PROMPT_CHARS is ~{_prompt_token_estimate} tokens of a "
          f"{OLLAMA_NUM_CTX} window, leaving {_generation_room} for reasoning and the "
          f"answer — under the {MIN_GENERATION_TOKENS} a reasoning model needs to reach "
          "one. Lower MAX_PROMPT_CHARS or raise num_ctx.")
if 0 < OLLAMA_NUM_PREDICT < MIN_GENERATION_TOKENS:
    print(f"WARNING: OLLAMA_NUM_PREDICT={OLLAMA_NUM_PREDICT} bounds the reasoning trace "
          "as well as the answer. Below the trace length the reply comes back empty; "
          "size it from the eval_count values in STAGE_TIMINGS, or leave it at 0.")

print(f"Project root : {PROJECT_ROOT}")
print(f"Modules      : {PIPELINE_DIR}")
print(f"Papers in    : {PAPER_INPUT_DIR}")
print(f"Database     : {DATABASE_URL}")
print(f"Ollama       : {OLLAMA_MODEL} at {OLLAMA_HOST} "
      f"(num_ctx {OLLAMA_NUM_CTX}, num_predict "
      f"{OLLAMA_NUM_PREDICT or 'unset'}, think {OLLAMA_THINK}, "
      f"{GOLD_WORKERS} worker(s))")
print(f"Table struct : {TABLE_STRUCTURE} ({TABLE_STRUCTURE_MODE})")
print(f"Run shape    : shards of {SHARD_SIZE}, chart batch {CHART_BATCH_SIZE}, "
      f"cache {'on' if CACHE_ENABLED else 'off'}")
print("Server side  : export OLLAMA_FLASH_ATTENTION=1 OLLAMA_KV_CACHE_TYPE=q8_0 "
      f"OLLAMA_NUM_PARALLEL={OLLAMA_PARALLEL_SLOTS} OLLAMA_MAX_LOADED_MODELS=1 "
      "before `ollama serve`")

## Bronze

Raw Docling capture: markdown, one CSV per table and one PNG per figure, plus a manifest every later stage reads.

In [ ]:
from datetime import datetime, timezone

import fitz
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.settings import settings
from docling_core.types.doc import PictureItem, TableItem

try:
    from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
except ImportError as exc:
    raise ImportError(
        "docling >= 2.26 is required: AcceleratorOptions moved to "
        "docling.datamodel.accelerator_options. Importing the old location instead "
        "silently changes which pipeline options are honoured, so this stops here. "
        "Run: pip install -U 'docling>=2.26'"
    ) from exc


settings.perf.page_batch_size = DOCLING_PAGE_BATCH_SIZE
settings.perf.doc_batch_size = DOCLING_DOC_BATCH_SIZE
settings.perf.doc_batch_concurrency = DOCLING_DOC_BATCH_CONCURRENCY
settings.perf.page_batch_concurrency = DOCLING_PAGE_BATCH_CONCURRENCY

HASH_CHUNK_BYTES = 1024 * 1024


def file_hash(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(HASH_CHUNK_BYTES), b""):
            digest.update(chunk)
    return digest.hexdigest()


def pdf_looks_native(pdf_path: Path, sample_pages: int = 2) -> bool:
    try:
        with fitz.open(pdf_path) as document:
            sample = [document[index].get_text("text") or ""
                      for index in range(min(sample_pages, len(document)))]
    except (fitz.FileDataError, fitz.EmptyFileError, RuntimeError, OSError):
        return False
    return sum(len(page.strip()) for page in sample) >= NATIVE_TEXT_THRESHOLD


def _docling_options(*, native_pdf: bool, table_structure: bool):
    options = PdfPipelineOptions()
    options.do_ocr = not native_pdf
    options.do_table_structure = table_structure
    if table_structure:
        options.table_structure_options.mode = (
            TableFormerMode.ACCURATE if TABLE_STRUCTURE_MODE == "accurate" else TableFormerMode.FAST
        )
        options.table_structure_options.do_cell_matching = True
    options.generate_picture_images = True
    options.generate_page_images = False
    options.images_scale = IMAGES_SCALE
    if DOCLING_ARTIFACTS_PATH:
        options.artifacts_path = DOCLING_ARTIFACTS_PATH
    options.accelerator_options = AcceleratorOptions(
        device=AcceleratorDevice.CUDA if CUDA_AVAILABLE else AcceleratorDevice.CPU,
        num_threads=DOCLING_THREADS,
    )
    return options


def docling_flags(*, native_pdf: bool, table_structure: bool = TABLE_STRUCTURE) -> dict:
    """Everything about the conversion that changes its output. Hashed into the Bronze
    cache key, so a knob that only changes speed does not invalidate a stored manifest."""
    return {
        "native_pdf": native_pdf,
        "ocr_enabled": not native_pdf,
        "table_structure_enabled": table_structure,
        "table_structure_mode": TABLE_STRUCTURE_MODE if table_structure else None,
        "images_scale": IMAGES_SCALE,
    }


_CONVERTERS: dict = {}
_CONVERTER_LOCK = threading.Lock()


def get_docling_converter(*, native_pdf: bool, table_structure: bool = TABLE_STRUCTURE):
    """Memoized per configuration. A converter loads the layout and table models on
    construction and holds them for as long as it lives, so building one per run re-pays
    the load every run, and a corpus mixing native and scanned PDFs pays it twice."""
    key = (native_pdf, table_structure, TABLE_STRUCTURE_MODE, IMAGES_SCALE)
    with _CONVERTER_LOCK:
        converter = _CONVERTERS.get(key)
        if converter is None:
            with stage_timer("docling_model_load", native_pdf=native_pdf):
                converter = DocumentConverter(format_options={
                    InputFormat.PDF: PdfFormatOption(
                        pipeline_options=_docling_options(
                            native_pdf=native_pdf, table_structure=table_structure))
                })
            _CONVERTERS[key] = converter
        return converter


def release_docling_converters() -> None:
    """Only worth calling when the card cannot hold Docling and the LLM at once."""
    with _CONVERTER_LOCK:
        _CONVERTERS.clear()
    free_memory()


def caption_text(element: Any, document: Any) -> Optional[str]:
    """Resolve a caption to text. `element.captions` holds references, not strings."""
    getter = getattr(element, "caption_text", None)
    if callable(getter):
        try:
            text = getter(document)
        except (TypeError, AttributeError, KeyError):
            text = None
        if text and str(text).strip():
            return str(text).strip()

    parts = []
    for reference in getattr(element, "captions", None) or []:
        text = getattr(reference, "text", None)
        if text is None:
            try:
                text = getattr(reference.resolve(document), "text", None)
            except (AttributeError, KeyError, ValueError):
                text = None
        if text:
            parts.append(str(text))
    return " ".join(parts).strip() or None


def _page_number(element: Any) -> Optional[int]:
    provenance = getattr(element, "prov", None) or []
    if not provenance:
        return None
    try:
        return int(provenance[0].page_no)
    except (AttributeError, TypeError, ValueError):
        return None


CAPTURE_FAILURES = (ValueError, TypeError, KeyError, OSError)


def _export_table(element: Any, document: Any, path: Path) -> dict:
    try:
        try:
            frame = element.export_to_dataframe(document)
        except TypeError:
            frame = element.export_to_dataframe()
        frame.to_csv(path, index=False)
    except CAPTURE_FAILURES as exc:
        return {"csv_path": None, "error": f"{type(exc).__name__}: {exc}"}
    return {"csv_path": str(path), "row_count": int(frame.shape[0]),
            "col_count": int(frame.shape[1])}


def _export_figure(element: Any, document: Any, path: Path) -> dict:
    try:
        image = element.get_image(document)
        if image is None:
            return {"image_path": None, "error": "no image payload"}
        image.save(path)
    except CAPTURE_FAILURES as exc:
        return {"image_path": None, "error": f"{type(exc).__name__}: {exc}"}
    return {"image_path": str(path), "width": int(image.width), "height": int(image.height)}


def _heading_level(element: Any, tree_level: int) -> int:
    level = getattr(element, "level", None)
    try:
        return int(level)
    except (TypeError, ValueError):
        return int(tree_level)


def _capture_items(document: Any, tables_root: Path, figures_root: Path) -> tuple:
    tables, figures, texts = [], [], []
    heading = None

    for order, (element, tree_level) in enumerate(document.iterate_items()):
        base = {
            "order": order,
            "docling_item_ref": str(getattr(element, "self_ref", None) or f"#/unknown/{order}"),
            "page_number": _page_number(element),
        }
        if isinstance(element, TableItem):
            index = len(tables) + 1
            path = tables_root / f"table_{index:03d}.csv"
            tables.append({"table_index": index, **base,
                           "caption": caption_text(element, document),
                           **_export_table(element, document, path)})
        elif isinstance(element, PictureItem):
            index = len(figures) + 1
            path = figures_root / f"figure_{index:03d}.png"
            figures.append({"figure_index": index, **base,
                            "caption": caption_text(element, document),
                            **_export_figure(element, document, path)})
        else:
            text = str(getattr(element, "text", "") or "").strip()
            label = str(getattr(element, "label", "") or "").lower()
            is_heading = "heading" in label or "section_header" in label
            if is_heading:
                heading = text or heading
            if text:
                texts.append({**base, "heading": heading, "text": text, "label": label,
                              "is_heading": is_heading,
                              "level": _heading_level(element, tree_level) if is_heading else None})

    return tables, figures, texts


MANIFEST_NAME = "manifest.json"


def write_manifest(directory: Path, manifest: dict) -> None:
    """Written last and renamed into place: the manifest is the stage's completion
    marker, so a directory left behind by a crash is never mistaken for a cached one."""
    directory.mkdir(parents=True, exist_ok=True)
    temporary = directory / f".{MANIFEST_NAME}.{os.getpid()}.{threading.get_ident()}"
    temporary.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str),
                         encoding="utf-8")
    os.replace(temporary, directory / MANIFEST_NAME)


def read_manifest(directory: Path) -> Optional[dict]:
    path = Path(directory) / MANIFEST_NAME
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


def bronze_cache_key(pdf_hash: str, flags: dict) -> str:
    return content_key("bronze", pdf_hash, json.dumps(flags, sort_keys=True))


def _bronze_artifacts_present(manifest: dict) -> bool:
    """A manifest whose CSVs or PNGs were cleaned up is not a usable cache hit."""
    paths = [manifest.get("markdown_path")]
    paths += [item.get("csv_path") for item in manifest.get("tables", [])]
    paths += [item.get("image_path") for item in manifest.get("figures", [])]
    return all(Path(path).exists() for path in paths if path)


def load_cached_bronze(pdf_path: Path, cache_key: str) -> Optional[dict]:
    if not CACHE_ENABLED:
        return None
    manifest = read_manifest(BRONZE_ROOT / pdf_path.stem)
    if manifest is None or manifest.get("cache_key") != cache_key:
        return None
    return manifest if _bronze_artifacts_present(manifest) else None


def _bronze_manifest(pdf_path: Path, document: Any, flags: dict, pdf_hash: str,
                     cache_key: str, elapsed_seconds: float) -> dict:
    """Everything Bronze keeps from one converted document, written under its own slug."""
    slug = pdf_path.stem
    paper_root = BRONZE_ROOT / slug
    tables_root = paper_root / "tables"
    figures_root = paper_root / "figures"
    for directory in (tables_root, figures_root):
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)

    markdown_path = paper_root / f"{slug}.md"
    markdown_path.write_text(document.export_to_markdown(), encoding="utf-8")

    table_records, figure_records, text_records = _capture_items(
        document, tables_root, figures_root)

    manifest = {
        "paper_slug": slug,
        "source_pdf": str(pdf_path),
        "file_hash": pdf_hash,
        "cache_key": cache_key,
        "native_pdf": flags["native_pdf"],
        "docling_flags": flags,
        "page_count": len(getattr(document, "pages", {}) or {}),
        "markdown_path": str(markdown_path),
        "tables": table_records,
        "figures": figure_records,
        "texts": text_records,
        "elapsed_seconds": elapsed_seconds,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    write_manifest(paper_root, manifest)
    print(f"        {slug}: {len(table_records)} tables, {len(figure_records)} figures, "
          f"{len(text_records)} text items in {elapsed_seconds}s")
    return manifest


def _group_by_nativeness(pdf_paths: Sequence[Path]) -> list:
    """One group per pipeline configuration. Native papers first, so the common case runs
    without an OCR model ever being constructed."""
    groups: dict = {}
    for path in pdf_paths:
        groups.setdefault(pdf_looks_native(path), []).append(path)
    return sorted(groups.items(), reverse=True)


CONVERSION_KEPT = (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS)


def _result_source(result, by_name: dict) -> Optional[Path]:
    """The paper a `ConversionResult` came from, read off the result rather than off a
    parallel loop counter: a stream that skips, fails or reorders would otherwise file
    one paper's tables under another paper's name. Unattributable is None, never a
    guess — a mislabelled manifest is worse than a missing one."""
    name = getattr(getattr(result, "input", None), "file", None)
    return None if name is None else by_name.get(Path(str(name)).name)


def stream_bronze(pdf_paths: Sequence):
    """Yield one Bronze manifest per paper as Docling produces it, cached papers first.

    Streaming rather than materialising: at hundreds of papers the list of manifests is
    itself the memory wall, since each one carries every text item of its paper.
    """
    for native_pdf, paths in _group_by_nativeness([Path(path) for path in pdf_paths]):
        flags = docling_flags(native_pdf=native_pdf)
        hashes = {path: file_hash(path) for path in paths}
        keys = {path: bronze_cache_key(hashes[path], flags) for path in paths}

        pending = []
        for path in paths:
            cached = load_cached_bronze(path, keys[path])
            if cached is None:
                pending.append(path)
                continue
            record_metric("bronze_cached", path.stem, seconds=0.0)
            print(f"        {path.name}: Bronze reused from cache")
            yield cached

        if not pending:
            continue

        print(f"Bronze: {len(pending)} paper(s) | native_pdf={native_pdf} "
              f"| ocr={flags['ocr_enabled']} "
              f"| tables={flags['table_structure_enabled']}"
              + (f" | {len(paths) - len(pending)} cached" if len(paths) > len(pending) else ""))
        converter = get_docling_converter(native_pdf=native_pdf)
        by_name = {path.name: path for path in pending}
        remaining = list(pending)

        started = time.perf_counter()
        for result in converter.convert_all(pending, raises_on_error=False):
            elapsed = round(time.perf_counter() - started, 2)
            source = _result_source(result, by_name)
            if source in remaining:
                remaining.remove(source)
            if source is None:
                print(f"        a conversion result ({result.status}) could not be "
                      "attributed to a source file — skipped")
            elif result.status in CONVERSION_KEPT:
                record_metric("docling", source.stem, seconds=elapsed,
                              status=str(result.status))
                yield _bronze_manifest(source, result.document, flags,
                                       hashes[source], keys[source], elapsed)
            else:
                record_metric("docling_failed", source.stem, seconds=elapsed,
                              status=str(result.status))
                print(f"        {source.name} did not convert ({result.status}) — skipped")
            del result
            started = time.perf_counter()

        for missed in remaining:
            print(f"        {missed.name} was never returned by Docling — skipped")
            record_metric("docling_failed", missed.stem, seconds=0.0, status="not returned")

## The Schema Gate

One structural contract for tables and chart-converted figures alike: an asset passes only if it has an ordered axis, labels and numeric values. No keyword list, so it survives a change of corpus.

In [ ]:
MIN_AXIS_POINTS = 3
MIN_SERIES_POINTS = 3
MIN_NUMERIC_RATIO = 0.6
MIN_ENUMERATION_ROWS = 6
EPS = 1e-9

_SD_SPLIT = re.compile(r"\s*(?:±|\+/-|\+-)\s*")
_PAREN_TAIL = re.compile(r"\s*\([^)]*\)\s*$")
_LEAD_CMP = re.compile(r"^[<>≤≥~≈=]+\s*")
_TRAIL_MARK = re.compile(r"\s*(?:[*†‡§¶]+|[a-zA-Z]{1,3})$")
_THOUSANDS = re.compile(r"^[-+]?\d{1,3}(?:,\d{3})+$")
_DECIMAL_COMMA = re.compile(r"^[-+]?\d+,\d+$")


@lru_cache(maxsize=16384)
def _parse_number_text(value: str) -> Optional[float]:
    text = value.replace("−", "-").replace(" ", " ").strip()
    if not text:
        return None

    text = _SD_SPLIT.split(text, 1)[0].strip()
    text = _PAREN_TAIL.sub("", text).strip()
    text = _LEAD_CMP.sub("", text).strip()
    text = text.rstrip("%").strip()
    text = _TRAIL_MARK.sub("", text).strip()
    if not text:
        return None

    if _THOUSANDS.match(text):
        text = text.replace(",", "")
    elif _DECIMAL_COMMA.match(text):
        text = text.replace(",", ".")

    try:
        number = float(text)
    except ValueError:
        return None
    return number if math.isfinite(number) else None


def parse_number(value: Any) -> Optional[float]:
    """Parse a table cell to a float, or None."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        number = float(value)
        return number if math.isfinite(number) else None
    return _parse_number_text(str(value).strip())


_MEAN_NOTATION = re.compile(
    r"±|\+/-|\+-|\bmeans?\b|\baverages?\b|\bsd\b|\bsem\b|\bstdev\b"
    r"|\(\s*\d+(?:\.\d+)?\s*\)\s*$",
    re.IGNORECASE,
)


def declares_mean(text: Any) -> bool:
    """True when a cell, a header or a caption says its numbers are already averaged."""
    return bool(_MEAN_NOTATION.search(str(text or "")))


_HEADER_NUMBER = re.compile(r"[-+]?\d*\.?\d+")
_ARTIFACT_HEADER = re.compile(
    r"^(unnamed[:_ ]|column[_ ]?\d+$|level[_ ]?\d+$|_duplicated_|index$)", re.IGNORECASE
)


def parse_axis_label(name: Any) -> Optional[float]:
    """The ordinal position a column *name* denotes, if any."""
    text = str(name).strip().replace("−", "-")
    if not text or _ARTIFACT_HEADER.match(text):
        return None
    match = _HEADER_NUMBER.search(text)
    if not match:
        return None
    number = float(match.group())
    return number if math.isfinite(number) else None


_LABEL_PARTS = re.compile(r"\s*[,;|]\s*|\s+-\s+|\s+–\s+|\s+—\s+")


@lru_cache(maxsize=16384)
def _split_axis(text: str) -> tuple:
    parts = [part for part in _LABEL_PARTS.split(text) if part and part.strip()]
    if not parts:
        return None, text
    for index, part in enumerate(parts):
        position = parse_axis_label(part)
        if position is not None:
            residue = " ".join(other.strip() for at, other in enumerate(parts) if at != index)
            return position, residue.strip(" -_,")
    return None, text


def split_axis_from_label(label: Any) -> tuple:
    """"control, 6 days" -> (6.0, "control"). A label carrying no position comes back
    unchanged beside None, so a caller can tell "not keyed" from "keyed with no arm"."""
    return _split_axis(str(label or "").strip())


_GREEK_TO_LATIN = str.maketrans({
    "α": "alpha", "β": "beta", "γ": "gamma", "δ": "delta", "ε": "epsilon",
    "ζ": "zeta", "η": "eta", "θ": "theta", "ι": "iota", "κ": "kappa",
    "λ": "lambda", "μ": "mu", "ν": "nu", "ξ": "xi", "ο": "omicron",
    "π": "pi", "ρ": "rho", "ς": "sigma", "σ": "sigma", "τ": "tau",
    "υ": "upsilon", "φ": "phi", "χ": "chi", "ψ": "psi", "ω": "omega",
})
_NON_ALNUM = re.compile(r"[^a-z0-9]+")
_UNDERSCORES = re.compile(r"_+")


@lru_cache(maxsize=32768)
def _canonical_key(text: str) -> str:
    folded = unicodedata.normalize("NFKD", text).lower().translate(_GREEK_TO_LATIN)
    folded = folded.encode("ascii", "ignore").decode("ascii")
    return _UNDERSCORES.sub("_", _NON_ALNUM.sub("_", folded)).strip("_")


def canonical_key(text: Any) -> str:
    """The form every "same term?" comparison uses: ascii, lower, punctuation collapsed."""
    return _canonical_key(str(text or ""))


DUPLICATE_SUFFIX = "__dup"
_DUPLICATE_TAIL = re.compile(re.escape(DUPLICATE_SUFFIX) + r"\d+$")

_DOSE = re.compile(
    r"""^\s*
    (?P<lead>.*?)\s*
    (?P<amount>\d+(?:[.,]\d+)?)\s*
    (?P<unit>%|ppm|ppb|mg/(?:g|ml|kg|l)|g/(?:kg|l)|µg/g|ug/g|mm|mg|w/w|v/v|w/v)
    \s*(?P<trail>.*?)\s*$""",
    re.VERBOSE | re.IGNORECASE,
)


class Dose(NamedTuple):
    substance: Optional[str]
    amount: float
    unit: str


@lru_cache(maxsize=8192)
def _parse_dosed(text: str) -> Optional[Dose]:
    match = _DOSE.match(_DUPLICATE_TAIL.sub("", text))
    if not match:
        return None
    lead, digits = match.group("lead"), match.group("amount")
    while len(digits) > 1 and digits[0] == "0" and "." not in digits:
        lead, digits = lead + "0", digits[1:]
    substance = " ".join(part for part in (lead, match.group("trail")) if part).strip(" -_,")
    return Dose(substance or None, float(digits.replace(",", ".")), match.group("unit").lower())


def parse_dosed_label(label: Any) -> Optional[Dose]:
    """Split "0.5% Moringa Extract" into substance, amount and unit, or None."""
    return _parse_dosed(str(label or "").strip())


_TRAILING_UNIT = re.compile(r"^(?P<name>.*?)[\s_]*[\(\[\{](?P<unit>[^\)\]\}]{1,24})[\)\]\}]\s*$")

_STATS_TAIL = re.compile(
    r"\s*[\(\[\{][^\)\]\}]*"
    r"(?:±|\bmean\b|\bmedian\b|\bstdev\b|\bstd\b|\bsd\b|\bsem?\b|\bn\s*=)"
    r"[^\)\]\}]*[\)\]\}]\s*$",
    re.IGNORECASE,
)


def split_label_and_unit(label: Any) -> tuple:
    """Split "TVC (log CFU/g)" into ("TVC", "log CFU/g"). Unit is None if absent."""
    text = _DUPLICATE_TAIL.sub("", str(label or "").strip()).strip()
    while True:
        trimmed = _STATS_TAIL.sub("", text).strip()
        if trimmed == text or not trimmed:
            break
        text = trimmed
    if not text:
        return "", None
    match = _TRAILING_UNIT.match(text)
    if not match:
        return text, None
    name = match.group("name").strip(" _-")
    unit = match.group("unit").strip()
    if not name or not unit:
        return text, None
    return name, unit


@dataclass(slots=True)
class Observation:
    """One value at one axis point. `column_label` and `row_labels` stay apart because
    which one names the indicator is a semantic question the gate does not answer."""

    axis_value: float
    value: float
    column_label: Optional[str] = None
    row_labels: dict = field(default_factory=dict)
    reports_mean: bool = False


@dataclass
class SchemaFit:
    fits: bool
    reason: str
    orientation: Optional[str] = None
    axis_label: Optional[str] = None
    axis_values: list = field(default_factory=list)
    axis_confidence: float = 0.0
    axis_runs: int = 0
    value_columns: list = field(default_factory=list)
    label_columns: list = field(default_factory=list)
    observations: list = field(default_factory=list)
    columns: list = field(default_factory=list, repr=False)

    def summary(self) -> dict:
        return {
            "fits": self.fits,
            "reason": self.reason,
            "orientation": self.orientation,
            "axis_label": self.axis_label,
            "axis_points": self.axis_values,
            "axis_confidence": round(self.axis_confidence, 2),
            "axis_runs": self.axis_runs,
            "value_columns": self.value_columns,
            "label_columns": self.label_columns,
            "observation_count": len(self.observations),
        }


@dataclass
class _Column:
    """Never mutated after `_columns` builds it, so the statistics below can be cached:
    gating compares every column against every other."""

    position: int
    name: str
    raw: list
    numeric: list

    @cached_property
    def values(self) -> list:
        return [value for value in self.numeric if value is not None]

    @cached_property
    def entries(self) -> list:
        """The non-empty cells, stripped — the denominator of every ratio below."""
        return [text for text in (("" if cell is None else str(cell).strip())
                                  for cell in self.raw) if text]

    @cached_property
    def numeric_ratio(self) -> float:
        return len(self.values) / len(self.entries) if self.entries else 0.0

    @cached_property
    def is_numeric(self) -> bool:
        return self.numeric_ratio >= MIN_NUMERIC_RATIO and len(self.values) >= MIN_AXIS_POINTS

    @cached_property
    def names_an_entity_per_row(self) -> bool:
        """A different entry on every row, over enough rows to be no coincidence."""
        return (len(self.entries) >= MIN_ENUMERATION_ROWS
                and len(set(self.entries)) == len(self.entries))

    @cached_property
    def keyed(self) -> list:
        """Per row, the axis position its label carries and what is left of the label:
        `[(6.0, 'control'), (None, 'treatments'), ...]`, aligned with `raw`."""
        return [split_axis_from_label(cell) if cell is not None else (None, "")
                for cell in self.raw]

    @cached_property
    def keys_an_axis(self) -> bool:
        """Enough rows carry a position, over enough distinct positions, to be an axis."""
        positions = [position for position, _ in self.keyed if position is not None]
        return (len(positions) >= MIN_SERIES_POINTS
                and len(set(positions)) >= MIN_AXIS_POINTS)


def _columns(headers: Sequence[Any], rows: Sequence[Sequence[Any]]) -> list:
    transposed = list(zip_longest(*rows, fillvalue=None)) if rows else []
    result = []
    for index, name in enumerate(headers):
        raw = list(transposed[index]) if index < len(transposed) else [None] * len(rows)
        result.append(_Column(position=index, name=str(name), raw=raw,
                              numeric=[parse_number(cell) for cell in raw]))
    return result


def _reports_mean(column: _Column, row_index: int) -> bool:
    return declares_mean(column.raw[row_index]) or declares_mean(column.name)


def _row_labels(row: Sequence[Any], label_columns: Sequence[_Column]) -> dict:
    """This row's entry in each qualifier column."""
    labels = {}
    for column in label_columns:
        cell = row[column.position] if column.position < len(row) else None
        text = "" if cell is None else str(cell).strip()
        if text:
            labels[column.name] = text
    return labels


def _runs(numeric: Sequence[Optional[float]]) -> list:
    """Maximal non-decreasing runs; a drop starts a new one. A time axis either climbs
    once or resets per group, and both are visible without reading what it measures."""
    runs, current = [], []
    for value in numeric:
        if value is None:
            continue
        if current and value < current[-1] - EPS:
            runs.append(current)
            current = [value]
        else:
            current.append(value)
    if current:
        runs.append(current)
    return runs


def _enumerates_rows(column: _Column, columns: Sequence[_Column]) -> bool:
    """A column that is just the row's position — "No.", "#" — climbing perfectly while
    measuring nothing. Both conditions are needed: either alone catches a daily series."""
    positioned = [(index, value) for index, value in enumerate(column.numeric) if value is not None]
    if len(positioned) < MIN_ENUMERATION_ROWS:
        return False
    if len({value - index for index, value in positioned}) != 1:
        return False
    return any(other.names_an_entity_per_row for other in columns
               if other.position != column.position and not other.is_numeric)


def _axis_quality(column: _Column, columns: Sequence[_Column]) -> Optional[dict]:
    """Score a numeric column as an ordinal axis, or None if it cannot be one."""
    values = column.values
    if len(values) < MIN_AXIS_POINTS:
        return None
    if _enumerates_rows(column, columns):
        return None
    if parse_dosed_label(column.name):
        return None

    runs = _runs(column.numeric)
    if not runs or min(len(set(run)) for run in runs) < MIN_AXIS_POINTS:
        return None

    distinct = sorted(set(values))
    if len(distinct) < MIN_AXIS_POINTS:
        return None

    repeat_ratio = 1.0 - (len(distinct) / len(values))
    return {
        "runs": len(runs),
        "distinct": distinct,
        "repeat_ratio": repeat_ratio,
        "confidence": min(1.0, 0.4 + 0.3 * (len(runs) > 1) + 0.3 * repeat_ratio),
    }


def _fit_long(columns: list, rows: Sequence[Sequence[Any]],
              prefer_axis: Optional[str] = None) -> SchemaFit:
    numeric_columns = [column for column in columns if column.is_numeric]
    if not numeric_columns:
        return SchemaFit(fits=False, reason="no column holds 3 or more numeric values")

    candidates = []
    for column in numeric_columns:
        quality = _axis_quality(column, columns)
        if quality:
            candidates.append((column, quality))

    forced = None
    if prefer_axis:
        forced = next(
            (column for column in numeric_columns if canonical_key(column.name) == prefer_axis),
            None,
        )
    if forced is not None and not any(column is forced for column, _ in candidates):
        candidates.append((forced, {
            "runs": 0,
            "distinct": sorted(set(forced.values)),
            "repeat_ratio": 0.0,
            "confidence": 0.3,
        }))

    if not candidates:
        return SchemaFit(fits=False, reason="no column orders the rows like an axis")

    if forced is not None:
        column, quality = next(pair for pair in candidates if pair[0] is forced)
    else:
        column, quality = max(
            candidates,
            key=lambda pair: (
                pair[1]["runs"],
                pair[1]["repeat_ratio"] if pair[1]["runs"] > 1 else 0.0,
                -pair[0].position,
            ),
        )

    value_columns = [
        other
        for other in numeric_columns
        if other.position != column.position
        and len(other.values) >= MIN_SERIES_POINTS
        and len(set(other.values)) >= 2
    ]
    if not value_columns:
        return SchemaFit(
            fits=False,
            reason="an axis but no varying measured column alongside it",
            orientation="long",
            axis_label=column.name,
            axis_values=quality["distinct"],
        )

    label_columns = [other for other in columns if not other.is_numeric and other.entries]

    observations = []
    axis_numeric = column.numeric
    for row_index, row in enumerate(rows):
        axis_value = axis_numeric[row_index]
        if axis_value is None:
            continue
        row_labels = _row_labels(row, label_columns)
        for measured in value_columns:
            value = measured.numeric[row_index]
            if value is not None:
                observations.append(
                    Observation(axis_value=axis_value, value=value,
                                column_label=measured.name, row_labels=row_labels,
                                reports_mean=_reports_mean(measured, row_index))
                )

    return SchemaFit(
        fits=True,
        reason="long: one axis column, measured values in sibling columns",
        orientation="long",
        axis_label=column.name,
        axis_values=quality["distinct"],
        axis_confidence=quality["confidence"],
        axis_runs=quality["runs"],
        value_columns=[measured.name for measured in value_columns],
        label_columns=[label.name for label in label_columns],
        observations=observations,
    )


def _fit_wide(columns: list, rows: Sequence[Sequence[Any]]) -> SchemaFit:
    axis_columns = []
    for column in columns:
        position = parse_axis_label(column.name)
        if position is not None:
            axis_columns.append((position, column))

    if len(axis_columns) < MIN_AXIS_POINTS:
        return SchemaFit(fits=False, reason="fewer than 3 column names denote an axis position")

    positions = [position for position, _ in axis_columns]
    if any(later <= earlier for earlier, later in zip(positions, positions[1:])):
        return SchemaFit(fits=False, reason="axis-like column names do not increase left to right")

    if sum(1 for _, column in axis_columns if parse_dosed_label(column.name)) >= 2:
        return SchemaFit(fits=False, reason="column names are a dose ladder, not an axis")

    parseable = sum(len(column.values) for _, column in axis_columns)
    total = sum(len(column.entries) for _, column in axis_columns)
    if not total or parseable / total < MIN_NUMERIC_RATIO:
        return SchemaFit(fits=False, reason="cells under the axis columns are not numeric")

    axis_positions = {column.position for _, column in axis_columns}
    label_columns = [column for column in columns
                     if column.position not in axis_positions and column.entries]

    observations = []
    rows_used = 0
    for row_index, row in enumerate(rows):
        points = [
            (position, column)
            for position, column in axis_columns
            if column.numeric[row_index] is not None
        ]
        if len(points) < MIN_SERIES_POINTS:
            continue
        rows_used += 1
        row_labels = _row_labels(row, label_columns) or {"row": f"row {row_index + 1}"}
        for position, column in points:
            observations.append(
                Observation(axis_value=position, value=column.numeric[row_index],
                            column_label=None,
                            row_labels=row_labels,
                            reports_mean=_reports_mean(column, row_index))
            )

    if not rows_used:
        return SchemaFit(
            fits=False,
            reason="no row carries 3 or more points across the axis",
            orientation="wide",
        )

    return SchemaFit(
        fits=True,
        reason="wide: axis in the column names, one series per row",
        orientation="wide",
        axis_label="|".join(column.name for _, column in axis_columns),
        axis_values=positions,
        axis_confidence=0.9,
        axis_runs=1,
        value_columns=[column.name for _, column in axis_columns],
        label_columns=[label.name for label in label_columns],
        observations=observations,
    )


KEYED_LABEL = "row label"


def column_label(name: Any) -> str:
    """The name a column is known by."""
    return str(name or "").strip() or KEYED_LABEL


def _fit_keyed(columns: list, rows: Sequence[Sequence[Any]],
               prefer_axis: Optional[str] = None) -> SchemaFit:
    """The axis is inside a text label, one cell per row: "control, 6 days". The label's
    residue is carried as a row label under the column's own name, which is where the
    interpretation stage already looks for an arm."""
    candidates = [column for column in columns
                  if not column.is_numeric and column.keys_an_axis]
    if prefer_axis:
        named = [column for column in columns
                 if not column.is_numeric
                 and canonical_key(column_label(column.name)) == prefer_axis]
        candidates = named or candidates
    if not candidates:
        return SchemaFit(fits=False, reason="no label column carries an axis position")

    value_columns = [column for column in columns
                     if column.is_numeric and len(set(column.values)) >= 2]
    if not value_columns:
        return SchemaFit(fits=False, reason="a keyed axis but no varying measured column",
                         orientation="keyed")

    column = max(candidates, key=lambda item: (
        len({position for position, _ in item.keyed if position is not None}), -item.position))

    keyed = column.keyed
    arms_present = any(position is not None and residue for position, residue in keyed)

    other_labels = [item for item in columns
                    if item is not column and not item.is_numeric and item.entries]
    axis_name = column_label(column.name)
    observations = []
    used = set()
    for row_index, row in enumerate(rows):
        position, residue = keyed[row_index]
        if position is None or (arms_present and not residue):
            continue
        used.add(position)
        row_labels = _row_labels(row, other_labels)
        if residue:
            row_labels[axis_name] = residue
        for measured in value_columns:
            value = measured.numeric[row_index]
            if value is not None:
                observations.append(
                    Observation(axis_value=position, value=value,
                                column_label=measured.name, row_labels=row_labels,
                                reports_mean=_reports_mean(measured, row_index))
                )

    if len(used) < MIN_AXIS_POINTS:
        return SchemaFit(fits=False, reason="fewer than 3 axis positions survive keying",
                         orientation="keyed")

    positions = [position for position, _ in keyed if position is not None]
    return SchemaFit(
        fits=True,
        reason="keyed: the axis is inside a row label, measured values alongside",
        orientation="keyed",
        axis_label=axis_name,
        axis_values=sorted(used),
        axis_confidence=0.7,
        axis_runs=len(_runs(positions)),
        value_columns=[measured.name for measured in value_columns],
        label_columns=[item.name for item in other_labels] + [axis_name],
        observations=observations,
    )


def fits_schema(headers: Sequence[Any], rows: Sequence[Sequence[Any]],
                prefer_axis: Optional[str] = None) -> SchemaFit:
    """Does this table hold values observed over an ordinal axis?

    Every orientation is fitted and the stronger evidence wins: a long axis that
    *restarts*, then one spelled out in the column names, then a long axis that climbs
    once, then one keyed out of a row label. Wide cannot go first — a dose ladder in the
    header reads as a perfectly good ascending axis — and keyed goes last, because
    reading a number out of prose is the weakest evidence here.
    """
    if not headers or not rows:
        return SchemaFit(fits=False, reason="empty table")

    columns = _columns(headers, rows)

    if prefer_axis and any(not column.is_numeric
                           and canonical_key(column_label(column.name)) == prefer_axis
                           for column in columns):
        fit = _fit_keyed(columns, rows, prefer_axis=prefer_axis)
        fit.columns = columns
        return fit

    long = _fit_long(columns, rows, prefer_axis=prefer_axis)
    wide = keyed = None
    if not (long.fits and (long.axis_runs > 1 or prefer_axis)):
        wide = _fit_wide(columns, rows)
    if not (long.fits or (wide is not None and wide.fits)):
        keyed = _fit_keyed(columns, rows, prefer_axis=prefer_axis)

    if wide is not None and wide.fits:
        fit = wide
    elif long.fits:
        fit = long
    elif keyed is not None and keyed.fits:
        fit = keyed
    elif wide is not None and wide.orientation:
        fit = wide
    else:
        fit = long

    fit.columns = columns
    return fit


MIN_SIDE_PX = 120
MIN_AREA_PX = 40_000
ASPECT_BOUNDS = (0.2, 5.0)
MIN_BACKGROUND_FRACTION = 0.35
MAX_INK_FRACTION = 0.60
MIN_INK_FRACTION = 0.004
MIN_RULE_SPAN = 0.55
PROBE_LONG_SIDE = 320
INK_THRESHOLD = 32
COLOUR_LEVELS = 8
COLOUR_STEP = 256 // COLOUR_LEVELS

EDGE_STRENGTH = 24
AXIS_ALIGNED_RATIO = 3.0
MIN_AXIS_ALIGNED_EDGES = 0.45


@dataclass
class PlotProbe:
    plot_like: bool
    reason: str
    width: int = 0
    height: int = 0
    background_fraction: float = 0.0
    ink_fraction: float = 0.0
    distinct_colours: int = 0
    h_rule: float = 0.0
    v_rule: float = 0.0
    axis_aligned_edges: float = 0.0
    caption_verdict: str = "unknown"

    def summary(self) -> dict:
        return {
            "plot_like": self.plot_like,
            "reason": self.reason,
            "width": self.width,
            "height": self.height,
            "background_fraction": round(self.background_fraction, 3),
            "ink_fraction": round(self.ink_fraction, 3),
            "distinct_colours": self.distinct_colours,
            "h_rule": round(self.h_rule, 3),
            "v_rule": round(self.v_rule, 3),
            "axis_aligned_edges": round(self.axis_aligned_edges, 3),
            "caption_verdict": self.caption_verdict,
        }


def _rule_span(mask, axis: int) -> float:
    """How far the strongest rule reaches, as a fraction of the frame: an unbroken run,
    or a line mostly inked along its length so a dashed spine still counts."""
    grid = mask if axis == 1 else mask.T
    span = grid.shape[1]
    if span == 0:
        return 0.0

    pad = np.zeros((grid.shape[0], 1), dtype=bool)
    padded = np.concatenate([pad, grid, pad], axis=1).astype(np.int8)
    deltas = np.diff(padded, axis=1)
    _, starts = np.where(deltas == 1)
    _, ends = np.where(deltas == -1)
    longest_run = int((ends - starts).max()) if starts.size else 0

    return max(longest_run / span, float(grid.mean(axis=1).max()))


def _axis_aligned_edge_fraction(pixels) -> float:
    """Share of the image's edges that run along one of the two axes. Tick marks, spines,
    gridlines and bar sides are all axis-aligned, so a plot is strongly bimodal here; a
    photograph's edges point everywhere.
    """
    grey = pixels.mean(axis=2)
    if min(grey.shape) < 3:
        return 0.0
    horizontal = np.abs(np.diff(grey, axis=1))[:-1, :]
    vertical = np.abs(np.diff(grey, axis=0))[:, :-1]
    strong = np.maximum(horizontal, vertical) >= EDGE_STRENGTH
    if not strong.any():
        return 0.0
    across, down = horizontal[strong], vertical[strong]
    aligned = ((across >= AXIS_ALIGNED_RATIO * down)
               | (down >= AXIS_ALIGNED_RATIO * across))
    return float(aligned.mean())


_PLOT_CAPTION_HINTS = re.compile(
    r"\bvs\.?\b|\bversus\b|\bas\s+a\s+function\s+of\b|\bover\s+(?:time|storage)\b"
    r"|\bstorage\s+(?:day|time|period)\b|\bchanges?\s+(?:in|of)\b|\bevolution\s+of\b"
    r"|\bcurve\b|\bkinetics?\b|\bgrowth\b|\bcount\b|\baxis\b"
    r"|\blog\s*(?:10)?\s*cfu\b|\bmg\s*/\s*\d*\s*(?:g|kg|ml|l)\b|\bday\s*\d+\b|\b\d+\s*°\s*c\b",
    re.IGNORECASE)
_PHOTO_CAPTION_HINTS = re.compile(
    r"\bphotograph\b|\bphotos?\b|\bappearance\b|\bvisual\s+aspect\b|\bmicrograph\b"
    r"|\bscanning\s+electron\b|\btransmission\s+electron\b|\b(?:sem|tem)\s+(?:image|micrograph)\b"
    r"|\bschematic\b|\bflow\s*[- ]?chart\b|\bdiagram\s+of\b|\bapparatus\b|\bset[- ]?up\b",
    re.IGNORECASE)


def caption_plot_verdict(*texts) -> str:
    """"plot", "photo" or "unknown" from the words around a figure. A caption naming both
    is unknown: one of the two patterns matched something incidental."""
    joined = " ".join(str(text or "") for text in texts)
    if not joined.strip():
        return "unknown"
    says_plot = bool(_PLOT_CAPTION_HINTS.search(joined))
    says_photo = bool(_PHOTO_CAPTION_HINTS.search(joined))
    if says_plot == says_photo:
        return "unknown"
    return "plot" if says_plot else "photo"


def plot_likeness(image_path: str, *, caption=None, nearby_text=None) -> PlotProbe:
    """Cheap, content-free test for 'this image could be a data display'."""
    try:
        image = Image.open(image_path).convert("RGB")
    except (OSError, ValueError, Image.DecompressionBombError) as exc:
        return PlotProbe(plot_like=False, reason=f"unreadable image ({exc})")

    verdict = caption_plot_verdict(caption, nearby_text)
    width, height = image.size
    aspect = width / height if height else 0.0
    if not (ASPECT_BOUNDS[0] <= aspect <= ASPECT_BOUNDS[1]):
        return PlotProbe(False, f"aspect ratio {aspect:.2f} outside plot range", width, height,
                         caption_verdict=verdict)

    if min(width, height) < MIN_SIDE_PX or width * height < MIN_AREA_PX:
        return PlotProbe(False, "too small to be a data display", width, height,
                         caption_verdict=verdict)

    scale = PROBE_LONG_SIDE / max(width, height)
    if scale < 1.0:
        image = image.resize(
            (max(1, int(width * scale)), max(1, int(height * scale))), Image.BILINEAR
        )
    pixels = np.asarray(image, dtype=np.int16)

    quantised = (pixels // COLOUR_STEP).astype(np.uint8)
    codes = (quantised[..., 0] * COLOUR_LEVELS ** 2
             + quantised[..., 1] * COLOUR_LEVELS + quantised[..., 2])
    counts = np.bincount(codes.ravel(), minlength=COLOUR_LEVELS ** 3)
    ground_code = int(counts.argmax())
    background_fraction = float(counts.max()) / codes.size

    ground = pixels[codes == ground_code].mean(axis=0)
    ink = (np.abs(pixels - ground).max(axis=2) > INK_THRESHOLD)
    ink_fraction = float(ink.mean())

    probe = PlotProbe(
        plot_like=False,
        reason="",
        width=width,
        height=height,
        background_fraction=background_fraction,
        ink_fraction=ink_fraction,
        distinct_colours=int((counts > 0).sum()),
        caption_verdict=verdict,
    )

    if background_fraction < MIN_BACKGROUND_FRACTION:
        probe.reason = "no uniform ground — photographic or dense imagery"
        return probe
    if ink_fraction > MAX_INK_FRACTION:
        probe.reason = "image is mostly ink"
        return probe
    if ink_fraction < MIN_INK_FRACTION:
        probe.reason = "image is effectively blank"
        return probe

    probe.h_rule = _rule_span(ink, axis=1)
    probe.v_rule = _rule_span(ink, axis=0)
    if probe.h_rule < MIN_RULE_SPAN or probe.v_rule < MIN_RULE_SPAN:
        probe.reason = "no axis-like rules in both directions"
        return probe

    probe.axis_aligned_edges = _axis_aligned_edge_fraction(pixels)
    if FIGURE_EDGE_FILTER and probe.axis_aligned_edges < MIN_AXIS_ALIGNED_EDGES:
        probe.reason = (f"only {probe.axis_aligned_edges:.0%} of edges run along an axis "
                        "— photographic rather than drawn")
        return probe
    if FIGURE_CAPTION_FILTER and verdict == "photo":
        probe.reason = "the caption calls it a photograph, micrograph or schematic"
        return probe

    probe.plot_like = True
    probe.reason = "uniform ground with rules on both axes"
    return probe


PROBE_FILTER_COLUMNS = ["filter", "would_reject", "of_survivors", "share"]


def probe_filter_report(packages: Sequence[dict]) -> pd.DataFrame:
    """What the two opt-in filters would have rejected, over figures the geometry probe
    already admitted. Read this before setting FIGURE_EDGE_FILTER or
    FIGURE_CAPTION_FILTER: the share is only half the answer, and the other half is
    opening the images it names and counting how many were real charts.
    """
    probes = [item.get("probe") or {}
              for package in packages
              for item in chain(package.get("figures", []), package.get("review", []))
              if item.get("is_figure")]
    survivors = [probe for probe in probes if probe.get("plot_like")]
    counted = [
        ("axis_aligned_edges", sum(1 for probe in survivors
                                   if probe.get("axis_aligned_edges", 1.0)
                                   < MIN_AXIS_ALIGNED_EDGES)),
        ("caption_says_photo", sum(1 for probe in survivors
                                   if probe.get("caption_verdict") == "photo")),
    ]
    frame = pd.DataFrame(
        [{"filter": name, "would_reject": count, "of_survivors": len(survivors),
          "share": round(count / len(survivors), 3) if survivors else 0.0}
         for name, count in counted],
        columns=PROBE_FILTER_COLUMNS)
    display(frame)
    return frame

## Silver

Deterministic cleaning, then the gate. Output is a package of gated assets with their observations parsed into (axis, label, value) tuples.

In [ ]:
import polars as pl

_POSITIONAL_HEADER = re.compile(r"^(unnamed[:_ ]|column[_ ]?\d+$|\d+$|_duplicated_)", re.IGNORECASE)

HEADER_PROMOTION_RATIO = 0.6
FOOTNOTE_MIN_CHARS = 40


def dedupe_names(names: Sequence[Any]) -> list:
    """Make column names unique, first occurrence untouched: a two-row header repeats its
    group label, so lifting it collapses two columns onto one name."""
    seen: dict = {}
    unique = []
    for name in names:
        text = str(name)
        seen[text] = seen.get(text, 0) + 1
        unique.append(text if seen[text] == 1 else f"{text}{DUPLICATE_SUFFIX}{seen[text]}")
    return unique


def promote_header_row(frame: pl.DataFrame) -> pl.DataFrame:
    """Lift row 0 into the header when Docling left the columns positional."""
    if frame.height < 2:
        return frame
    threshold = max(1, int(HEADER_PROMOTION_RATIO * frame.width))
    if sum(1 for name in frame.columns if _POSITIONAL_HEADER.match(str(name))) < threshold:
        return frame
    first = [str(value).strip() if value is not None else "" for value in frame.row(0)]
    if sum(1 for value in first if value) < threshold:
        return frame

    promoted = frame.slice(1)
    promoted.columns = dedupe_names(new or old for old, new in zip(frame.columns, first))
    return promoted


def drop_footnote_rows(frame: pl.DataFrame) -> pl.DataFrame:
    """Drop trailing rows holding one long free-text cell — the shape of a footnote."""
    height = frame.height
    while height:
        filled = [str(value).strip() for value in frame.row(height - 1)
                  if value is not None and str(value).strip()]
        if len(filled) == 1 and len(filled[0]) > FOOTNOTE_MIN_CHARS:
            height -= 1
            continue
        break
    return frame if height == frame.height else frame.slice(0, height)


def clean_table_frame(frame: pl.DataFrame) -> pl.DataFrame:
    frame = promote_header_row(frame)
    frame.columns = dedupe_names(frame.columns)
    stripped = [pl.col(name).str.strip_chars()
                for name, dtype in zip(frame.columns, frame.dtypes) if dtype == pl.Utf8]
    if stripped:
        frame = frame.with_columns(stripped)
    frame = frame.select([name for name in frame.columns if not frame[name].is_null().all()])
    if frame.height and frame.width:
        frame = frame.filter(pl.any_horizontal([pl.col(c).is_not_null() for c in frame.columns]))
    return drop_footnote_rows(frame)


def frame_to_records(frame: pl.DataFrame) -> tuple:
    """The (headers, rows) pair the gate consumes."""
    return list(frame.columns), [list(row) for row in frame.iter_rows()]


def as_markdown_table(headers: Sequence[Any], rows: Sequence[Sequence[Any]], limit: int = 5) -> str:
    """A small pipe table for prompts and previews, without pulling in tabulate."""
    def cell(value):
        return "" if value is None else str(value).replace("|", "\\|").strip()

    head = [cell(h) for h in headers]
    lines = ["| " + " | ".join(head) + " |", "| " + " | ".join("---" for _ in head) + " |"]
    for row in rows[:limit]:
        lines.append("| " + " | ".join(cell(value) for value in row) + " |")
    if len(rows) > limit:
        lines.append(f"_({len(rows) - limit} more rows)_")
    return "\n".join(lines)


_MARKDOWN_HEADING = re.compile(r"^#{1,6}\s+(.*)$")


def markdown_sections(markdown_text: str) -> list:
    sections = []
    title, lines = "Document", []

    def close():
        if lines:
            sections.append({"section_title": title,
                             "content_markdown": "\n".join(lines).strip()})

    for line in markdown_text.splitlines():
        heading = _MARKDOWN_HEADING.match(line.strip())
        if not heading:
            lines.append(line)
            continue
        close()
        title, lines = heading.group(1).strip(), []
    close()

    kept = [section for section in sections if section["content_markdown"]]
    for order, section in enumerate(kept):
        section["docling_item_ref"] = f"#/sections/{order}"
        section["page_number"] = None
        section["level"] = 1
        section["item_refs"] = []
    return kept


PREAMBLE_TITLE = "Document"


def document_sections(texts: Sequence[dict]) -> list:
    sections, current = [], None

    def open_section(item):
        return {"section_title": (item or {}).get("text") or PREAMBLE_TITLE,
                "docling_item_ref": (item or {}).get("docling_item_ref"),
                "page_number": (item or {}).get("page_number"),
                "level": (item or {}).get("level") or 1,
                "item_refs": [], "lines": []}

    for item in texts:
        if item.get("is_heading"):
            if current is not None:
                sections.append(current)
            current = open_section(item)
            continue
        if current is None:
            current = open_section(None)
        current["item_refs"].append(item["docling_item_ref"])
        current["lines"].append(item["text"])
        if current["page_number"] is None:
            current["page_number"] = item.get("page_number")
    if current is not None:
        sections.append(current)

    kept = []
    for order, section in enumerate(sections):
        content = "\n\n".join(section.pop("lines")).strip()
        if not content:
            continue
        kept.append({**section, "content_markdown": content,
                     "docling_item_ref": section["docling_item_ref"] or f"#/sections/{order}"})
    return kept


def paper_sections(manifest: dict) -> tuple:
    texts = manifest.get("texts") or []
    if any(item.get("is_heading") for item in texts):
        return document_sections(texts), "docling items"
    return markdown_sections(
        Path(manifest["markdown_path"]).read_text(encoding="utf-8")), "markdown headings"


_METHODS_HEAD = re.compile(
    r"materials?\s*(?:and|&)\s*methods?|methods?\s*(?:and|&)\s*materials?"
    r"|^\s*\d*\.?\s*(?:methods?|methodolog\w*|experimental)\b"
    r"|sample\s+(?:preparation|processing)|preparation\s+of\b", re.IGNORECASE)
_METHODS_STOP = re.compile(
    r"^\s*\d*\.?\s*(?:results?|discussion|conclusion|acknowledg\w*|references|bibliography)\b",
    re.IGNORECASE)


def methods_section_refs(sections: Sequence[dict]) -> set:
    nested = len({section.get("level") or 1 for section in sections}) > 1
    found, head_level = set(), None
    for section in sections:
        title = section.get("section_title") or ""
        level = section.get("level") or 1
        if head_level is None:
            if _METHODS_HEAD.search(title):
                head_level = level
                found.add(section["docling_item_ref"])
            continue
        if _METHODS_STOP.search(title) or (nested and level <= head_level):
            break
        found.add(section["docling_item_ref"])
    return found


def build_text_index(texts: Sequence[dict]) -> dict:
    """Bucket the paper's text items by page, once, for every asset to share."""
    by_page: dict = {}
    for item in texts:
        by_page.setdefault(item.get("page_number"), []).append(item)
    return {"all": list(texts), "by_page": by_page, "context": {}}


def local_context(text_index: dict, anchor: dict, budget: int = FIGURE_CONTEXT_CHARS) -> dict:
    """Text near an asset, chosen by position rather than by words: the page first, then
    reading order for a page that is all figure. Memoised per anchor — the probe and the
    gated record both want the same passage, and ranking every text item twice per asset
    is pure CPU spent while the GPU waits."""
    cache = text_index.setdefault("context", {})
    order = anchor.get("order", 0)
    page = anchor.get("page_number")
    key = (anchor.get("docling_item_ref"), order, page, budget)
    hit = cache.get(key)
    if hit is not None:
        return hit

    candidates = text_index["by_page"].get(page) if page is not None else None
    if not candidates:
        candidates = text_index["all"]

    ranked = sorted(candidates, key=lambda item: abs(item.get("order", 0) - order))
    chosen, used = [], 0
    for item in ranked:
        text = item.get("text", "")
        if used + len(text) > budget and chosen:
            break
        chosen.append(item)
        used += len(text)
    chosen.sort(key=lambda item: item.get("order", 0))
    headings = [item.get("heading") for item in chosen if item.get("heading")]
    context = {
        "context_markdown": "\n\n".join(item["text"] for item in chosen),
        "context_refs": [item["docling_item_ref"] for item in chosen],
        "section_hint": headings[0] if headings else None,
    }
    cache[key] = context
    return context


_chart_model = None
_CHART_MODEL_LOCK = threading.Lock()


def get_chart_model():
    """Load PP-Chart2Table once per kernel, on the device chosen at startup. Raises rather
    than degrading the run into "this paper has no plottable figures"."""
    global _chart_model
    with _CHART_MODEL_LOCK:
        if _chart_model is not None:
            return _chart_model
        try:
            from paddleocr import ChartParsing
        except ImportError as exc:
            raise ImportError(
                "paddleocr with ChartParsing is required to test figures against the "
                "schema. Install it, or set REQUIRE_FIGURE_DATA=0 to admit figures on "
                "the visual probe alone."
            ) from exc
        with stage_timer("chart_model_load"):
            _chart_model = ChartParsing(
                model_name="PP-Chart2Table",
                engine="transformers",
                engine_config={"dtype": "float16" if CUDA_AVAILABLE else "float32"},
                device=DEVICE,
            )
        print(f"PP-Chart2Table loaded on {DEVICE}.")
        return _chart_model


def release_chart_model() -> None:
    """Only when the card cannot hold the VLM and the LLM at once: with num_ctx sized to
    fit, releasing it between stages just re-pays the load on the next shard."""
    global _chart_model
    with _CHART_MODEL_LOCK:
        _chart_model = None
    free_memory()


def parse_markdown_table(text: str) -> Optional[tuple]:
    """Parse the pipe table PP-Chart2Table returns into (headers, rows)."""
    if not text or not text.strip():
        return None
    lines = [line for line in text.splitlines() if line.strip()]
    lines = [line for line in lines if not re.match(r"^\s*\|?[\s\-:]+(\|[\s\-:]+)*\|?\s*$", line)]
    if len(lines) < 2:
        return None
    grid = []
    for line in lines:
        cells = [cell.strip() for cell in line.strip().strip("|").split("|")]
        if any(cells):
            grid.append(cells)
    if len(grid) < 2:
        return None
    width = max(len(row) for row in grid)
    grid = [row + [None] * (width - len(row)) for row in grid]
    return dedupe_names(grid[0]), grid[1:]


def figure_result(status: str, **extra) -> dict:
    """One shape for every outcome, so callers branch on `status` and never on a key."""
    return {"status": status, "headers": None, "rows": None, **extra}


def _model_inputs(image_paths: Sequence[str], max_side: int, workdir: Path) -> list:
    """Bounded copies of the figures, positionally aligned with `image_paths`. The model
    tiles what it is given, so its cost follows the pixel count; the archived PNG keeps
    its full IMAGES_SCALE resolution. Named after the source figure and the bound, so a
    re-preparation at a smaller size cannot collide with the copy it replaces."""
    prepared = []
    for path in image_paths:
        try:
            with Image.open(path) as opened:
                opened.load()
                if max(opened.size) <= max_side:
                    prepared.append(str(path))
                    continue
                scale = max_side / max(opened.size)
                resized = opened.convert("RGB").resize(
                    (max(1, int(opened.width * scale)), max(1, int(opened.height * scale))),
                    Image.LANCZOS)
            target = workdir / f"chart_{Path(path).stem}_{max_side}.png"
            resized.save(target)
            prepared.append(str(target))
        except (OSError, ValueError):
            prepared.append(str(path))
    return prepared


def _is_out_of_memory(exc: BaseException) -> bool:
    """Newer torch raises a dedicated class, older a plain RuntimeError."""
    dedicated = getattr(torch, "cuda", None) and getattr(torch.cuda, "OutOfMemoryError", None)
    return (dedicated is not None and isinstance(exc, dedicated)) or (
        isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower())


_CHART_PREDICT_KWARGS: Optional[dict] = None


def _chart_predict(model, chunk: Sequence[str]) -> list:
    """One model call, with decoding bounded. Chart-to-table is long-output autoregressive
    generation; without a cap one malformed figure decodes until the model stops itself.
    The keyword moved between paddleocr releases, so the first call finds the one that
    works and every call after goes straight to it."""
    global _CHART_PREDICT_KWARGS
    payload = [{"image": str(path)} for path in chunk]
    if _CHART_PREDICT_KWARGS is not None:
        return list(model.predict(input=payload, batch_size=len(chunk),
                                  **_CHART_PREDICT_KWARGS))

    for kwargs in ({"max_new_tokens": CHART_MAX_NEW_TOKENS},
                   {"max_length": CHART_MAX_NEW_TOKENS}):
        try:
            predictions = list(model.predict(input=payload, batch_size=len(chunk), **kwargs))
        except TypeError:
            continue
        _CHART_PREDICT_KWARGS = kwargs
        print(f"        chart generation bounded by {next(iter(kwargs))}"
              f"={CHART_MAX_NEW_TOKENS}")
        return predictions

    _CHART_PREDICT_KWARGS = {}
    print("        WARNING: this paddleocr build accepts no generation bound on "
          "predict(); chart decoding is unbounded")
    return list(model.predict(input=payload, batch_size=len(chunk)))


def _convert_chunk(model, chunk: Sequence[str]) -> list:
    """Results are positional, so a short list is a hard error rather than a forgiving
    zip that would silently discard the tail."""
    predictions = _chart_predict(model, chunk)
    if len(predictions) != len(chunk):
        raise RuntimeError(
            f"PP-Chart2Table returned {len(predictions)} predictions for {len(chunk)} "
            f"figures; results are positional, so the mismatch cannot be attributed "
            f"and the paper is not partially converted."
        )
    results = []
    for prediction in predictions:
        parsed = parse_markdown_table(prediction.get("result", ""))
        results.append(figure_result("converted", headers=parsed[0], rows=parsed[1])
                       if parsed else figure_result("unparseable"))
    return results


def chart_cache_key(image_digest: str) -> str:
    return content_key("chart", image_digest, CHART_MODEL_VERSION,
                       CHART_MAX_PIXELS, CHART_MIN_PIXELS, CHART_MAX_NEW_TOKENS)


def load_cached_conversion(image_digest: str) -> Optional[dict]:
    """A figure is stable across re-runs, so the VLM should never see the same image
    twice — within a corpus or across restarts."""
    if not CACHE_ENABLED:
        return None
    path = CHART_CACHE_ROOT / f"{chart_cache_key(image_digest)}.json"
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


def store_cached_conversion(image_digest: str, result: dict) -> None:
    if not CACHE_ENABLED or result.get("status") == "oom":
        return
    path = CHART_CACHE_ROOT / f"{chart_cache_key(image_digest)}.json"
    temporary = path.with_suffix(f".{os.getpid()}.{threading.get_ident()}.tmp")
    temporary.write_text(json.dumps(result, ensure_ascii=False, default=str), encoding="utf-8")
    os.replace(temporary, path)


def _pixel_count(image_path: str) -> int:
    """Read from the header, not by decoding: this only orders the batch."""
    try:
        with Image.open(image_path) as opened:
            return opened.width * opened.height
    except (OSError, ValueError):
        return 0


def convert_figures(image_paths: Sequence[str]) -> list:
    """Recover a table from each chart image, positionally aligned with `image_paths`.

    Called once per shard rather than once per paper: model invocation dominates the cost,
    and a paper with two charts converted on its own runs at batch-size-1, the worst case
    for GPU utilisation. Identical images convert once, and every conversion is cached on
    the image's own digest.
    """
    if not image_paths:
        return []
    if not REQUIRE_FIGURE_DATA:
        return [figure_result("unavailable") for _ in image_paths]

    digests = [file_hash(Path(path)) for path in image_paths]
    first_seen = {}
    for index, digest in enumerate(digests):
        first_seen.setdefault(digest, index)

    by_digest, pending = {}, []
    for digest, index in first_seen.items():
        cached = load_cached_conversion(digest)
        if cached is None:
            pending.append(digest)
        else:
            by_digest[digest] = cached

    if pending:
        pending.sort(key=lambda digest: _pixel_count(image_paths[first_seen[digest]]))
        paths = [image_paths[first_seen[digest]] for digest in pending]
        with stage_timer("chart_vlm", figures=len(paths),
                         reused=len(image_paths) - len(paths)) as timing:
            with tempfile.TemporaryDirectory(prefix="chart_inputs_") as scratch:
                converted = _convert_within_memory(paths, Path(scratch))
            timing["converted"] = sum(1 for item in converted
                                      if item["status"] == "converted")
        for digest, result in zip(pending, converted):
            by_digest[digest] = result
            store_cached_conversion(digest, result)
    elif image_paths:
        record_metric("chart_vlm_cached", figures=len(image_paths), seconds=0.0)

    return [by_digest[digest] for digest in digests]


def _convert_within_memory(image_paths: Sequence[str], scratch: Path) -> list:
    """Convert every figure, giving up something at each out-of-memory in turn: halve the
    batch, then show the model a smaller copy. A figure that survives both costs itself,
    not the paper.
    """
    model = get_chart_model()
    max_side = CHART_MAX_PIXELS
    prepared = _model_inputs(image_paths, max_side, scratch)
    results, batch, index = [], max(1, CHART_BATCH_SIZE), 0

    while index < len(prepared):
        chunk = prepared[index:index + batch]
        try:
            results.extend(_convert_chunk(model, chunk))
        except Exception as exc:
            if not _is_out_of_memory(exc):
                raise
            del exc
            free_memory()

            if batch > 1:
                batch = max(1, batch // 2)
                print(f"        chart conversion out of memory; batch -> {batch}")
            elif max_side > CHART_MIN_PIXELS:
                max_side = max(CHART_MIN_PIXELS, max_side // 2)
                prepared[index:] = _model_inputs(image_paths[index:], max_side, scratch)
                print(f"        still out of memory; showing the model {max_side} px")
            else:
                results.append(figure_result("oom"))
                index += 1
                print(f"        figure {index} does not fit in memory — skipped")
            continue
        index += len(chunk)
    return results


def _observation_payload(fit: SchemaFit) -> list:
    """Serialise observations with their labels still unresolved."""
    return [
        {
            "axis_value": observation.axis_value,
            "value": observation.value,
            "column_label": observation.column_label,
            "row_labels": observation.row_labels,
            "reports_mean": observation.reports_mean,
        }
        for observation in fit.observations
    ]


def _asset_base(meta: dict, kind: str) -> dict:
    return {
        "kind": kind,
        "index": meta.get("table_index") if kind == "table" else meta.get("figure_index"),
        "docling_item_ref": meta.get("docling_item_ref"),
        "caption": meta.get("caption"),
        "page_number": meta.get("page_number"),
    }


MIN_REFERENCE_ENTITIES = 3
REVIEW_PREVIEW_ROWS = env_int("REVIEW_PREVIEW_ROWS", "10")


def _write_figure_csv(headers: Sequence[Any], rows: Sequence[Sequence[Any]], path: Path) -> None:
    pl.DataFrame(
        {name: [row[i] if i < len(row) else None for row in rows]
         for i, name in enumerate(headers)},
        strict=False,
    ).write_csv(path)


def _is_reference_table(fit: SchemaFit, row_count: int) -> bool:
    """A catalogue of entities against quantities — a column of distinct names beside a
    column of numbers — rather than a series over time. A column whose labels key an axis
    is a series that was not read, not a catalogue, so it disqualifies the whole table.
    """
    if not fit.columns or row_count < MIN_REFERENCE_ENTITIES:
        return False
    names = numbers = False
    for column in fit.columns:
        if not column.entries:
            continue
        if column.numeric_ratio >= MIN_NUMERIC_RATIO:
            numbers = True
        elif column.keys_an_axis:
            return False
        elif len(set(column.entries)) >= MIN_REFERENCE_ENTITIES:
            names = True
    return names and numbers


def _could_key_an_axis(column) -> bool:
    """Three distinct positions in the column's own labels — what `_fit_keyed` needs
    before it will accept the column, whether it chose it or the model named it."""
    return len({position for position, _ in column.keyed if position is not None}) \
        >= MIN_AXIS_POINTS


def _needs_review(fit: SchemaFit) -> bool:
    """Shaped like data, but the gate could not key it: numbers and labels, no axis in
    either. Far more often a measurement table the gate misread than it is furniture, so
    it goes to the model to have its axis named rather than into `rejected`.

    Only where an answer could change the verdict. The model is asked for the name of a
    column, and the keyed fit that unlocks needs three distinct positions in that
    column's own cells — a table where no label column carries three is one no answer
    can promote, and asking about it spends a whole reasoning call to be told nothing.
    """
    if not fit.columns:
        return False
    if not any(column.is_numeric for column in fit.columns):
        return False
    return any(_could_key_an_axis(column) for column in fit.columns
               if not column.is_numeric and column.entries)


def _gated_record(meta: dict, fit: SchemaFit, text_index: dict, headers, **extra) -> dict:
    return {**meta, "headers": headers, "gate": fit.summary(),
            **local_context(text_index, meta), **extra}


def _review_record(meta: dict, fit: SchemaFit, text_index: dict, headers, rows, **extra) -> dict:
    """Kept whole: `adjudicate_review` re-gates these rows once the model names the axis."""
    return _gated_record(
        meta, fit, text_index, headers, rows=rows, why=fit.reason,
        preview_markdown=as_markdown_table(headers, rows, limit=REVIEW_PREVIEW_ROWS), **extra)


def _accepted_record(meta: dict, fit: SchemaFit, text_index: dict, headers, rows, **extra) -> dict:
    return _gated_record(
        meta, fit, text_index, headers,
        preview_markdown=as_markdown_table(headers, rows),
        observations=_observation_payload(fit), **extra)


def gate_table(table_meta: dict, text_index: dict, tables_dir: Path) -> tuple:
    """Clean, gate and stage one native table. Returns (verdict, record)."""
    base = _asset_base(table_meta, "table")
    csv_path = table_meta.get("csv_path")
    if not csv_path:
        return "rejected", {**base, "stage": "bronze", "reason": table_meta.get("error", "no CSV")}

    frame = clean_table_frame(pl.read_csv(csv_path, infer_schema_length=50, ignore_errors=True))
    headers, rows = frame_to_records(frame)
    fit = fits_schema(headers, rows)
    cleaned_csv_path = tables_dir / Path(csv_path).name
    staged = {"cleaned_csv_path": str(cleaned_csv_path)}

    if not fit.fits:
        if _is_reference_table(fit, len(rows)):
            frame.write_csv(cleaned_csv_path)
            return "reference", _gated_record(
                table_meta, fit, text_index, headers, rows=rows, **staged,
                why=f"no measurement axis ({fit.reason}); rows name entities")
        if _needs_review(fit):
            frame.write_csv(cleaned_csv_path)
            return "review", _review_record(table_meta, fit, text_index, headers, rows,
                                            is_figure=False, **staged)
        return "rejected", {**base, "stage": "schema_gate", "reason": fit.reason}

    frame.write_csv(cleaned_csv_path)
    return "accepted", _accepted_record(
        table_meta, fit, text_index, headers, rows, is_figure=False, **staged,
        row_count=frame.height, col_count=frame.width)


def probe_figure(figure_meta: dict, text_index: Optional[dict] = None) -> tuple:
    """Cheap visual test, no model. Split from the gate so every survivor converts in
    one batched call, and given the words around the figure so the caption filter has
    something to read."""
    base = _asset_base(figure_meta, "figure")
    if not figure_meta.get("image_path"):
        return "rejected", {**base, "stage": "bronze", "reason": figure_meta.get("error", "no image")}
    nearby = (local_context(text_index, figure_meta)["context_markdown"]
              if text_index is not None else None)
    probe = plot_likeness(figure_meta["image_path"],
                          caption=figure_meta.get("caption"), nearby_text=nearby)
    if not probe.plot_like:
        return "rejected", {**base, "stage": "probe", "reason": probe.reason,
                            "probe": probe.summary()}
    return "probed", probe


def gate_converted_figure(figure_meta: dict, probe, conversion: dict,
                          text_index: dict, figures_dir: Path) -> tuple:
    """Gate one already-converted figure. Returns (verdict, record)."""
    base = _asset_base(figure_meta, "figure")
    if conversion["status"] != "converted":
        if conversion["status"] == "unavailable" and not REQUIRE_FIGURE_DATA:
            return "accepted", {
                **figure_meta,
                "is_figure": True,
                "probe": probe.summary(),
                "conversion_status": "unavailable",
                "gate": {"fits": None, "reason": "admitted on the visual probe alone"},
                "observations": [],
                **local_context(text_index, figure_meta),
            }
        return "rejected", {
            **base,
            "stage": "conversion",
            "reason": conversion.get("error") or conversion["status"],
            "probe": probe.summary(),
        }

    headers, rows = conversion["headers"], conversion["rows"]
    fit = fits_schema(headers, rows)
    figure_csv_path = figures_dir / f"figure_{figure_meta['figure_index']:03d}.csv"
    staged = {"is_figure": True, "probe": probe.summary(), "conversion_status": "converted",
              "csv_path": str(figure_csv_path)}

    if not fit.fits:
        if not _needs_review(fit):
            return "rejected", {**base, "stage": "schema_gate", "reason": fit.reason,
                                "probe": probe.summary()}
        _write_figure_csv(headers, rows, figure_csv_path)
        return "review", _review_record(figure_meta, fit, text_index, headers, rows, **staged)

    _write_figure_csv(headers, rows, figure_csv_path)
    return "accepted", _accepted_record(figure_meta, fit, text_index, headers, rows, **staged)


ASSET_FAILURES = (ValueError, TypeError, KeyError, IndexError, OSError, pl.exceptions.PolarsError)


_VERDICT_SECTION = {"reference": "references", "review": "review", "rejected": "rejected"}
GATED_SECTIONS = ("tables", "figures", "references", "review", "rejected")


def _dispatch(gated: dict, accepted: str, verdict: str, record: dict) -> None:
    gated[accepted if verdict == "accepted" else _VERDICT_SECTION[verdict]].append(record)


def _reject(gated: dict, meta: dict, kind: str, exc: BaseException) -> None:
    gated["rejected"].append({**_asset_base(meta, kind), "stage": "error",
                              "reason": f"{type(exc).__name__}: {exc}"})


def _gate_tables_and_probe(manifest: dict, text_index: dict, tables_dir: Path) -> tuple:
    """Everything Silver decides without the chart model: every table gated, every figure
    probed. Returns (gated sections, [(figure_meta, probe)]). One malformed asset costs
    itself, not the paper."""
    gated = {name: [] for name in GATED_SECTIONS}

    for meta in manifest.get("tables", []):
        try:
            _dispatch(gated, "tables", *gate_table(meta, text_index, tables_dir))
        except ASSET_FAILURES as exc:
            _reject(gated, meta, "table", exc)

    probed = []
    for meta in manifest.get("figures", []):
        try:
            verdict, payload = probe_figure(meta, text_index)
        except ASSET_FAILURES as exc:
            _reject(gated, meta, "figure", exc)
            continue
        if verdict == "probed":
            probed.append((meta, payload))
        else:
            gated["rejected"].append(payload)
    return gated, probed


def _gate_converted_figures(gated: dict, probed: Sequence, conversions: Sequence,
                            text_index: dict, figures_dir: Path) -> None:
    """The results come back positionally, one per probed figure."""
    for (meta, probe), conversion in zip(probed, conversions):
        try:
            _dispatch(gated, "figures",
                      *gate_converted_figure(meta, probe, conversion, text_index, figures_dir))
        except ASSET_FAILURES as exc:
            _reject(gated, meta, "figure", exc)


def _gate_report(manifest: dict, gated: dict) -> dict:
    rejected = gated["rejected"]
    return {
        "tables_in": len(manifest.get("tables", [])),
        "tables_accepted": len(gated["tables"]),
        "figures_in": len(manifest.get("figures", [])),
        "figures_accepted": len(gated["figures"]),
        "rejected_by_stage": {
            stage: sum(1 for item in rejected if item["stage"] == stage)
            for stage in sorted({item["stage"] for item in rejected})
        },
        "references": len(gated["references"]),
        "review": len(gated["review"]),
        "observations": sum(len(item["observations"])
                            for item in chain(gated["tables"], gated["figures"])),
    }


def silver_cache_key(manifest: dict) -> str:
    """Keyed on the Bronze that produced it and on every setting that changes a verdict.
    REQUIRE_FIGURE_DATA is in here because it changes data semantics, not just speed."""
    return content_key(
        "silver", manifest.get("cache_key") or manifest.get("file_hash"), GATE_VERSION,
        CHART_MODEL_VERSION, REQUIRE_FIGURE_DATA, FIGURE_EDGE_FILTER, FIGURE_CAPTION_FILTER,
        CHART_MAX_PIXELS, CHART_MIN_PIXELS, CHART_MAX_NEW_TOKENS, REVIEW_PREVIEW_ROWS,
        FIGURE_CONTEXT_CHARS)


def _silver_artifacts_present(package: dict) -> bool:
    paths = [item.get("cleaned_csv_path") for item in package.get("tables", [])]
    paths += [item.get("csv_path") for item in package.get("figures", [])
              if item.get("conversion_status") == "converted"]
    return all(Path(path).exists() for path in paths if path)


def load_cached_silver(manifest: dict) -> Optional[dict]:
    if not CACHE_ENABLED:
        return None
    package = read_manifest(SILVER_ROOT / manifest["paper_slug"])
    if package is None or package.get("cache_key") != silver_cache_key(manifest):
        return None
    if not _silver_artifacts_present(package):
        return None
    print(f"Silver: {manifest['paper_slug']} reused from cache "
          f"| {package['gate_report']['observations']} observations")
    return package


@dataclass
class SilverStaging:
    """One paper carried as far as Silver can go without the chart model, so a whole
    shard's figures can be converted in one pass."""

    manifest: dict
    cache_key: str
    silver_dir: Path
    tables_dir: Path
    figures_dir: Path
    text_index: dict
    gated: dict
    probed: list

    @property
    def image_paths(self) -> list:
        return [meta["image_path"] for meta, _ in self.probed]


def stage_silver_package(manifest: dict) -> SilverStaging:
    slug = manifest["paper_slug"]
    silver_dir = SILVER_ROOT / slug
    tables_dir, figures_dir = silver_dir / "tables", silver_dir / "figures"
    for directory in (tables_dir, figures_dir):
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)
    (silver_dir / MANIFEST_NAME).unlink(missing_ok=True)

    text_index = build_text_index(manifest.get("texts", []))
    with stage_timer("gate_probe", slug) as timing:
        gated, probed = _gate_tables_and_probe(manifest, text_index, tables_dir)
        timing["figures_probed"] = len(probed)
    return SilverStaging(manifest=manifest, cache_key=silver_cache_key(manifest),
                         silver_dir=silver_dir, tables_dir=tables_dir,
                         figures_dir=figures_dir, text_index=text_index,
                         gated=gated, probed=probed)


def finish_silver_package(staging: SilverStaging, conversions: Sequence) -> dict:
    manifest, gated = staging.manifest, staging.gated
    slug = manifest["paper_slug"]

    with stage_timer("gate_figures", slug):
        _gate_converted_figures(gated, staging.probed, conversions,
                                staging.text_index, staging.figures_dir)
    report = _gate_report(manifest, gated)
    sections, section_source = paper_sections(manifest)
    methods = methods_section_refs(sections)

    silver_manifest = {
        "paper_slug": slug,
        "source_pdf": manifest["source_pdf"],
        "file_hash": manifest["file_hash"],
        "cache_key": staging.cache_key,
        "sections": sections,
        "methods_refs": sorted(methods),
        **gated,
        "gate_report": report,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    write_manifest(staging.silver_dir, silver_manifest)
    print(f"Silver: {slug} | tables {report['tables_accepted']}/{report['tables_in']} "
          f"| figures {report['figures_accepted']}/{report['figures_in']} "
          f"| {report['observations']} observations "
          f"| {report['references']} reference | {report['review']} for review "
          f"| rejected {report['rejected_by_stage']}")
    print(f"        {len(sections)} sections from {section_source}, "
          + (f"{len(methods)} of them methods" if methods else
             "NONE matched as methods — the protocol prompt sees the whole paper"))
    return silver_manifest


def build_silver_shard(manifests: Sequence[dict]) -> list:
    """Silver for a whole shard, with one chart-model pass over every figure in it.

    Returns (manifest, package) for the papers that got through. A paper that raises here
    is reported and dropped — a malformed table costing its own paper is the point of
    sharding, and losing the shard behind it is not.
    """
    staged, packages = [], {}
    for manifest in manifests:
        try:
            cached = load_cached_silver(manifest)
            if cached is not None:
                packages[manifest["paper_slug"]] = cached
            else:
                staged.append(stage_silver_package(manifest))
        except Exception as exc:
            print(f"Silver: {manifest['paper_slug']} failed while staging "
                  f"({type(exc).__name__}: {exc}) — skipped")
            record_metric("silver_failed", manifest["paper_slug"], seconds=0.0,
                          error=f"{type(exc).__name__}: {exc}")

    conversions = convert_figures([path for staging in staged
                                   for path in staging.image_paths])
    at = 0
    for staging in staged:
        slug = staging.manifest["paper_slug"]
        count = len(staging.probed)
        try:
            packages[slug] = finish_silver_package(
                staging, conversions[at:at + count])
        except Exception as exc:
            print(f"Silver: {slug} failed while gating its figures "
                  f"({type(exc).__name__}: {exc}) — skipped")
            record_metric("silver_failed", slug, seconds=0.0,
                          error=f"{type(exc).__name__}: {exc}")
        at += count
    return [(manifest, packages[manifest["paper_slug"]]) for manifest in manifests
            if manifest["paper_slug"] in packages]


REVIEW_SYSTEM_PROMPT = """\
You are reading tables and charts recovered from one scientific paper.

Each asset below has numbers in it, but the software could not tell which column orders
the rows into a series. Name that column. Do NOT transcribe any values.

For each asset return one object:

{
  "docling_item_ref": REQUIRED - copy it from the asset,
  "axis_column": the exact name, copied from this asset's `columns`, of the column whose
                 cells carry the point each row was observed at - a storage day, a
                 concentration, a time. The cell may hold more than that, as in
                 "control, 6 days"; name it anyway. Use null if no column does.
  "not_a_series": true when this is not repeated measurements at all - a composition
                  breakdown, a list of equipment, a statistics summary - else false,
  "why": one short sentence
}

Rules:
  * `axis_column` must be copied verbatim from that asset's `columns`, or be null. An
    invented name is discarded. An unheaded column is listed as "row label"; that is a
    name you may use.
  * A column of measured quantities is NOT the axis. The axis is what those quantities
    were measured against, and it is a column of text here - if it were a column of plain
    numbers the software would already have found it.
  * If several columns could be it, pick the one whose values repeat across groups.

Return ONLY valid JSON: {"assets": [ ... ]}
"""


def _review_prompt(package: dict) -> str:
    return json.dumps({
        "paper_slug": package["paper_slug"],
        "assets": [{
            "docling_item_ref": asset.get("docling_item_ref"),
            "kind": "figure" if asset.get("is_figure") else "table",
            "caption": asset.get("caption"),
            "columns": [column_label(name) for name in asset.get("headers", [])],
            "why_unresolved": asset.get("why"),
            "preview": asset.get("preview_markdown"),
        } for asset in package.get("review", [])],
    }, ensure_ascii=False, indent=2, default=str)


def adjudicate_review(package: dict, client) -> int:
    """Ask the model which column is the axis, re-gate with its answer, and return how
    many assets that promoted. An asset whose hint does not make the table fit stays in
    `review` carrying what the model said, so a bad hint is legible rather than silent.
    """
    review = package.get("review") or []
    if not review or client is None:
        return 0

    try:
        with stage_timer("adjudicate", package["paper_slug"], assets=len(review)):
            payload, _ = client.json_completion(
                system_prompt=REVIEW_SYSTEM_PROMPT, user_prompt=_review_prompt(package),
                cache_scope="review", paper=package["paper_slug"])
    except (RuntimeError, ValueError, KeyError) as exc:
        print(f"        review adjudication unavailable ({exc}) — assets stay unresolved")
        return 0

    hints = {}
    for entry in payload.get("assets") or []:
        try:
            hint = AssetHint.model_validate(entry)
        except ValidationError:
            continue
        hints[hint.docling_item_ref] = hint

    promoted, still_open = 0, []
    for asset in review:
        hint = hints.get(asset.get("docling_item_ref"))
        headers = [str(name) for name in asset.get("headers", [])]
        named = {column_label(name) for name in headers}
        if hint is None or hint.not_a_series or not hint.axis_column:
            asset["hint"] = "not a series" if (hint and hint.not_a_series) else "no answer"
            asset["why"] = (hint.why if hint else None) or asset.get("why")
            still_open.append(asset)
            continue
        if hint.axis_column not in named:
            asset["hint"] = f"named a column that is not there: {hint.axis_column!r}"
            still_open.append(asset)
            continue

        fit = fits_schema(headers, asset.get("rows") or [],
                          prefer_axis=canonical_key(hint.axis_column))
        if not fit.fits or fit.orientation != "keyed":
            asset["hint"] = (f"{hint.axis_column!r} does not key the rows "
                             f"({fit.reason if not fit.fits else fit.orientation + ' fit'})")
            still_open.append(asset)
            continue

        asset["gate"] = fit.summary()
        asset["observations"] = _observation_payload(fit)
        asset["hint"] = f"axis named by the model: {hint.axis_column!r}"
        asset.pop("rows", None)
        (package["figures"] if asset.get("is_figure") else package["tables"]).append(asset)
        promoted += 1

    package["review"] = still_open
    report = package.setdefault("gate_report", {})
    report["review"] = len(still_open)
    report["review_promoted"] = promoted
    report["tables_accepted"] = len(package.get("tables", []))
    report["figures_accepted"] = len(package.get("figures", []))
    report["observations"] = sum(
        len(item.get("observations", []))
        for item in chain(package.get("tables", []), package.get("figures", [])))
    print(f"        review: {promoted} of {promoted + len(still_open)} assets keyed by the model")
    return promoted


GATE_DECISION_COLUMNS = ["kind", "index", "page", "caption", "verdict", "why",
                         "axis", "points", "observations"]
DECISION_CAPTION_CHARS = 60


def show_gate_decisions(silver_package: dict) -> pd.DataFrame:
    """Every asset and why it is in or out — the table to read when benchmarking."""
    def row(item, verdict, why, gate=None):
        gate = gate or {}
        return {
            "kind": item.get("kind") or ("table" if "cleaned_csv_path" in item else "figure"),
            "index": item.get("index") or item.get("table_index") or item.get("figure_index"),
            "page": item.get("page_number"),
            "caption": (item.get("caption") or "")[:DECISION_CAPTION_CHARS],
            "verdict": verdict,
            "why": why,
            "axis": gate.get("axis_label"),
            "points": len(gate.get("axis_points") or []),
            "observations": len(item.get("observations", [])),
        }

    rows = [row(item, "accepted", item.get("gate", {}).get("reason"), item.get("gate"))
            for item in chain(silver_package.get("tables", []),
                              silver_package.get("figures", []))]
    rows += [row(item, "reference", item.get("why"))
             for item in silver_package.get("references", [])]
    rows += [row(item, f"review: {item.get('hint') or 'unresolved'}", item.get("why"))
             for item in silver_package.get("review", [])]
    rows += [row(item, f"rejected @ {item['stage']}", item["reason"])
             for item in silver_package.get("rejected", [])]
    frame = pd.DataFrame(rows, columns=GATE_DECISION_COLUMNS)
    if not frame.empty:
        frame = frame.sort_values(["kind", "index"], na_position="last")
    display(frame)
    return frame

## Vocabularies

Data, not code. Four closed sets — everything reaching the database is one of these values or is flagged for review.

In [ ]:
from gold_schema import (
    EVIDENCE_METHODS, EVIDENCE_SOURCE_TYPES, FUNCTIONAL_CLASS_TIERS, FUNCTIONAL_CLASSES,
    INDICATOR_TYPES, INGREDIENT_SOURCES, UNCLASSIFIED_CLASS, UNKNOWN_SOURCE,
)

UNRESOLVED_INDICATOR, UNSPECIFIED_UNIT = "unresolved indicator", "unspecified"

from vocabulary import KINDS, load_yaml_vocabulary

VOCABULARY_PATH = PIPELINE_DIR / "vocabulary.yaml"
REVIEW_PATH = PROJECT_ROOT / "vocabulary_review.json"

SOURCE = load_yaml_vocabulary(VOCABULARY_PATH)

print(f"Vocabulary: {len(SOURCE.terms)} terms "
      + ", ".join(f"{len(SOURCE.of_kind(kind))} {kind}" for kind in KINDS)
      + f" | {len(SOURCE.units)} canonical units "
      f"| {len(SOURCE.discarded)} discarded quantities | from {VOCABULARY_PATH.name}")

## Silver: Normalisation

The only place names are resolved. Gold receives a package where the matrix, the arms, their concentrations, the indicator names, types and units, and every ingredient's class and origin are already decided.

In [ ]:
from collections import Counter

_CFU_UNIT = re.compile(r"\bcfu\b", re.IGNORECASE)
_DISCARDED = re.compile(
    "(?:^|_)(" + "|".join(sorted((canonical_key(name) for name in SOURCE.discarded),
                                 key=len, reverse=True)) + ")(?:_|$)")
_UNIT_CANONICAL = {canonical_key(alias): preferred
                   for preferred, aliases in SOURCE.units.items()
                   for alias in [preferred, *aliases]}
_KNOWN_UNIT_KEYS = {canonical_key(preferred) for preferred in SOURCE.units}


def canonical_unit(unit):
    text = str(unit or "").strip()
    return _UNIT_CANONICAL.get(canonical_key(text), text) if text else None


@dataclass(eq=False)
class Term:
    key: str
    name: str
    kind: str
    unit: Optional[str] = None
    threshold: Optional[float] = None
    functional_class: Optional[str] = None
    source: Optional[str] = None
    indicator_type: Optional[str] = None
    aliases: list = field(default_factory=list)


class Vocabulary:
    """Alias -> preferred term, and the ledger of what it changed."""

    def __init__(self):
        self._by_kind = {kind: {} for kind in KINDS}
        self.unresolved = {kind: {} for kind in KINDS}
        self.changes, self.flags = {}, {}
        self.discarded = Counter()
        self._pattern = {}

    def add(self, term: Term) -> None:
        table = self._by_kind.setdefault(term.kind, {})
        for key in [term.key, *(canonical_key(alias) for alias in term.aliases)]:
            taken = table.get(key)
            if taken is not None and taken.name != term.name:
                self.flag(term.kind, term.name, f"key {key!r} already claimed by {taken.name!r}")
            table[key] = term
        self._pattern.pop(term.kind, None)

    def _compile(self, kind):
        """One alternation per kind, longest alternative first so "aerobic plate count"
        beats "count". Per-alias regexes blow through Python's cache at this call rate."""
        keys = sorted((key for key in self._by_kind.get(kind, {}) if len(key) >= 2),
                      key=len, reverse=True)
        self._pattern[kind] = re.compile(
            "(?:^|_)(" + "|".join(re.escape(key) for key in keys) + ")(?:_|$)"
        ) if keys else None
        return self._pattern[kind]

    def pattern(self, kind):
        return self._pattern[kind] if kind in self._pattern else self._compile(kind)

    @classmethod
    def from_source(cls, source) -> "Vocabulary":
        vocabulary = cls()
        for entry in source.terms:
            vocabulary.add(Term(
                key=canonical_key(entry["name"]),
                name=entry["name"], kind=entry["kind"],
                unit=canonical_unit(entry.get("unit")),
                threshold=entry.get("threshold"),
                functional_class=entry.get("functional_class"), source=entry.get("source"),
                indicator_type=entry.get("indicator_type"), aliases=entry.get("aliases", []),
            ))
        return vocabulary

    def save_review(self, path: Path) -> None:
        path.write_text(json.dumps({
            "unresolved": {kind: sorted(names.values())
                           for kind, names in self.unresolved.items() if names},
            "flags": [{"kind": kind, "value": value, "reason": reason}
                      for (kind, value), reason in sorted(self.flags.items())],
            "discarded": [{"kind": kind, "value": value, "occurrences": count}
                          for (kind, value), count in sorted(self.discarded.items())],
        }, ensure_ascii=False, indent=2), encoding="utf-8")

    def resolve(self, text, kind) -> Optional[Term]:
        return self._by_kind.get(kind, {}).get(canonical_key(text))

    def names_in(self, text, kind) -> set:
        """Every known term of `kind` appearing in a phrase, longest match per span."""
        key = canonical_key(text)
        pattern = self.pattern(kind) if key else None
        if pattern is None:
            return set()
        table = self._by_kind[kind]
        return {table[match.group(1)] for match in pattern.finditer(key)}

    def find_in(self, text, kind) -> Optional[Term]:
        """Longest known term appearing anywhere in a phrase, for captions and prose."""
        key = canonical_key(text)
        pattern = self.pattern(kind) if key else None
        if pattern is None:
            return None
        best = max((match.group(1) for match in pattern.finditer(key)), key=len, default=None)
        return self._by_kind[kind][best] if best else None

    def terms_of(self, kind) -> dict:
        return self._by_kind.get(kind, {})

    def note_unresolved(self, text, kind) -> None:
        text = str(text or "").strip()
        if text:
            self.unresolved.setdefault(kind, {})[canonical_key(text)] = text

    def note_change(self, kind, raw, canonical, **extra) -> None:
        raw = str(raw or "").strip()
        if raw:
            self.changes[(kind, canonical_key(raw))] = {
                "kind": kind, "raw": raw, "canonical": canonical, **extra}

    def flag(self, kind, value, reason) -> None:
        value = str(value or "").strip()
        if value:
            self.flags[(kind, value)] = reason

    def normalise_ingredient(self, raw) -> Optional[dict]:
        """name + functional_class + source together: one decision, not three. An unknown
        substance returns None and is parked under `review`."""
        text = str(raw or "").strip()
        term = self.resolve(text, "ingredient")
        if term is None:
            self.note_unresolved(text, "ingredient")
            self.discarded[("ingredient", text)] += 1
            return None
        klass = term.functional_class if term.functional_class in FUNCTIONAL_CLASSES else None
        source = term.source if term.source in INGREDIENT_SOURCES else None
        for field_name, value in (("functional_class", klass), ("source", source)):
            if value is None:
                self.flag("ingredient", term.name, f"{field_name} outside its vocabulary")
        resolved = {"name": term.name, "functional_class": klass or UNCLASSIFIED_CLASS,
                    "source": source or UNKNOWN_SOURCE}
        self.note_change("ingredient", text, term.name, **{
            k: v for k, v in resolved.items() if k != "name"})
        return resolved

    def normalise_indicator(self, raw_label) -> Optional[dict]:
        """The unit is split off and canonicalised before anything is matched, then the
        vocabulary's unit wins. None means a discarded quantity and the caller drops the
        observation. An unknown indicator, unlike an unknown ingredient, keeps the paper's
        wording and is flagged — a measurement nobody has named is still a measurement."""
        raw_name, raw_unit = split_label_and_unit(raw_label)
        unit = canonical_unit(raw_unit)
        if not raw_name:
            return {"name": UNRESOLVED_INDICATOR, "indicator_type": "chemical",
                    "unit": UNSPECIFIED_UNIT, "threshold": None}
        if _DISCARDED.search(canonical_key(raw_name)):
            self.discarded[("indicator", raw_name)] += 1
            return None
        term = self.resolve(raw_name, "indicator")
        unit = (term.unit if term and term.unit else unit) or UNSPECIFIED_UNIT
        inferred = "microbial" if _CFU_UNIT.search(unit) else "chemical"
        if term is None:
            self.note_unresolved(raw_name, "indicator")
            self.flag("indicator", raw_name, f"not in vocabulary; type inferred {inferred!r}")
            return {"name": raw_name, "indicator_type": inferred, "unit": unit, "threshold": None}
        indicator_type = term.indicator_type
        if indicator_type not in INDICATOR_TYPES:
            indicator_type = inferred
            self.flag("indicator", term.name, f"no valid indicator_type; inferred {inferred!r}")
        self.note_change("indicator", raw_label, term.name,
                         indicator_type=indicator_type, unit=unit)
        return {"name": term.name, "indicator_type": indicator_type, "unit": unit,
                "threshold": term.threshold}


_LABEL_SEPARATORS = (" - ", " – ", " — ", ": ", " | ", " _ ")
_OCR_DIGITS = {"0": "o", "1": "l", "5": "s"}
_OCR_IN_WORD = re.compile(r"(?<=[A-Za-z])[015](?=[A-Za-z]|$|[+/&\s])")


def unfold_ocr_digits(text):
    return _OCR_IN_WORD.sub(lambda match: _OCR_DIGITS[match.group()], text)


@dataclass
class Treatment:
    """One arm. `ingredients` is the additive dimension — empty means a control — and
    `condition` is whatever else the label names: the factor a study crosses with its
    doses, "Vacuum" in "Vacuum + 1% TEO"."""

    key: str
    label: str
    condition: Optional[str]
    ingredients: list = field(default_factory=list)

    @property
    def dosed(self):
        return bool(self.ingredients)


def _dosed_ingredients(dose: Dose, vocabulary) -> tuple:
    """(substances the label names with their concentration, the parts that named none).
    The second is the arm's condition: no vocabulary decides it, so a factorial dimension
    nobody thought to seed still separates two arms."""
    if not (dose and dose.substance):
        return [], []
    found, unmatched = [], []
    for part in re.split(r"\s*[+&]\s*", unfold_ocr_digits(dose.substance)):
        part = part.strip(" -_,")
        if not part:
            continue
        resolved = vocabulary.normalise_ingredient(part) if vocabulary else None
        if resolved:
            found.append({**resolved, "amount": dose.amount, "unit": dose.unit})
        else:
            unmatched.append(part)
    return found, unmatched


def _dose_key(ingredients):
    """What separates one rung of a dose ladder from the next."""
    return "|".join(f"{item['name']}={item['amount']}{item['unit']}"
                    for item in sorted(ingredients, key=lambda item: item["name"]))


def parse_treatment(label, vocabulary=None) -> Treatment:
    """Read one arm: its substances, their concentrations, and its condition. The dose
    rides on each ingredient rather than entering a name, so it lands in `concentration`.
    The condition is in the key because two arms sharing a dose ladder and differing only
    in how they were packed are two arms."""
    text = _DUPLICATE_TAIL.sub("", str(label or "").strip())
    dose = parse_dosed_label(text)
    ingredients, unmatched = _dosed_ingredients(dose, vocabulary)
    condition = " ".join(unmatched) or None
    if vocabulary is not None and dose and not ingredients:
        vocabulary.flag("ingredient", text,
                        "dosed arm whose substance is not in the vocabulary — reads as a "
                        "control, and the substance is keying the arm as a condition")
    substances = " + ".join(item["name"] for item in ingredients)
    return Treatment(
        key=canonical_key(f"{condition or ''}|{substances}|{_dose_key(ingredients)}")
            or canonical_key(text) or "unspecified",
        label=text, condition=condition, ingredients=ingredients)


def _has_separator(label):
    return any(separator in label for separator in _LABEL_SEPARATORS)


def split_indicator_and_treatment(label, lexicon):
    """Split "PBC (log cfu/g) - Control1" when the tail is a known arm."""
    text = str(label or "").strip()
    for separator in _LABEL_SEPARATORS:
        if separator in text:
            head, _, tail = text.rpartition(separator)
            if canonical_key(tail) in lexicon:
                return head.strip(), tail.strip()
    return text, None


def _candidate_columns(asset):
    """Labels that could name an arm, in two groups: value column headers, and the row
    labels a keyed table leaves behind. They stay apart because `_collect_arms` treats a
    dose anywhere in a group as evidence about the whole group.
    """
    gate = asset.get("gate") or {}
    axis = {canonical_key(part) for part in str(gate.get("axis_label") or "").split("|")}

    def keep(names):
        return [name for name in names if canonical_key(name) not in axis]

    headers = keep(str(name) for name in gate.get("value_columns", []))
    labels, seen = [], set()
    for observation in asset.get("observations", []):
        for value in (observation.get("row_labels") or {}).values():
            if value not in seen:
                seen.add(value)
                labels.append(str(value))
    return [headers, keep(labels)]


def _collect_arms(columns_per_asset, vocabulary, parsed):
    """An arm carries a dose, or stands beside one. Recurrence alone is not enough: a
    paper measuring pH in two tables makes "pH" recur exactly like an arm."""
    arms, dosed, siblings = {}, set(), set()
    for columns in columns_per_asset:
        keyed = []
        for name in columns:
            key = canonical_key(name)
            if key not in parsed:
                parsed[key] = parse_treatment(name, vocabulary)
            keyed.append((key, parsed[key]))
        has_dose = any(arm.dosed for _, arm in keyed)
        for key, arm in keyed:
            arms.setdefault(key, arm)
            if arm.dosed:
                dosed.add(key)
            elif has_dose:
                siblings.add(key)
    return {key: arm for key, arm in arms.items() if key in dosed or key in siblings}


def build_treatment_lexicon(assets, vocabulary=None) -> dict:
    """The paper's arms, keyed by the label as printed. Plain headers first, then
    "PBC (log cfu/g) - Teo1%" unpacked using them — otherwise that reads as one arm per
    indicator, with the unit inside a substance name."""
    columns = [group for asset in assets for group in _candidate_columns(asset)]
    parsed: dict = {}
    lexicon = _collect_arms(
        [[name for name in group if not _has_separator(name)] for group in columns],
        vocabulary, parsed)
    lexicon.update(_collect_arms([
        [split_indicator_and_treatment(name, lexicon)[1] or name
         for name in group if _has_separator(name)]
        for group in columns], vocabulary, parsed))
    return lexicon


_CAPTION_PREFIX = re.compile(r"^\s*(?:fig(?:ure)?|table|scheme|chart)\s*\.?\s*\d+\s*[.:)-]*\s*",
                             re.IGNORECASE)

SECTION_SCAN_CHARS = env_int("SECTION_SCAN_CHARS", "2000")
EVIDENCE_LABEL_CHARS = 80


def _paper_texts(package):
    """Title, methods and captions, in document order."""
    for section in package.get("sections", []):
        where = f"section:{section['section_title']}"[:EVIDENCE_LABEL_CHARS]
        yield where, section["section_title"]
        yield where, section["content_markdown"][:SECTION_SCAN_CHARS]
    for asset in chain(package.get("tables", []), package.get("figures", [])):
        if asset.get("caption"):
            yield asset.get("docling_item_ref") or "caption", asset["caption"]


def resolve_matrix(package, vocabulary) -> dict:
    """The food the paper studied, from the first text that names one."""
    for where, text in _paper_texts(package):
        term = vocabulary.find_in(text, "matrix")
        if term:
            return {"name": term.name, "evidence": where}
    return {"name": None, "evidence": None}


def caption_indicator(caption, vocabulary) -> Optional[str]:
    term = vocabulary.find_in(_CAPTION_PREFIX.sub("", str(caption or "").strip()), "indicator")
    return term.name if term else None


def caption_is_discarded(caption, vocabulary) -> bool:
    """The whole asset measures something we do not keep. Only stands when nothing in the
    caption is a known indicator: "Changes in Hardness, pH, ... and APC" names both kinds,
    and dropping it would cost five columns to avoid one.
    """
    text = _CAPTION_PREFIX.sub("", str(caption or "").strip())
    if not _DISCARDED.search(canonical_key(text)):
        return False
    return caption_indicator(text, vocabulary) is None


def context_indicator(context, vocabulary) -> Optional[str]:
    """For a captionless figure, and only when the surrounding text names exactly one
    known quantity: picking one of several would misattribute real numbers."""
    found = {term.name for term in vocabulary.names_in(context, "indicator")}
    return found.pop() if len(found) == 1 else None


@dataclass(slots=True)
class Measurement:
    axis_value: float
    condition: Optional[str]
    arm_key: str
    indicator: str
    indicator_type: str
    unit: str
    threshold: Optional[float]
    value: float
    is_figure: bool
    item_ref: Optional[str]
    page_number: Optional[int]
    reports_mean: bool = False
    ingredients: list = field(default_factory=list)


def _assign_roles(observation, lexicon, indicator_columns, strategies):
    """Which label names the quantity and which names the arm."""
    column = observation.get("column_label")
    indicator_label = treatment_label = None
    if column:
        head, tail = split_indicator_and_treatment(column, lexicon)
        if tail is not None:
            indicator_label, treatment_label = head, tail
            strategies.add("indicator and arm packed into the column header")
        elif canonical_key(column) in lexicon:
            treatment_label = column
            strategies.add("value columns are arms")
        else:
            indicator_label = column
            strategies.add("value columns are indicators")
    for name, value in (observation.get("row_labels") or {}).items():
        if canonical_key(value) in lexicon or canonical_key(name) in lexicon:
            treatment_label = treatment_label or value
            strategies.add("row labels are arms")
        elif name in indicator_columns and indicator_label is None:
            indicator_label = value
            strategies.add("row labels name the indicator")
        elif treatment_label is None:
            treatment_label = value
            strategies.add("row labels qualify the arm")
    return indicator_label, treatment_label


CAPTION_SUMMARY_CHARS = 60


def _asset_note(item_ref, caption, how, measurements=()) -> dict:
    """What one asset contributed, in the shape `show_normalization_report` reads."""
    return {
        "item_ref": item_ref,
        "caption": caption,
        "how": how,
        "indicators": sorted({m.indicator for m in measurements}),
        "conditions": sorted({m.condition for m in measurements if m.condition}),
        "arms": len({m.arm_key for m in measurements}),
    }


def _named_by_text(asset, vocabulary, caption_name, claimed) -> tuple:
    """The indicator the asset's own text names, and whether that came from prose rather
    than from the caption."""
    if caption_name is not None:
        return caption_name, False
    guess = context_indicator(asset.get("context_markdown"), vocabulary)
    if guess and canonical_key(guess) not in claimed:
        return guess, True
    return None, False


def interpret_asset(asset, lexicon, vocabulary, *, caption_name=None, claimed=()):
    gate = asset.get("gate") or {}
    observations = asset.get("observations", [])
    if caption_is_discarded(asset.get("caption"), vocabulary):
        name = _CAPTION_PREFIX.sub("", str(asset.get("caption")).strip())[:CAPTION_SUMMARY_CHARS]
        vocabulary.discarded[("indicator", name)] = len(observations)
        return [], _asset_note(
            asset.get("docling_item_ref"), name,
            ["discarded: caption names a sensory or gravimetric quantity"])

    caption_name, from_context = _named_by_text(asset, vocabulary, caption_name, claimed)
    labels = [str(name) for name in gate.get("label_columns", [])]
    indicator_columns = [name for name in labels if canonical_key(name) not in lexicon]
    is_figure = bool(asset.get("is_figure"))
    item_ref = asset.get("docling_item_ref")
    page_number = asset.get("page_number")
    caption_means = declares_mean(asset.get("caption"))
    measurements, strategies, dropped = [], set(), 0

    for observation in observations:
        indicator_label, treatment_label = _assign_roles(
            observation, lexicon, indicator_columns, strategies)
        if indicator_label is None and caption_name:
            indicator_label = caption_name
            strategies.add("indicator inferred from nearby text" if from_context
                           else "indicator taken from the caption")
        resolved = vocabulary.normalise_indicator(indicator_label)
        if resolved is None:
            dropped += 1
            continue
        arm = lexicon.get(canonical_key(treatment_label)) if treatment_label else None
        measurements.append(Measurement(
            axis_value=float(observation["axis_value"]),
            condition=arm.condition if arm else None,
            arm_key=arm.key if arm else canonical_key(treatment_label or "unspecified"),
            indicator=resolved["name"], indicator_type=resolved["indicator_type"],
            unit=resolved["unit"], threshold=resolved["threshold"],
            value=float(observation["value"]), is_figure=is_figure,
            item_ref=item_ref, page_number=page_number,
            reports_mean=bool(observation.get("reports_mean")) or caption_means,
            ingredients=arm.ingredients if arm else [],
        ))

    if dropped:
        strategies.add(f"{dropped} observations of discarded indicators dropped")
    return measurements, _asset_note(
        item_ref, (asset.get("caption") or "")[:EVIDENCE_LABEL_CHARS],
        sorted(strategies) or ["no labels to resolve"], measurements)


def interpret_paper(assets, lexicon, vocabulary):
    """Captions are read across the paper first: an indicator another figure claimed by
    caption is evidence against a captionless one guessing the same name from prose."""
    caption_names = [caption_indicator(asset.get("caption"), vocabulary) for asset in assets]
    claimed = {canonical_key(name) for name in caption_names if name}
    measurements, notes = [], []
    for asset, caption_name in zip(assets, caption_names):
        found, note = interpret_asset(asset, lexicon, vocabulary,
                                      caption_name=caption_name, claimed=claimed)
        measurements.extend(found)
        notes.append(note)
    return measurements, notes


_PROPORTION_UNITS = ("%", "peak area %", "area %", "w/w", "v/v", "g/100g")


def _column_is_numeric(rows, index):
    """The same test the gate applies to a column, over raw rows rather than a fit."""
    filled = [row[index] for row in rows
              if index < len(row) and row[index] is not None and str(row[index]).strip()]
    if not filled:
        return False
    parsed = sum(parse_number(cell) is not None for cell in filled)
    return parsed / len(filled) >= MIN_NUMERIC_RATIO


def _proportion_unit(header) -> Optional[str]:
    """A unit, or nothing. The bracket at the end of a header is not always one:
    "Concentration (mean ± Stdev)" gave `concentration_unit = "mean ± Stdev"`."""
    _, unit = split_label_and_unit(header)
    candidate = canonical_unit(unit) if unit else None
    if candidate and canonical_key(candidate) in _KNOWN_UNIT_KEYS:
        return candidate
    return next((token for token in _PROPORTION_UNITS
                 if token in str(header or "").lower()), None)


_CATALOGUE_TOTALS = {"total", "sum", "others", "other", "unknown"}


def ingredients_from_reference(asset, vocabulary) -> list:
    """A composition table's rows are substances, so they belong in `ingredients`. Only
    rows the vocabulary can name survive. The column group prefix ("TEO.Concentration")
    is kept as the preparation; `source` is the origin, and comes from the vocabulary."""
    headers = [str(name) for name in asset.get("headers", [])]
    rows = asset.get("rows") or []
    if not headers or not rows:
        return []
    numeric = [_column_is_numeric(rows, index) for index in range(len(headers))]
    name_columns = [index for index, flag in enumerate(numeric) if not flag]
    amount_columns = [index for index, name in enumerate(headers)
                      if numeric[index]
                      and any(token in name.lower() for token in _PROPORTION_UNITS)]
    caption = asset.get("caption")
    item_ref = asset.get("docling_item_ref")
    found = []
    for name_index in name_columns:
        header = headers[name_index]
        prefix = header.split(".")[0].strip() if "." in header else None
        partner = next((index for index in amount_columns
                        if prefix and headers[index].startswith(prefix + ".")),
                       amount_columns[0] if amount_columns else None)
        unit = _proportion_unit(headers[partner]) if partner is not None else None
        preparation = prefix or caption or "composition table"
        for row in rows:
            raw = str(row[name_index] or "").strip() if name_index < len(row) else ""
            if not raw or canonical_key(raw) in _CATALOGUE_TOTALS:
                continue
            resolved = vocabulary.normalise_ingredient(raw)
            if resolved is None:
                continue
            found.append({
                **resolved,
                "preparation": preparation,
                "amount": parse_number(row[partner]) if partner is not None and partner < len(row) else None,
                "unit": unit,
                "item_ref": item_ref,
            })
    return found


def normalise_silver(package: dict, vocabulary=None) -> dict:
    """Resolve one gated package in place: the matrix, the arms and their concentrations,
    the indicator names, types and units, every ingredient's class and origin. Everything
    is decided here — Gold assembles records and resolves nothing."""
    vocabulary = vocabulary or VOCABULARY
    assets = package.get("tables", []) + package.get("figures", [])
    lexicon = build_treatment_lexicon(assets, vocabulary)
    measurements, notes = interpret_paper(assets, lexicon, vocabulary)
    reading = {
        "matrix": resolve_matrix(package, vocabulary),
        "catalogue": [item for reference in package.get("references", [])
                      for item in ingredients_from_reference(reference, vocabulary)],
        "lexicon": lexicon, "measurements": measurements, "notes": notes,
    }
    package["reading"] = reading
    return reading


VOCABULARY = Vocabulary.from_source(SOURCE)
print(f"Vocabulary: {sum(len(VOCABULARY.terms_of(k)) for k in KINDS)} lookup keys "
      f"over {len(SOURCE.terms)} terms")

## Gold

Schema-first assembly: every record is validated before SQLite sees it, and free-form model output never writes directly. The local model is asked only for the two fields that exist solely in prose.

In [ ]:
from pydantic import ValidationError

from gold_schema import (
    AssetHint, EvidenceSpan, ExperimentIngredientRecord, ExperimentRecord, FigureDocument,
    GoldBundle, IndicatorRecord, IngredientRecord, MeasurementRecord, PaperDocument,
    ProtocolRecord, SectionDocument, SharedProtocol, TableDocument,
)

GOLD_JSON_SCHEMA = GoldBundle.model_json_schema()

_RETRYABLE_STATUS = range(500, 600)
HTTP_REASON_CHARS = 300


REPLY_SNIPPET_CHARS = 200


def _outermost_object(text: str):
    """The first `{` to the last `}`, parsed. A reasoning model that narrates around its
    answer still emits exactly one object, and this finds it."""
    opened, closed = text.find("{"), text.rfind("}")
    if opened < 0 or closed <= opened:
        return None
    try:
        return json.loads(text[opened:closed + 1])
    except ValueError:
        return None


def _reply_object(response: dict) -> dict:
    """The answer, from wherever the model actually put it.

    `json.loads(content)` alone was fatal on a reply of two newlines: generation had ended
    before any JSON was emitted and the paper died on a JSONDecodeError that said nothing
    about why. Content first, then an object embedded in it, then the reasoning channel —
    and if all three fail, an error carrying the counters that explain it.
    """
    message = response.get("message") or {}
    content = message.get("content") or ""
    thinking = message.get("thinking") or ""

    for candidate in (content.strip(), content, thinking):
        if not candidate.strip():
            continue
        try:
            parsed = json.loads(candidate)
        except ValueError:
            parsed = _outermost_object(candidate)
        if isinstance(parsed, dict):
            if candidate is thinking:
                print("        the answer came back in the reasoning channel, not in "
                      "content — recovered")
            return parsed

    raise RuntimeError(
        "Ollama returned no JSON object: "
        f"done_reason={response.get('done_reason')!r}, "
        f"eval_count={response.get('eval_count')}, "
        f"prompt_eval_count={response.get('prompt_eval_count')}, "
        f"thinking={len(thinking)} chars, content={content[:REPLY_SNIPPET_CHARS]!r}. "
        "An empty reply with think on usually means generation was cut off before the "
        "answer: raise OLLAMA_NUM_PREDICT or unset it, or lower MAX_PROMPT_CHARS.")


def _http_reason(exc: urllib.error.HTTPError) -> str:
    """Ollama says which field it rejected in the response body, and `str(HTTPError)` is
    only "HTTP Error 400: Bad Request" — which names nothing."""
    try:
        body = exc.read().decode("utf-8", "replace").strip()
    except (OSError, ValueError):
        body = ""
    try:
        body = json.loads(body).get("error") or body
    except ValueError:
        pass
    return f"{exc.reason} — {body[:HTTP_REASON_CHARS]}" if body else str(exc.reason)


def load_cached_completion(key: str) -> Optional[dict]:
    """A reasoning call is the most expensive thing in the pipeline and the most likely to
    be repeated: a hundreds-of-paper run gets restarted, and without this every restart
    re-pays every call it had already answered."""
    if not CACHE_ENABLED:
        return None
    path = GOLD_CACHE_ROOT / f"{key}.json"
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


def store_cached_completion(key: str, payload: dict, prompt_tokens: int) -> None:
    if not CACHE_ENABLED:
        return
    path = GOLD_CACHE_ROOT / f"{key}.json"
    temporary = path.with_suffix(f".{os.getpid()}.{threading.get_ident()}.tmp")
    temporary.write_text(
        json.dumps({"payload": payload, "prompt_tokens": prompt_tokens},
                   ensure_ascii=False, default=str), encoding="utf-8")
    os.replace(temporary, path)


class OllamaJSONClient:
    """The pipeline's only model call, to a local Ollama server over its native API.
    Stdlib only: no SDK, no key, nothing to point at a hosted provider by accident.

    Safe to share between threads: it holds no per-call state, and one server with
    several slots is how two Gold calls run at once — two servers would each load their
    own copy of the weights.
    """

    def __init__(self, *, host=OLLAMA_HOST, model=OLLAMA_MODEL, json_schema=GOLD_JSON_SCHEMA):
        self.url = f"{host.rstrip('/')}/api/chat"
        self.host = host.rstrip("/")
        self.model = model
        self.json_schema = json_schema

    @property
    def options(self) -> dict:
        """`num_predict` bounds the whole generation, reasoning included, so a cap below
        the model's natural trace length ends the call before any JSON is emitted and the
        reply comes back as whitespace. It is sent only when it has been set deliberately;
        unset, `num_ctx` still bounds a spiral at the context window.

        Size it from the `eval_count` values in STAGE_TIMINGS, at roughly p95 + 25%. That
        measurement has to come first: a number picked ahead of it cost three papers.
        """
        options = {"temperature": 0.0, "num_ctx": OLLAMA_NUM_CTX}
        if OLLAMA_NUM_PREDICT > 0:
            options["num_predict"] = OLLAMA_NUM_PREDICT
        return options

    @property
    def response_format(self):
        """A JSON Schema switches on grammar-constrained decoding: invalid structure
        becomes unreachable and generation stops as the object closes. Off until
        `verify_schema_format()` confirms the grammar is not applied to the reasoning
        channel too, which mangles the answer on some builds."""
        return self.json_schema if (OLLAMA_JSON_SCHEMA and self.json_schema) else "json"

    def _post(self, body: dict, *, timeout=OLLAMA_TIMEOUT, retries=OLLAMA_RETRIES) -> dict:
        """One chat call, retried on transport failures and 5xx with exponential backoff
        and jitter. Without this a single timeout at paper 180 discards 180 papers of
        vision work."""
        encoded = json.dumps(body).encode("utf-8")
        last = None
        for attempt in range(1, max(1, retries) + 1):
            request = urllib.request.Request(
                self.url, data=encoded, headers={"Content-Type": "application/json"})
            try:
                with urllib.request.urlopen(request, timeout=timeout) as response:
                    return json.load(response)
            except urllib.error.HTTPError as exc:
                if exc.code not in _RETRYABLE_STATUS:
                    raise RuntimeError(
                        f"Ollama refused the request ({exc.code}): {_http_reason(exc)}"
                    ) from exc
                last = exc
            except (urllib.error.URLError, TimeoutError) as exc:
                last = exc
            if attempt < retries:
                delay = OLLAMA_RETRY_BACKOFF ** (attempt - 1) * (1.0 + random.random())
                print(f"        ollama call failed ({last}); retry {attempt}/{retries - 1} "
                      f"in {delay:.1f}s")
                time.sleep(delay)
        raise RuntimeError(
            f"Ollama unreachable at {self.url} after {retries} attempt(s): {last}. "
            "Start it with `ollama serve`.")

    def _record(self, payload: dict, paper, scope) -> int:
        """Every counter the server already returns, so `num_ctx` and `num_predict` can be
        sized from data instead of habit."""
        prompt_tokens = int(payload.get("prompt_eval_count") or 0)
        eval_tokens = int(payload.get("eval_count") or 0)
        eval_ns = int(payload.get("eval_duration") or 0)
        record_metric(
            "llm", paper, scope=scope,
            seconds=round(int(payload.get("total_duration") or 0) / 1e9, 3),
            prompt_eval_count=prompt_tokens, eval_count=eval_tokens,
            eval_seconds=round(eval_ns / 1e9, 3),
            tokens_per_second=round(eval_tokens / (eval_ns / 1e9), 2) if eval_ns else None,
            done_reason=payload.get("done_reason"))
        if payload.get("done_reason") == "length":
            print("        WARNING: generation was cut off at the token limit — the reply "
                  "is truncated, raise OLLAMA_NUM_PREDICT or lower MAX_PROMPT_CHARS")
        if prompt_tokens >= OLLAMA_NUM_CTX:
            print(f"        WARNING: the prompt filled all {OLLAMA_NUM_CTX} context "
                  "tokens — the methods prose at its head was dropped")
        return prompt_tokens

    def json_completion(self, *, system_prompt, user_prompt, cache_scope=None,
                        paper=None) -> tuple:
        """Returns (parsed JSON, prompt tokens the server actually read).

        Cached on the prompts themselves, so the key versions itself: editing a prompt or
        a model option invalidates exactly the answers it could have changed.
        """
        options = self.options
        response_format = self.response_format
        key = content_key(cache_scope or "call", self.model, system_prompt, user_prompt,
                          json.dumps(options, sort_keys=True), OLLAMA_THINK,
                          "schema" if response_format != "json" else "json")
        if cache_scope:
            cached = load_cached_completion(key)
            if cached is not None:
                record_metric("llm_cached", paper, seconds=0.0, scope=cache_scope)
                print(f"        {cache_scope}: reply reused from cache")
                return cached["payload"], int(cached.get("prompt_tokens") or 0)

        response = self._post({
            "model": self.model, "stream": False, "format": response_format,
            "think": OLLAMA_THINK, "keep_alive": OLLAMA_KEEP_ALIVE, "options": options,
            "messages": [{"role": "system", "content": system_prompt},
                         {"role": "user", "content": user_prompt}],
        })
        prompt_tokens = self._record(response, paper, cache_scope)
        parsed = _reply_object(response)
        if cache_scope:
            store_cached_completion(key, parsed, prompt_tokens)
        return parsed, prompt_tokens

    def unload(self) -> None:
        """Drop the model now rather than at the end of its keep-alive window. Not part of
        a run any more — the weights are meant to stay resident — but still the way to
        hand the card back when the notebook is done with it."""
        try:
            self._post({"model": self.model, "messages": [], "stream": False,
                        "keep_alive": 0}, timeout=OLLAMA_PROBE_TIMEOUT, retries=1)
        except RuntimeError as exc:
            print(f"        could not unload {self.model}: {exc}")


RESIDENCY_COLUMNS = ["model", "size_gb", "vram_gb", "processor"]


def report_model_residency(host=OLLAMA_HOST) -> pd.DataFrame:
    """What `ollama ps` prints, over the API. The single most important number in this
    notebook: anything short of 100% GPU means layers are on the CPU and decode has
    fallen from ~20-25 tok/s to ~3-5, which with reasoning on turns a 2 minute call into
    10. Lower OLLAMA_NUM_CTX until this reads 100%."""
    try:
        with urllib.request.urlopen(f"{host.rstrip('/')}/api/ps",
                                    timeout=OLLAMA_PROBE_TIMEOUT) as response:
            models = json.load(response).get("models", [])
    except (urllib.error.URLError, TimeoutError, ValueError) as exc:
        print(f"Could not read {host}/api/ps: {exc}")
        return pd.DataFrame(columns=RESIDENCY_COLUMNS)

    rows = []
    for entry in models:
        size = int(entry.get("size") or 0)
        vram = int(entry.get("size_vram") or 0)
        share = vram / size if size else 0.0
        rows.append({"model": entry.get("name") or entry.get("model"),
                     "size_gb": round(size / 1e9, 2), "vram_gb": round(vram / 1e9, 2),
                     "processor": f"{share:.0%} GPU" if share < 0.999 else "100% GPU"})
    frame = pd.DataFrame(rows, columns=RESIDENCY_COLUMNS)
    display(frame)
    for row in rows:
        if row["processor"] != "100% GPU":
            print(f"  {row['model']} is only {row['processor']}: lower OLLAMA_NUM_CTX, or "
                  "set OLLAMA_FLASH_ATTENTION=1 and OLLAMA_KV_CACHE_TYPE=q8_0 on the server.")
    return frame


SCHEMA_PROBE_PROMPT = ('Return the JSON object {"paper": {"title": "probe"}} and nothing '
                       'else.')


def verify_schema_format(client=None) -> bool:
    """Whether this Ollama build can take a JSON Schema as `format` while `think` is on.

    Grammar-constrained decoding and a reasoning channel interact: where the grammar is
    applied to the thinking tokens as well, the reply comes back mangled. One call
    answers it. Run this before setting OLLAMA_JSON_SCHEMA=1.
    """
    client = client or OllamaJSONClient()
    response = client._post({
        "model": client.model, "stream": False, "format": GOLD_JSON_SCHEMA,
        "think": OLLAMA_THINK, "keep_alive": OLLAMA_KEEP_ALIVE,
        "options": {"temperature": 0.0, "num_ctx": OLLAMA_NUM_CTX, "num_predict": 512},
        "messages": [{"role": "user", "content": SCHEMA_PROBE_PROMPT}],
    })
    message = response.get("message") or {}
    thinking = (message.get("thinking") or "").strip()
    try:
        json.loads(message.get("content") or "")
        content_ok = True
    except ValueError:
        content_ok = False

    reasoning_ok = bool(thinking) or not OLLAMA_THINK
    if content_ok and reasoning_ok:
        print("Schema format is safe here: valid JSON content and an intact reasoning "
              "channel. Set OLLAMA_JSON_SCHEMA=1.")
    else:
        print("Schema format is NOT safe on this build "
              f"(valid content: {content_ok}, reasoning channel populated: {bool(thinking)}). "
              "Leave OLLAMA_JSON_SCHEMA=0 and rely on the num_predict bound.")
    return content_ok and reasoning_ok


UNATTRIBUTED_MATRIX = "unattributed"

MAX_METHODS_SECTION_CHARS = env_int("MAX_METHODS_SECTION_CHARS", "12000")
MAX_OTHER_SECTION_CHARS = env_int("MAX_OTHER_SECTION_CHARS", "800")


def build_asset_prompt(asset, kind) -> dict:
    """What the model is told an asset is, never what is in it.

    The reply carries `treatment` and `weight_g` out of METHODS prose and nothing else, so
    the values themselves are dead weight here: one paper sent 480 observations across 11
    assets, about 70% of a prompt that then did not fit the context window. What is left
    is enough to cite the asset and to know what it measured.
    """
    gate = asset.get("gate", {})
    return {
        "kind": kind, "docling_item_ref": asset.get("docling_item_ref"),
        "page_number": asset.get("page_number"), "caption": asset.get("caption"),
        "section_hint": asset.get("section_hint"), "axis_label": gate.get("axis_label"),
        "axis_points": gate.get("axis_points"), "value_columns": gate.get("value_columns"),
        "label_columns": gate.get("label_columns"), "orientation": gate.get("orientation"),
        "values_are_approximate": kind == "figure",
        "observation_count": len(asset.get("observations", [])),
        "nearby_text": (asset.get("context_markdown") or "")[:FIGURE_CONTEXT_CHARS],
    }


_WHITESPACE = re.compile(r"\s+")


def _searchable(text) -> str:
    """Case- and whitespace-folded, so a quote is not failed for rewrapped lines."""
    return _WHITESPACE.sub(" ", str(text or "")).strip().lower()


def reference_text_index(package: dict) -> dict:
    """{docling_item_ref: searchable text} over exactly the items the prompt shows, so a
    ref that fails to resolve here is one the model was never given."""
    index = {}
    for section in package.get("sections", []):
        ref = section.get("docling_item_ref")
        if ref:
            index[ref] = _searchable(
                f"{section.get('section_title', '')}\n{section.get('content_markdown', '')}")
    for asset in chain(package.get("tables", []), package.get("references", []),
                       package.get("figures", [])):
        ref = asset.get("docling_item_ref")
        if not ref:
            continue
        parts = [asset.get("caption") or "", asset.get("context_markdown") or "",
                 asset.get("preview_markdown") or ""]
        parts += [f"{item.get('column_label')} {item.get('row_labels')} "
                  f"{item.get('axis_value')} {item.get('value')}"
                  for item in asset.get("observations", [])]
        index[ref] = _searchable("\n".join(parts))
    return index


STATED_VERBATIM, STATED_UNQUOTED = 1.0, 0.5
METHOD_SCORES = {"derived": 0.7, "inferred": 0.3}
CHART_READING_PENALTY = 0.6


def score_evidence(span, index: dict) -> float:
    """Grade one span. Confidence is an evaluation, never a number the model chose:
    whether its quote is in the item it cited is testable, so that tops the scale and
    everything else is placed by distance from it."""
    resolved = index.get(span.docling_item_ref)
    if resolved is None:
        return 0.0
    if span.method == "stated":
        quote = _searchable(span.exact_text)
        score = STATED_VERBATIM if quote and quote in resolved else STATED_UNQUOTED
    else:
        score = METHOD_SCORES[span.method]
    if span.value_is_approximate or span.source_type == "figure":
        score *= CHART_READING_PENALTY
    return round(score, 3)


def score_bundle_evidence(bundle: GoldBundle, index: dict) -> GoldBundle:
    """Overwrite every span's confidence with a computed one. Mutates, and returns."""
    for record in bundle.experiments:
        for span in record.evidence:
            span.confidence = score_evidence(span, index)
    return bundle


def _as_day(axis_value) -> Optional[int]:
    try:
        number = float(axis_value)
    except (TypeError, ValueError):
        return None
    return int(round(number)) if number >= 0 and abs(number - round(number)) <= 1e-6 else None


def _add_ingredients(record, ingredients) -> None:
    known = {item.ingredient_name for item in record.ingredients}
    for item in ingredients:
        if item["name"] in known:
            continue
        known.add(item["name"])
        record.ingredients.append(IngredientRecord(
            ingredient_name=item["name"], functional_class=item["functional_class"],
            source=item["source"]))
        record.experiment_ingredients.append(ExperimentIngredientRecord(
            ingredient_name=item["name"], concentration=item.get("amount"),
            concentration_unit=item.get("unit")))


def _collapse_replicates(readings) -> tuple:
    """One value, and how many readings it stands for. Where a group holds both
    replicates and a stated mean, only the stated means are kept: folding one in beside
    the replicates behind it would weight them twice.
    """
    stated = [value for value, reports_mean in readings if reports_mean]
    kept = stated or [value for value, _ in readings]
    return sum(kept) / len(kept), len(kept)


def build_experiments(measurements, matrix) -> tuple:
    """One record per arm, keyed on `arm_key` — Silver built it from the arm's substances
    and their amounts, and nothing else separates one rung of a dose ladder from the next.
    `treatment` is left unset on purpose: it is prose the gate never reads.
    """
    grouped, off_axis = {}, 0
    for measurement in measurements:
        day = _as_day(measurement.axis_value)
        if day is None:
            off_axis += 1
            continue
        state = grouped.get(measurement.arm_key)
        if state is None:
            record = ExperimentRecord(meat_matrix=matrix)
            _add_ingredients(record, measurement.ingredients)
            state = grouped[measurement.arm_key] = (record, {}, {}, set())
        record, readings, indicators, seen_evidence = state

        indicator_key = (measurement.indicator, measurement.unit)
        readings.setdefault((day, *indicator_key), []).append(
            (measurement.value, measurement.reports_mean))
        indicators.setdefault(
            indicator_key, (measurement.indicator_type, measurement.threshold))

        evidence_key = (measurement.item_ref, "indicator_value")
        if measurement.item_ref and evidence_key not in seen_evidence:
            seen_evidence.add(evidence_key)
            record.evidence.append(EvidenceSpan(
                field_name="indicator_value", docling_item_ref=measurement.item_ref,
                page_number=measurement.page_number,
                source_type="figure" if measurement.is_figure else "table",
                method="stated", value_is_approximate=measurement.is_figure))

    for record, readings, indicators, _ in grouped.values():
        for (name, unit), (indicator_type, threshold) in indicators.items():
            record.indicators.append(IndicatorRecord(
                indicator_name=name, indicator_type=indicator_type,
                indicator_unit=unit, indicator_threshold=threshold))
        for (day, name, unit), values in sorted(readings.items()):
            value, replicates = _collapse_replicates(values)
            indicator_type, threshold = indicators[(name, unit)]
            record.measurements.append(MeasurementRecord(
                day=day, indicator_name=name, indicator_type=indicator_type,
                indicator_unit=unit, indicator_value=value,
                indicator_threshold=threshold, replicates=replicates))

    return [record for record, *_ in grouped.values()], off_axis


def catalogue_experiment(reading, matrix) -> Optional[ExperimentRecord]:
    """A row for composition-table substances to hang from without pretending to be an
    arm. Its shape says so: ingredients, and not one measurement."""
    if not reading["catalogue"]:
        return None
    record = ExperimentRecord(meat_matrix=matrix)
    _add_ingredients(record, reading["catalogue"])
    return record


def unmeasured_experiment(matrix) -> ExperimentRecord:
    """A paper with no gated series still has a matrix and a protocol. Without a row to
    hang them on it would be stored as a title and nothing else."""
    return ExperimentRecord(meat_matrix=matrix)


def heuristic_gold_bundle(package: dict) -> GoldBundle:
    """Assemble records from the Silver reading, resolving nothing: every number is fixed
    by the time this returns and only the two prose-borne fields are missing."""
    reading = package.get("reading") or normalise_silver(package)
    matrix = reading["matrix"]["name"] or UNATTRIBUTED_MATRIX
    experiments, off_axis = build_experiments(reading["measurements"], matrix)
    catalogue = catalogue_experiment(reading, matrix)
    if catalogue:
        experiments.append(catalogue)
    if not experiments:
        experiments.append(unmeasured_experiment(matrix))
        print("        no gated series — one row kept for the matrix and the protocol")

    conditions = sorted({arm.condition for arm in reading["lexicon"].values() if arm.condition})
    print(f"        matrix: {matrix!r} | {len(reading['lexicon'])} arms | "
          f"{len(experiments)} experiments | arm labels also name: "
          f"{', '.join(conditions) or 'nothing but their substances'}")
    if off_axis:
        print(f"        {off_axis} observations sit off the integer-day axis")
    averaged = sum(1 for record in experiments
                   for item in record.measurements if item.replicates > 1)
    if averaged:
        print(f"        {averaged} measurements averaged from replicates printed separately")

    return GoldBundle(
        paper=PaperDocument(title=Path(package["source_pdf"]).stem.replace("_", " ").title()),
        sections=[SectionDocument(section_title=s["section_title"],
                                  content_markdown=s["content_markdown"],
                                  docling_item_ref=s.get("docling_item_ref"),
                                  page_number=s.get("page_number"))
                  for s in package.get("sections", [])],
        tables=[TableDocument(caption=t.get("caption"),
                              csv_filepath=t.get("cleaned_csv_path") or t.get("csv_path"),
                              structured_json={"gate": t.get("gate", {}),
                                               "headers": t.get("headers", [])},
                              docling_item_ref=t.get("docling_item_ref"))
                for t in chain(package.get("tables", []), package.get("references", []))],
        figures=[FigureDocument(caption=f.get("caption"), image_filepath=f["image_path"],
                                docling_item_ref=f.get("docling_item_ref"))
                 for f in package.get("figures", [])],
        experiments=experiments,
    )


def _section_prompt(package: dict) -> tuple:
    methods_refs = set(package.get("methods_refs") or ())
    shown, methods_count = [], 0
    for section in package.get("sections", []):
        is_methods = section.get("docling_item_ref") in methods_refs
        methods_count += is_methods
        budget = MAX_METHODS_SECTION_CHARS if is_methods else MAX_OTHER_SECTION_CHARS
        shown.append({"docling_item_ref": section.get("docling_item_ref"),
                      "section_title": section["section_title"],
                      "page_number": section.get("page_number"),
                      "is_methods": is_methods,
                      "content_markdown": section["content_markdown"][:budget]})
    return shown, methods_count


_TEMPERATURE_C = re.compile(r"(-?\d{1,3}(?:[.,]\d+)?)\s*(?:°|º|\bdeg(?:rees)?\.?\s*)\s*C\b",
                            re.IGNORECASE)
_DURATION_DAYS = re.compile(r"\b(\d{1,3})\s*(?:d|days?)\b(?!\s*[-–]?\s*\d)", re.IGNORECASE)
MAX_RESOLVED_VALUES = 12


def _methods_prose(package: dict) -> str:
    refs = set(package.get("methods_refs") or ())
    sections = package.get("sections", [])
    chosen = [s for s in sections if s.get("docling_item_ref") in refs] or sections
    return "\n".join(section.get("content_markdown", "") for section in chosen)


def _numbers_in(pattern, text) -> list:
    found = set()
    for match in pattern.finditer(text):
        try:
            found.add(float(match.group(1).replace(",", ".")))
        except ValueError:
            continue
    return sorted(found)[:MAX_RESOLVED_VALUES]


def resolved_facts(package: dict, bundle: GoldBundle) -> dict:
    """What the pipeline already knows, handed over as settled rather than left to be
    inferred: the matrix and the arms come from the vocabulary, the storage days and
    indicators from the gated tables, the temperatures and durations from the methods."""
    methods = _methods_prose(package)
    measured = [record for record in bundle.experiments if record.measurements]
    return {
        "meat_matrix": bundle.experiments[0].meat_matrix if bundle.experiments else None,
        "arms_total": len(bundle.experiments),
        "arms_measured": len(measured),
        "storage_days": sorted({item.day for record in measured
                                for item in record.measurements}),
        "indicators": sorted({f"{item.indicator_name} ({item.indicator_unit})"
                              for record in measured for item in record.indicators}),
        "temperatures_c_stated_in_methods": _numbers_in(_TEMPERATURE_C, methods),
        "durations_days_stated_in_methods": _numbers_in(_DURATION_DAYS, methods),
    }


GOLD_SYSTEM_PROMPT = """\
You extract protocol metadata from the METHODS section of a food-science paper.

Each experiment arm is already resolved with matrix, ingredients, concentrations,
indicators, and measurements. Do not repeat those fields.

`resolved` holds what the software has already established: the matrix, how many arms
there are, the storage days measured, the indicators, and every temperature and duration
stated in the METHODS. Treat those as given. Do not re-derive them, do not contradict
them, and do not spend reasoning on establishing them again.

Extract:
- treatment for every experiment arm
- weight_g for every experiment arm
- evidence supporting each field

## treatment

Extract what the Methods say was **done to the food sample and how it was stored**.

Use only the Preparation / Materials and Methods section. Use the paper's own terms and keep the same order as the paper. Do not invent, reorganize, or label the procedure.

Include:

* sample preparation or handling
* processing or physical treatment
* packaging
* storage conditions or location
* storage duration
* sampling during storage

### Where to put it

**Shared procedure:**
If the same procedure was done to **all arms**, put it in `protocol`. Set every arm's `treatment` to `null`.

**Arm-specific procedure:**
If a procedure was done to **only one arm**, put it in that arm's `treatment`.

**No procedure:**
Use `null` only if the Methods describe **no sample handling or storage at all**. Storage alone is not `null`; describe the storage.

### Exclude

Do not include:

* ingredient names
* concentrations, doses, or percentages
* arm/group names
* comparisons between groups

Describe **what was done**, not which group received which treatment.

GOOD:
"Fillets were washed, drained and portioned, dipped twice in the coating dispersion for 120 s, packed in polystyrene trays overwrapped with PVC film, and stored at 4 °C for 15 days."

BAD:
"Samples received 0%, 1% and 2% thyme oil coatings."

## weight_g

Return the mass in grams of ONE experimental sample unit.

Priority:
1. directly stated by the paper
2. calculated from explicit values in the paper
3. inferred from explicit contextual information in the paper
4. null

Inference is allowed only when the paper provides enough information to support it.

Allowed examples:
- batch mass and number of samples are stated
- dimensions and density are stated
- preparation details constrain the mass

Do not infer from:
- typical food sizes
- external knowledge
- other papers
- unstated assumptions

For inferred values, explain the reasoning in evidence and mark:
"method": "inferred"

## evidence

Provide evidence for every field, including null values.

Each evidence object:

{
  "field_name": "treatment" | "weight_g",
  "docling_item_ref": REQUIRED, and copied from `valid_docling_item_refs`,
  "page_number": integer | null,
  "source_type": "prose" | "table" | "figure",
  "source_label": string | null,
  "exact_text": REQUIRED for stated prose evidence,
  "method": "stated" | "derived" | "inferred",
  "rationale": REQUIRED for derived or inferred values
}

## the reply

"protocol" is the handling every arm went through, and is where the answer belongs when
the METHODS describe one procedure for all groups. "experimental_groups" is how many
groups the METHODS define, which may differ from the number of arms listed below.

Return exactly one "experiments" object per experiment_index provided.

Return ONLY valid JSON:

{
  "protocol": "..." | null,
  "experimental_groups": integer | null,
  "evidence": [ ... supporting "protocol", with "field_name": "treatment" ... ],
  "experiments": [
    {
      "experiment_index": 0,
      "treatment": "..." | null,
      "weight_g": 25.0,
      "evidence": [...]
    }
  ]
}
"""

def _arm_prompt(bundle: GoldBundle) -> list:
    """The arms as the model sees them, each under the index that joins its reply back.
    Doses are shown so it can tell them apart, not so it can return them."""
    return [{
        "experiment_index": index,
        "meat_matrix": record.meat_matrix,
        "ingredients": [{"ingredient_name": link.ingredient_name,
                         "concentration": link.concentration,
                         "concentration_unit": link.concentration_unit}
                        for link in record.experiment_ingredients],
        "days": sorted({m.day for m in record.measurements}),
        "indicators": sorted({m.indicator_name for m in record.measurements}),
    } for index, record in enumerate(bundle.experiments)]


VALIDATION_MESSAGE_CHARS = 60


def _schema_problem(exc: ValidationError) -> str:
    return (f"{exc.error_count()} problems, "
            f"first: {exc.errors()[0]['msg'][:VALIDATION_MESSAGE_CHARS]}")


def _shared_protocol(payload: dict) -> SharedProtocol:
    """The paper-level half of the reply, validated apart from the arms so a malformed
    shared block costs the fallback rather than the per-arm answers beside it."""
    try:
        return SharedProtocol.model_validate(payload)
    except ValidationError as exc:
        print(f"        shared protocol does not fit the schema — ignored "
              f"({_schema_problem(exc)})")
        return SharedProtocol()


def apply_protocols(bundle: GoldBundle, payload: dict) -> GoldBundle:
    """Merge the model's reply into the assembled bundle. Mutates, and returns.

    Each arm is validated on its own, so a malformed entry costs that arm and not the
    paper, and an index outside the bundle means an invented arm. An arm still without a
    protocol inherits the paper's shared one, and the evidence supporting it.
    """
    shared = _shared_protocol(payload)
    bundle.experimental_groups = shared.experimental_groups

    entries = payload.get("experiments")
    if not isinstance(entries, list):
        print("        protocol reply has no experiments list — only the shared protocol applies")
        entries = []
    for position, entry in enumerate(entries):
        try:
            record = ProtocolRecord.model_validate(entry)
        except ValidationError as exc:
            print(f"        protocol entry {position} does not fit the schema — dropped "
                  f"({_schema_problem(exc)})")
            continue
        if not 0 <= record.experiment_index < len(bundle.experiments):
            print(f"        protocol names experiment {record.experiment_index}, which "
                  f"the gate did not find — dropped")
            continue
        experiment = bundle.experiments[record.experiment_index]
        if record.treatment:
            experiment.treatment = record.treatment
        experiment.weight_g = record.weight_g
        experiment.evidence.extend(record.evidence)

    inherited = 0
    if shared.protocol:
        for experiment in bundle.experiments:
            if experiment.treatment is None and experiment.measurements:
                experiment.treatment = shared.protocol
                experiment.evidence.extend(shared.evidence)
                inherited += 1

    arms = [experiment for experiment in bundle.experiments if experiment.measurements]
    notes = [f"{inherited} inherited the paper's shared protocol"] if inherited else []
    if shared.experimental_groups:
        notes.append(f"the methods name {shared.experimental_groups} groups")
    described = sum(1 for experiment in arms if experiment.treatment)
    print(f"        protocols: {described}/{len(arms)} measured arms"
          + (" | " + " | ".join(notes) if notes else ""))
    return bundle


def _trim_asset_context(payload: dict, factor: float) -> None:
    bound = max(120, int(FIGURE_CONTEXT_CHARS * factor))
    for asset in chain(payload.get("tables", []), payload.get("figures", [])):
        asset["nearby_text"] = (asset.get("nearby_text") or "")[:bound]


def _trim_other_sections(payload: dict, factor: float) -> None:
    bound = max(120, int(MAX_OTHER_SECTION_CHARS * factor))
    for section in payload.get("sections", []):
        if not section.get("is_methods"):
            section["content_markdown"] = section["content_markdown"][:bound]


PROMPT_TRIM_LADDER = (
    ("figure and table context", _trim_asset_context),
    ("non-methods sections", _trim_other_sections),
)

MIN_METHODS_SECTION_CHARS = env_int("MIN_METHODS_SECTION_CHARS", "600")
MAX_METHODS_FIT_PASSES = 4


def _serialise_prompt(payload: dict) -> str:
    return json.dumps(payload, ensure_ascii=False, indent=2, default=str)


def _fit_methods_to_budget(payload: dict, budget: int) -> str:
    """Share what is left of the budget across however many methods sections there are.

    A per-section constant cannot do this. A paper with fourteen methods subsections
    sends fourteen times whatever the constant is, so it stays over budget no matter how
    far the constant is wound down; the bound has to be derived from the budget and the
    section count instead. Escaping means the serialised length exceeds the raw text, so
    the share is recomputed from the real length rather than solved once.

    Where even the floor times the section count is too much, sections are dropped rather
    than shredded: twenty-seven 600-character fragments say less than eight whole
    subsections. The ones kept are the first, because methods run in document order and
    sample preparation and storage — the only two things being asked about — come before
    the analytical procedures.
    """
    text = _serialise_prompt(payload)
    for _ in range(MAX_METHODS_FIT_PASSES):
        if len(text) <= budget:
            return text
        methods = [section for section in payload.get("sections", [])
                   if section.get("is_methods") and section["content_markdown"]]
        if not methods:
            return text
        overhead = len(text) - sum(len(section["content_markdown"]) for section in methods)
        room = budget - overhead
        if room // len(methods) < MIN_METHODS_SECTION_CHARS:
            keep = max(1, room // MIN_METHODS_SECTION_CHARS)
            if keep < len(methods):
                print(f"        prompt budget: keeping the first {keep} of "
                      f"{len(methods)} methods subsections whole")
            for section in methods[keep:]:
                section["content_markdown"] = ""
            methods = methods[:keep]
        share = max(MIN_METHODS_SECTION_CHARS, room // len(methods))
        if share >= max(len(section["content_markdown"]) for section in methods):
            return text
        for section in methods:
            section["content_markdown"] = section["content_markdown"][:share]
        text = _serialise_prompt(payload)
    return text


def _drop_asset_context(payload: dict) -> None:
    for asset in chain(payload.get("tables", []), payload.get("figures", [])):
        asset.pop("nearby_text", None)


def _drop_reference_tables(payload: dict) -> None:
    payload["reference_tables"] = []


def _drop_other_sections(payload: dict) -> None:
    payload["sections"] = [section for section in payload.get("sections", [])
                           if section.get("is_methods")]


def _drop_asset_detail(payload: dict) -> None:
    """Down to what a citation needs: which asset it is and what it was called."""
    keep = ("kind", "docling_item_ref", "page_number", "caption")
    for name in ("tables", "figures"):
        payload[name] = [{field: asset.get(field) for field in keep}
                         for asset in payload.get(name, [])]


PROMPT_LAST_RESORT = (
    ("asset context", _drop_asset_context),
    ("reference tables", _drop_reference_tables),
    ("non-methods sections", _drop_other_sections),
    ("asset detail", _drop_asset_detail),
)


def fit_prompt_budget(payload: dict, budget: int = MAX_PROMPT_CHARS,
                      paper=None) -> tuple:
    """Serialise the prompt within `budget` characters.

    The budget is a guarantee, not a target. An earlier version stopped at the end of its
    proportional trims and sent whatever was left, which on one paper was 69% over and
    was rejected outright — current Ollama answers an oversized prompt with a 400, not
    with the silent truncation the warning used to promise.
    """
    text = _serialise_prompt(payload)
    if len(text) <= budget:
        return payload, text

    original = len(text)

    def done(step):
        print(f"        prompt budget: {original} -> {len(text)} chars, {step}")
        record_metric("prompt_trimmed", paper, seconds=0.0, chars_before=original,
                      chars_after=len(text), last_step=step)
        return payload, text

    for name, trim in PROMPT_TRIM_LADDER:
        for factor in (0.5, 0.25):
            trim(payload, factor)
            text = _serialise_prompt(payload)
            if len(text) <= budget:
                return done(f"{name} trimmed to {factor:.0%}")

    text = _fit_methods_to_budget(payload, budget)
    if len(text) <= budget:
        methods = sum(1 for section in payload.get("sections", [])
                      if section.get("is_methods"))
        return done(f"methods prose shared across {methods} section(s)")

    for step, drop in PROMPT_LAST_RESORT:
        drop(payload)
        text = _fit_methods_to_budget(payload, budget)
        if len(text) <= budget:
            return done(f"{step} dropped")

    print(f"        WARNING: prompt is {len(text)} chars with nothing left to drop, over "
          f"the {budget} budget — the server will refuse it")
    record_metric("prompt_trimmed", paper, seconds=0.0, chars_before=original,
                  chars_after=len(text), last_step="over budget")
    return payload, text


def build_gold_from_silver(package: dict, client, index: dict) -> GoldBundle:
    """Assemble from Silver, then ask the local model only for what prose alone carries.
    No fallback to the bare bundle: an arm with no protocol and no mass looks exactly
    like a paper that described neither."""
    bundle = heuristic_gold_bundle(package)
    sections, methods_count = _section_prompt(package)
    tables = [build_asset_prompt(t, "table") for t in package.get("tables", [])]
    figures = [build_asset_prompt(f, "figure") for f in package.get("figures", [])]
    references = [{"caption": r.get("caption"), "headers": r.get("headers"),
                   "docling_item_ref": r.get("docling_item_ref")}
                  for r in package.get("references", [])]
    shown_refs = sorted({item.get("docling_item_ref")
                         for item in chain(sections, tables, figures, references)
                         if item.get("docling_item_ref") in index})

    def assemble(budget):
        return fit_prompt_budget({
            "paper_slug": package["paper_slug"],
            "resolved": resolved_facts(package, bundle),
            "sections": sections,
            "arms": _arm_prompt(bundle),
            "tables": tables,
            "figures": figures,
            "reference_tables": references,
            "valid_docling_item_refs": shown_refs,
        }, budget=budget, paper=package["paper_slug"])

    slug = package["paper_slug"]
    budget = MAX_PROMPT_CHARS
    for attempt in range(2):
        _, user_prompt = assemble(budget)
        estimate = int((len(GOLD_SYSTEM_PROMPT) + len(user_prompt)) / CHARS_PER_TOKEN)
        print(f"        gold prompt: ~{estimate} tokens over {len(bundle.experiments)} "
              f"arms | {methods_count}/{len(sections)} sections are methods")
        try:
            with stage_timer("gold", slug, prompt_chars=len(user_prompt)):
                reply, prompt_tokens = client.json_completion(
                    system_prompt=GOLD_SYSTEM_PROMPT, user_prompt=user_prompt,
                    cache_scope="gold", paper=slug)
            break
        except RuntimeError as exc:
            if "exceed_context_size" not in str(exc) or attempt:
                raise
            budget //= 2
            print(f"        the server refused the prompt as too long — rebuilding it at "
                  f"{budget} chars and trying once more")

    if prompt_tokens:
        print(f"        ollama read {prompt_tokens}/{OLLAMA_NUM_CTX} context tokens")
    apply_protocols(bundle, reply)
    return score_bundle_evidence(bundle, index)

## Validation and Report

Cross-record rules the per-record models cannot express, plus what the vocabulary changed, discarded, or could not name.

In [ ]:
VIOLATION_COLUMNS = ["rule", "table", "key", "detail"]

SUMMARY_CHARS = 60
REPORT_WIDTH = 78
REPORT_REASON_CHARS = 82


def _short(text) -> str:
    text = " ".join(str(text or "").split())
    return text if len(text) <= SUMMARY_CHARS else text[:SUMMARY_CHARS - 1] + "…"


_PLACEHOLDER_TREATMENT = {"none", "na", "n_a", "nil", "not_specified", "not_stated",
                          "not_reported", "not_described", "unknown", "unspecified", "control"}


def _check_treatment(record, key, substances):
    """No vocabulary applies to free prose. What is checkable is that it is a protocol
    and not one of the things the schema keeps elsewhere."""
    if record.treatment is None:
        if record.measurements:
            yield "arm_needs_a_protocol", "experiments", key, \
                "the study measured this arm but no protocol was extracted for it"
        return
    if canonical_key(record.treatment) in _PLACEHOLDER_TREATMENT:
        yield "treatment_must_not_be_a_placeholder", "experiments", key, \
            f"{_short(record.treatment)!r} says nothing — a null would say the same honestly"
    if canonical_key(record.treatment) in substances:
        yield "treatment_must_not_be_an_ingredient", "experiments", key, \
            f"{_short(record.treatment)!r} is also an ingredient name"
    if parse_dosed_label(record.treatment):
        yield "dose_belongs_in_concentration", "experiments", key, \
            f"{_short(record.treatment)!r} carries an amount — it belongs in concentration"


def _check_attribution(record, key, index):
    """No value without a source. A null field is exempt: absence of a span is how "never
    addressed" is recorded, distinct from "the paper says there was none"."""
    attributed = {span.field_name for span in record.evidence}
    for field_name in ("treatment", "weight_g"):
        if getattr(record, field_name) is not None and field_name not in attributed:
            yield f"{field_name}_needs_evidence", "experiments", key, \
                f"{field_name} is set, but no evidence span says where it came from"
    if index is None:
        return
    for span in record.evidence:
        if span.docling_item_ref not in index:
            yield "evidence_ref_must_resolve", "evidence", f"{key}/{span.field_name}", \
                f"{span.docling_item_ref!r} is not an item the model was shown"


def _check_ingredients(record, key, vocabulary):
    declared = {i.ingredient_name for i in record.ingredients}
    for item in record.ingredients:
        if vocabulary.resolve(item.ingredient_name, "ingredient") is None:
            yield "ingredient_must_be_in_vocabulary", "ingredients", item.ingredient_name, \
                "not a known substance — a table read as a catalogue by shape alone"
        if item.functional_class == UNCLASSIFIED_CLASS:
            yield "ingredient_needs_a_functional_class", "ingredients", item.ingredient_name, \
                "unclassified — add functional_class to vocabulary.yaml"
        if item.source == UNKNOWN_SOURCE:
            yield "ingredient_needs_a_source", "ingredients", item.ingredient_name, \
                "unknown origin — add source to vocabulary.yaml"
    for link in record.experiment_ingredients:
        if link.ingredient_name not in declared:
            yield "link_must_reference_a_declared_ingredient", "experiment_ingredients", \
                f"{key}/{link.ingredient_name}", "no matching ingredient record"
        elif link.concentration is None:
            yield "link_needs_a_concentration", "experiment_ingredients", \
                f"{key}/{link.ingredient_name}", "no amount was read for this arm"


def _check_indicators(record, key):
    declared = {(i.indicator_name, i.indicator_unit) for i in record.indicators}
    for item in record.indicators:
        if _DISCARDED.search(canonical_key(item.indicator_name)):
            yield "indicator_must_not_be_sensory", "indicators", item.indicator_name, \
                "sensory or gravimetric — should have been dropped in Silver"
        if item.indicator_name == UNRESOLVED_INDICATOR:
            yield "indicator_needs_a_name", "indicators", f"{key}/{item.indicator_unit}", \
                "no caption or label named the quantity"
        if item.indicator_unit == UNSPECIFIED_UNIT:
            yield "indicator_needs_a_unit", "indicators", f"{key}/{item.indicator_name}", \
                "no unit in the label and none in the vocabulary"
    for item in record.measurements:
        if (item.indicator_name, item.indicator_unit) not in declared:
            yield "measurement_must_reference_a_declared_indicator", "measurements", \
                f"{key}/day {item.day}/{item.indicator_name}", "indicator not listed on this arm"


def _check_group_count(bundle):
    """The gate finds arms in the tables, the methods say how many groups the study
    defined, and neither side sees the difference alone: a phantom arm from a misread
    label, or a group whose data never reached a table."""
    expected = bundle.experimental_groups
    if not expected:
        return
    measured = sum(1 for record in bundle.experiments if record.measurements)
    if measured != expected:
        yield "experiment_count_must_match_the_study", "experiments", f"{measured} arms", \
            f"the methods define {expected} experimental groups"


def validate_bundle_semantics(bundle: GoldBundle, index: dict | None = None,
                              vocabulary=None) -> pd.DataFrame:
    """Cross-record rules the per-record models cannot express. Never raises: one pass
    shows everything wrong, and an empty frame means the bundle is consistent. Without
    `index` the citation rule is skipped, not guessed."""
    vocabulary = vocabulary or VOCABULARY
    substances = {canonical_key(i.ingredient_name)
                  for record in bundle.experiments for i in record.ingredients}
    rows = list(_check_group_count(bundle))
    for record in bundle.experiments:
        key = (f"{record.meat_matrix}/{_short(record.treatment) or 'no protocol'}/"
               + (", ".join(sorted(i.ingredient_name for i in record.ingredients)) or "control"))
        rows += list(_check_treatment(record, key, substances))
        rows += list(_check_attribution(record, key, index))
        rows += list(_check_ingredients(record, key, vocabulary))
        rows += list(_check_indicators(record, key))
    return pd.DataFrame(rows, columns=VIOLATION_COLUMNS)


def _changed_frame(vocabulary) -> pd.DataFrame:
    """`renamed` separates a value rewritten ("TVB-N" -> "Total volatile basic nitrogen")
    from one merely confirmed ("pH" -> "pH")."""
    columns = ["kind", "raw", "canonical", "renamed", "functional_class", "source",
               "indicator_type", "unit"]
    frame = pd.DataFrame([{
        "kind": r["kind"], "raw": r["raw"], "canonical": r["canonical"],
        "renamed": canonical_key(r["raw"]) != canonical_key(r["canonical"]),
        "functional_class": r.get("functional_class"), "source": r.get("source"),
        "indicator_type": r.get("indicator_type"), "unit": r.get("unit"),
    } for r in vocabulary.changes.values()], columns=columns)
    return frame.sort_values(["kind", "canonical"], ignore_index=True) if not frame.empty else frame


def _vocabulary_frame() -> pd.DataFrame:
    return pd.DataFrame(
        [{"field": "indicators.indicator_type", "value": v, "rolls_up_to": ""}
         for v in INDICATOR_TYPES]
        + [{"field": "ingredients.functional_class", "value": leaf, "rolls_up_to": tier}
           for leaf, tier in FUNCTIONAL_CLASS_TIERS.items()]
        + [{"field": "ingredients.source", "value": v, "rolls_up_to": ""}
           for v in INGREDIENT_SOURCES]
        + [{"field": "evidence.method", "value": v, "rolls_up_to": ""}
           for v in EVIDENCE_METHODS]
        + [{"field": "evidence.source_type", "value": v, "rolls_up_to": ""}
           for v in EVIDENCE_SOURCE_TYPES])


def _review_frame(vocabulary, violations) -> pd.DataFrame:
    """A violation names the same value as the flag behind it and says more, so it wins.
    A discarded value is already reported, with its count, by `_discarded_frame`."""
    rows = []
    covered = {canonical_key(value) for _, value in vocabulary.discarded}

    def take(kind, value, reason):
        if canonical_key(value) not in covered:
            covered.add(canonical_key(value))
            rows.append({"kind": kind, "value": value, "reason": reason})

    if violations is not None and not violations.empty:
        for _, violation in violations.iterrows():
            take(violation["table"], violation["key"],
                 f"{violation['rule']}: {violation['detail']}")
    for (kind, value), reason in vocabulary.flags.items():
        take(kind, value, reason)
    for kind, names in vocabulary.unresolved.items():
        for name in names.values():
            take(kind, name, "not in vocabulary — kept verbatim")
    return pd.DataFrame(rows, columns=["kind", "value", "reason"])


def _discarded_frame(vocabulary) -> pd.DataFrame:
    """What never reached the database, and why. A drop nobody can see is a bug."""
    return pd.DataFrame(
        [{"kind": kind, "value": value, "occurrences": count,
          "reason": "sensory or gravimetric — not a shelf-life indicator" if kind == "indicator"
                    else "not a known substance — add it to vocabulary.yaml to keep it"}
         for (kind, value), count in sorted(vocabulary.discarded.items())],
        columns=["kind", "value", "occurrences", "reason"])


def normalization_report(violations: pd.DataFrame | None = None, vocabulary=None) -> dict:
    vocabulary = vocabulary or VOCABULARY
    return {"changed": _changed_frame(vocabulary), "vocabulary": _vocabulary_frame(),
            "discarded": _discarded_frame(vocabulary),
            "review": _review_frame(vocabulary, violations)}


def show_normalization_report(violations: pd.DataFrame | None = None) -> dict:
    report = normalization_report(violations)
    changed, review, discarded = report["changed"], report["review"], report["discarded"]
    print("─ normalisation ".ljust(REPORT_WIDTH, "─"))
    for kind, group in changed.groupby("kind") if not changed.empty else ():
        print(f"  {kind:<11}: {len(group):>3} raw -> {group['canonical'].nunique():>3} canonical  "
              f"({int(group['renamed'].sum())} rewritten, "
              f"{len(group) - group['canonical'].nunique()} merged)")
    print(f"  values changed         : {0 if changed.empty else int(changed['renamed'].sum())}")
    print(f"  controlled values      : {len(report['vocabulary'])}")
    for kind, group in discarded.groupby("kind") if not discarded.empty else ():
        print(f"  {kind + ' discarded':<22}: {len(group):>3} names, "
              f"{int(group['occurrences'].sum())} occurrences")
    print(f"  needing manual review  : {len(review)}")
    for reason, group in review.groupby("reason") if not review.empty else ():
        print(f"      {len(group):>3}  {reason[:REPORT_REASON_CHARS]}")
    print("─" * REPORT_WIDTH)
    for title, frame in (("changed", changed), ("vocabulary", report["vocabulary"]),
                         ("discarded", discarded), ("review", review)):
        if not frame.empty:
            print(f"\n{title}:")
            display(frame)
    return report


BUNDLE_DIFF_COLUMNS = ["paper", "field", "baseline", "current"]
VALUE_DECIMALS = 4
DIFF_VALUE_CHARS = 70


def bundle_fingerprint(bundle: GoldBundle) -> dict:
    """Everything about a bundle that a speed change must not move: the arms, their
    protocols and masses, their doses and every measurement on them. Sorted throughout,
    because the order arms come out in is not part of the answer."""
    experiments = [{
        "meat_matrix": record.meat_matrix,
        "treatment": _searchable(record.treatment) or None,
        "weight_g": None if record.weight_g is None else round(record.weight_g, VALUE_DECIMALS),
        "ingredients": sorted([link.ingredient_name, link.concentration,
                               link.concentration_unit]
                              for link in record.experiment_ingredients),
        "measurements": sorted([item.day, item.indicator_name, item.indicator_unit,
                                round(item.indicator_value, VALUE_DECIMALS),
                                item.replicates]
                               for item in record.measurements),
        "evidence_spans": len(record.evidence),
    } for record in bundle.experiments]
    experiments.sort(key=lambda item: json.dumps(item, sort_keys=True, default=str))
    return {"experimental_groups": bundle.experimental_groups, "experiments": experiments}


def _diff_value(value) -> str:
    text = json.dumps(value, default=str)
    return text if len(text) <= DIFF_VALUE_CHARS else text[:DIFF_VALUE_CHARS - 1] + "…"


def _diff_one(paper: str, before: dict, after: dict):
    def row(field, was, now):
        return {"paper": paper, "field": field,
                "baseline": _diff_value(was), "current": _diff_value(now)}

    if before.get("experimental_groups") != after.get("experimental_groups"):
        yield row("experimental_groups", before.get("experimental_groups"),
                  after.get("experimental_groups"))
    was, now = before.get("experiments", []), after.get("experiments", [])
    if len(was) != len(now):
        yield row("experiment count", len(was), len(now))
        return
    for index, (left, right) in enumerate(zip(was, now)):
        for name in ("meat_matrix", "treatment", "weight_g", "ingredients",
                     "measurements", "evidence_spans"):
            if left.get(name) != right.get(name):
                yield row(f"experiment {index}.{name}", left.get(name), right.get(name))


def compare_bundle_fingerprints(baseline: dict, current: dict) -> pd.DataFrame:
    """Every way this run's Gold differs from a saved one. An empty frame is the only
    result that clears a change to ship."""
    rows = []
    for paper in sorted(set(baseline) | set(current)):
        if paper not in current:
            rows.append({"paper": paper, "field": "paper", "baseline": "present",
                         "current": "missing"})
        elif paper not in baseline:
            rows.append({"paper": paper, "field": "paper", "baseline": "missing",
                         "current": "present"})
        else:
            rows += list(_diff_one(paper, baseline[paper], current[paper]))
    return pd.DataFrame(rows, columns=BUNDLE_DIFF_COLUMNS)


def show_bundle_diff(baseline_path, current: dict) -> pd.DataFrame:
    baseline = json.loads(Path(baseline_path).read_text(encoding="utf-8"))
    frame = compare_bundle_fingerprints(baseline, current)
    if frame.empty:
        print(f"Gold is semantically identical to {Path(baseline_path).name} "
              f"over {len(current)} paper(s).")
    else:
        print(f"Gold differs from {Path(baseline_path).name} in {len(frame)} place(s) "
              "— this change did more than make the pipeline faster:")
        display(frame)
    return frame

## Local SQLite Schema

Mirrors the production layout. The vocabularies are `CHECK` constraints here too, so a bad value fails on write.

In [ ]:
from sqlalchemy import (
    Boolean, CheckConstraint, Column, DateTime, Float, ForeignKey, Index, Integer, String,
    Text, UniqueConstraint, create_engine, insert, inspect, select,
)
from sqlalchemy.dialects import postgresql, sqlite
from sqlalchemy.orm import declarative_base, sessionmaker


def _sql_values(values):
    """A Python vocabulary as a SQL literal list, so the CHECK constraints cannot drift
    from it. Inputs are module constants, never anything read from a paper."""
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)


def _utc_now():
    """`datetime.utcnow` is deprecated and naive; the manifests are already aware."""
    return datetime.now(timezone.utc)


Base = declarative_base()
ENGINE = create_engine(DATABASE_URL, future=True)
SessionLocal = sessionmaker(bind=ENGINE, autoflush=False, autocommit=False, future=True)

class PaperRow(Base):
    __tablename__ = "papers"

    id = Column(Integer, primary_key=True, index=True)
    doi = Column(String, nullable=True, index=True)
    title = Column(String, nullable=False)
    abstract = Column(Text, nullable=True)
    published_year = Column(Integer, nullable=True)
    source_path = Column(String, nullable=False)
    file_hash = Column(String(64), unique=True, nullable=False, index=True)
    bronze_path = Column(String, nullable=False)
    silver_path = Column(String, nullable=True)
    markdown_path = Column(String, nullable=True)
    created_at = Column(DateTime, default=_utc_now)

class SectionRow(Base):
    __tablename__ = "sections"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    section_title = Column(String, nullable=False)
    content_markdown = Column(Text, nullable=False)
    embedding = Column(Text, nullable=True)
    section_order = Column(Integer, nullable=True)
    page_number = Column(Integer, nullable=True)
    docling_item_ref = Column(String, nullable=True)

class TableRow(Base):
    __tablename__ = "tables"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    caption = Column(Text, nullable=True)
    csv_filepath = Column(String, nullable=False)
    structured_json = Column(Text, nullable=True)
    docling_item_ref = Column(String, nullable=True)
    created_at = Column(DateTime, default=_utc_now)

class FigureRow(Base):
    __tablename__ = "figures"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    caption = Column(Text, nullable=True)
    image_filepath = Column(String, nullable=False)
    semantic_tags_json = Column(Text, nullable=True)
    docling_item_ref = Column(String, nullable=True)
    created_at = Column(DateTime, default=_utc_now)

class ExperimentRow(Base):
    """One arm. `treatment` is its protocol in prose, the additive dimension is
    `experiment_ingredients`, and a control has no rows there. No unique constraint:
    nothing here identifies an arm on its own."""

    __tablename__ = "experiments"

    experiment_id = Column(Integer, primary_key=True, index=True)
    meat_matrix = Column(String, nullable=False)
    treatment = Column(String, nullable=True)
    weight_g = Column(Float, nullable=True)
    __table_args__ = (
        CheckConstraint("weight_g IS NULL OR weight_g > 0",
                        name="ck_experiments_weight_positive"),
        Index("ix_experiments_matrix_treatment", "meat_matrix", "treatment"),
    )

class IngredientRow(Base):
    __tablename__ = "ingredients"

    ingredient_id = Column(Integer, primary_key=True, index=True)
    ingredient_name = Column(String, nullable=False, unique=True)
    functional_class = Column(String, nullable=False)
    source = Column(String, nullable=False)
    __table_args__ = (
        CheckConstraint(f"functional_class IN ({_sql_values(FUNCTIONAL_CLASSES + (UNCLASSIFIED_CLASS,))})",
                        name="ck_ingredients_functional_class"),
        CheckConstraint(f"source IN ({_sql_values(INGREDIENT_SOURCES + (UNKNOWN_SOURCE,))})",
                        name="ck_ingredients_source"),
    )

class ExperimentIngredientRow(Base):
    """`concentration` is where every amount lives."""

    __tablename__ = "experiment_ingredients"

    experiment_id = Column(Integer, ForeignKey("experiments.experiment_id", ondelete="CASCADE"), primary_key=True)
    ingredient_id = Column(Integer, ForeignKey("ingredients.ingredient_id", ondelete="CASCADE"), primary_key=True)
    concentration = Column(Float, nullable=True)
    concentration_unit = Column(String, nullable=True)

class IndicatorRow(Base):
    """Unique on (name, unit): `indicator_type` is a two-valued category, so keying on it
    would fold every microbial count in log CFU/g into one row."""

    __tablename__ = "indicators"

    indicator_id = Column(Integer, primary_key=True, index=True)
    indicator_name = Column(String, nullable=False)
    indicator_type = Column(String, nullable=False)
    indicator_unit = Column(String, nullable=False)
    indicator_threshold = Column(Float, nullable=True)
    __table_args__ = (
        UniqueConstraint("indicator_name", "indicator_unit", name="uq_indicator_name_unit"),
        CheckConstraint(f"indicator_type IN ({_sql_values(INDICATOR_TYPES)})",
                        name="ck_indicators_type"),
    )

class MeasurementRow(Base):
    __tablename__ = "measurements"

    experiment_id = Column(Integer, ForeignKey("experiments.experiment_id", ondelete="CASCADE"), primary_key=True)
    day = Column(Integer, primary_key=True)
    indicator_id = Column(Integer, ForeignKey("indicators.indicator_id", ondelete="CASCADE"), primary_key=True)
    indicator_value = Column(Float, nullable=False)
    replicates = Column(Integer, nullable=False, default=1)
    __table_args__ = (
        CheckConstraint("day >= 0", name="ck_measurements_day_non_negative"),
        CheckConstraint("replicates >= 1", name="ck_measurements_replicates_positive"),
    )

class EvidenceRow(Base):
    """What supports one extracted field. `method` and `rationale` say how far it sits
    from the paper's own words; `confidence` is computed, never taken from the model."""

    __tablename__ = "evidence"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    entity_type = Column(String, nullable=False)
    entity_key = Column(Text, nullable=False)
    field_name = Column(String, nullable=False)
    page_number = Column(Integer, nullable=True)
    source_type = Column(String, nullable=False)
    source_label = Column(String, nullable=True)
    exact_text = Column(Text, nullable=True)
    method = Column(String, nullable=False)
    rationale = Column(Text, nullable=True)
    confidence = Column(Float, nullable=True)
    value_is_approximate = Column(Boolean, default=False)
    docling_item_ref = Column(String, nullable=True)
    __table_args__ = (
        CheckConstraint(f"method IN ({_sql_values(EVIDENCE_METHODS)})",
                        name="ck_evidence_method"),
        CheckConstraint(f"source_type IN ({_sql_values(EVIDENCE_SOURCE_TYPES)})",
                        name="ck_evidence_source_type"),
        CheckConstraint("method = 'stated' OR (rationale IS NOT NULL AND rationale != '')",
                        name="ck_evidence_rationale_when_not_stated"),
    )


PAPER_STATUSES = ("pending", "bronze", "silver", "gold", "failed")


class PaperStatusRow(Base):
    __tablename__ = "paper_status"

    file_hash = Column(String(64), primary_key=True)
    paper_slug = Column(String, nullable=False)
    source_path = Column(String, nullable=False)
    status = Column(String, nullable=False)
    detail = Column(Text, nullable=True)
    run_id = Column(String, nullable=True, index=True)
    updated_at = Column(DateTime, default=_utc_now, onupdate=_utc_now)
    __table_args__ = (
        CheckConstraint(f"status IN ({_sql_values(PAPER_STATUSES)})",
                        name="ck_paper_status"),
    )


_REQUIRED_COLUMNS = {"indicators": {"indicator_name"},
                     "sections": {"docling_item_ref", "page_number"},
                     "experiments": {"weight_g"},
                     "measurements": {"replicates"},
                     "evidence": {"method", "rationale"}}
_REMOVED_COLUMNS = {"experiments": {"arm_label"}, "measurements": {"value_is_approximate"},
                    "papers": {"docling_json_path"}}

ALLOW_DB_RESET = env_flag("ALLOW_DB_RESET", "0")


def init_local_database() -> None:
    inspector = inspect(ENGINE)
    existing = set(inspector.get_table_names())

    def columns(table):
        return {column["name"] for column in inspector.get_columns(table)}

    stale = [f"{table} is missing {', '.join(sorted(required - columns(table)))}"
             for table, required in _REQUIRED_COLUMNS.items()
             if table in existing and required - columns(table)]
    stale += [f"{table} still has {', '.join(sorted(gone & columns(table)))}"
              for table, gone in _REMOVED_COLUMNS.items()
              if table in existing and gone & columns(table)]
    if stale:
        reason = "Stale database (" + "; ".join(stale) + ")"
        if not ALLOW_DB_RESET:
            raise RuntimeError(
                f"{reason}. Migrate it, or point DATABASE_URL at a fresh database. "
                "Set ALLOW_DB_RESET=1 only for a scratch sqlite file you mean to "
                "discard — every paper already extracted into it goes with it.")
        if ENGINE.dialect.name != "sqlite":
            raise RuntimeError(
                f"{reason} on {ENGINE.dialect.name}. Migrate it, or point DATABASE_URL at "
                "a fresh database — this notebook only knows how to reset a local file.")
        backup = LOCAL_DB_PATH.with_name(LOCAL_DB_PATH.name + ".pre-normalisation")
        ENGINE.dispose()
        shutil.move(str(LOCAL_DB_PATH), str(backup))
        print(f"{reason} moved to {backup.name}")
    Base.metadata.create_all(ENGINE)


_INSERT = {"postgresql": postgresql.insert, "sqlite": sqlite.insert}


def upsert(session, model, rows, conflict_columns, update_columns=()) -> None:
    """Without `update_columns`, a row that is already there is left as it is."""
    if not rows:
        return
    dialect = session.bind.dialect.name
    if dialect not in _INSERT:
        raise RuntimeError(
            f"No ON CONFLICT support wired for {dialect!r}; add it to _INSERT or point "
            "DATABASE_URL at sqlite or postgresql.")
    statement = _INSERT[dialect](model).values(rows)
    if update_columns:
        statement = statement.on_conflict_do_update(
            index_elements=conflict_columns,
            set_={name: getattr(statement.excluded, name) for name in update_columns})
    else:
        statement = statement.on_conflict_do_nothing(index_elements=conflict_columns)
    session.execute(statement)


STATUS_UPDATE_COLUMNS = ["paper_slug", "source_path", "status", "detail", "run_id",
                         "updated_at"]


DB_WRITE_LOCK = threading.Lock()


def set_paper_status(file_hash, paper_slug, source_path, status, *,
                     detail=None, run_id=None) -> None:
    """One row per paper, overwritten as it advances. Its own session and commit: a
    status that only lands if the rest of the paper succeeded records nothing useful.

    Under `DB_WRITE_LOCK` like every other writer. The vision thread advances a paper to
    `bronze` and `silver` while Gold threads are committing other papers, and sqlite
    takes one writer at a time — an unguarded status write is how a run dies of
    "database is locked" with every paper otherwise fine.
    """
    if status not in PAPER_STATUSES:
        raise ValueError(f"{status!r} is not one of {list(PAPER_STATUSES)}")
    with DB_WRITE_LOCK, SessionLocal() as session:
        upsert(session, PaperStatusRow,
               [{"file_hash": file_hash, "paper_slug": paper_slug,
                 "source_path": str(source_path), "status": status, "detail": detail,
                 "run_id": run_id, "updated_at": _utc_now()}],
               ["file_hash"], STATUS_UPDATE_COLUMNS)
        session.commit()


def completed_paper_hashes() -> set:
    """The papers a previous run carried all the way to Gold."""
    with SessionLocal() as session:
        return {row[0] for row in session.execute(
            select(PaperStatusRow.file_hash).where(PaperStatusRow.status == "gold"))}


def show_paper_status() -> pd.DataFrame:
    """Where every paper stands — what to read after a run dies."""
    with SessionLocal() as session:
        rows = session.execute(
            select(PaperStatusRow.paper_slug, PaperStatusRow.status,
                   PaperStatusRow.detail, PaperStatusRow.run_id,
                   PaperStatusRow.updated_at)).all()
    frame = pd.DataFrame(rows, columns=["paper_slug", "status", "detail", "run_id",
                                        "updated_at"])
    if not frame.empty:
        frame = frame.sort_values(["status", "paper_slug"], ignore_index=True)
    display(frame)
    return frame


def _vocabulary_ids(session) -> dict:
    return {
        "ingredient": {name: identifier for identifier, name in session.execute(
            select(IngredientRow.ingredient_id, IngredientRow.ingredient_name))},
        "indicator": {(name, unit): identifier for identifier, name, unit in session.execute(
            select(IndicatorRow.indicator_id, IndicatorRow.indicator_name,
                   IndicatorRow.indicator_unit))},
    }


def seed_vocabulary(session, vocabulary) -> dict:
    terms = {kind: sorted(set(vocabulary.terms_of(kind).values()), key=lambda t: t.name)
             for kind in ("ingredient", "indicator")}
    upsert(session, IngredientRow,
           [{"ingredient_name": term.name,
             "functional_class": term.functional_class or UNCLASSIFIED_CLASS,
             "source": term.source or UNKNOWN_SOURCE} for term in terms["ingredient"]],
           ["ingredient_name"], ["functional_class", "source"])
    upsert(session, IndicatorRow,
           [{"indicator_name": term.name,
             "indicator_type": term.indicator_type if term.indicator_type in INDICATOR_TYPES
                               else "chemical",
             "indicator_unit": term.unit or UNSPECIFIED_UNIT,
             "indicator_threshold": term.threshold} for term in terms["indicator"]],
           ["indicator_name", "indicator_unit"], ["indicator_type", "indicator_threshold"])
    session.commit()

    ids = _vocabulary_ids(session)
    print(f"Seeded {len(terms['ingredient'])} ingredients and {len(terms['indicator'])} "
          f"indicators | {len(ids['ingredient'])} + {len(ids['indicator'])} ids in memory")
    return ids


def register_new_terms(session, bundle, ids) -> dict:
    ingredients = {record.ingredient_name: record
                   for experiment in bundle.experiments for record in experiment.ingredients
                   if record.ingredient_name not in ids["ingredient"]}
    indicators = {(record.indicator_name, record.indicator_unit): record
                  for experiment in bundle.experiments
                  for record in chain(experiment.indicators, experiment.measurements)
                  if (record.indicator_name, record.indicator_unit) not in ids["indicator"]}
    if not ingredients and not indicators:
        return {"ingredients": 0, "indicators": 0}

    upsert(session, IngredientRow,
           [{"ingredient_name": r.ingredient_name, "functional_class": r.functional_class,
             "source": r.source} for r in ingredients.values()], ["ingredient_name"])
    upsert(session, IndicatorRow,
           [{"indicator_name": r.indicator_name, "indicator_type": r.indicator_type,
             "indicator_unit": r.indicator_unit, "indicator_threshold": r.indicator_threshold}
            for r in indicators.values()], ["indicator_name", "indicator_unit"])
    session.flush()
    ids.update(_vocabulary_ids(session))
    return {"ingredients": len(ingredients), "indicators": len(indicators)}


def _json_text(payload):
    return None if payload is None else json.dumps(payload, ensure_ascii=False, default=str)


def get_or_create_paper(session, manifest, bundle) -> PaperRow:
    paper = session.query(PaperRow).filter(PaperRow.file_hash == manifest["file_hash"]).one_or_none()
    if paper is None:
        paper = PaperRow(
            doi=bundle.paper.doi,
            title=bundle.paper.title,
            abstract=bundle.paper.abstract,
            published_year=bundle.paper.published_year,
            source_path=manifest["source_pdf"], file_hash=manifest["file_hash"],
            bronze_path=str(Path(manifest["markdown_path"]).parent),
            silver_path=str(SILVER_ROOT / manifest["paper_slug"]),
            markdown_path=manifest["markdown_path"])
        session.add(paper)
        session.flush()
    return paper


def persist_sections(session, paper_id, sections) -> None:
    for order, section in enumerate(sections, start=1):
        session.add(SectionRow(paper_id=paper_id, section_order=order,
                               section_title=section.section_title,
                               content_markdown=section.content_markdown,
                               embedding=section.embedding,
                               page_number=section.page_number,
                               docling_item_ref=section.docling_item_ref))


def persist_tables(session, paper_id, tables) -> None:
    for table in tables:
        session.add(TableRow(paper_id=paper_id, caption=table.caption,
                             csv_filepath=table.csv_filepath,
                             structured_json=_json_text(table.structured_json),
                             docling_item_ref=table.docling_item_ref))


def persist_figures(session, paper_id, figures) -> None:
    for figure in figures:
        session.add(FigureRow(paper_id=paper_id, caption=figure.caption,
                              image_filepath=figure.image_filepath,
                              semantic_tags_json=_json_text(figure.semantic_tags),
                              docling_item_ref=figure.docling_item_ref))


def _link_signature(links):
    return sorted((l.ingredient_name, l.concentration, l.concentration_unit) for l in links)


def _stored_signatures(session, meat_matrix, treatment) -> list:
    """Every stored arm with this matrix and protocol, with its ingredient signature.
    Two queries for the whole group, not a JOIN per candidate row."""
    rows = session.query(ExperimentRow).filter_by(
        meat_matrix=meat_matrix, treatment=treatment).all()
    if not rows:
        return []
    by_experiment = {row.experiment_id: [] for row in rows}
    links = (session.query(ExperimentIngredientRow.experiment_id,
                           IngredientRow.ingredient_name,
                           ExperimentIngredientRow.concentration,
                           ExperimentIngredientRow.concentration_unit)
             .join(IngredientRow,
                   IngredientRow.ingredient_id == ExperimentIngredientRow.ingredient_id)
             .filter(ExperimentIngredientRow.experiment_id.in_(list(by_experiment))))
    for experiment_id, name, concentration, unit in links:
        by_experiment[experiment_id].append((name, concentration, unit))
    return [[row, sorted(by_experiment[row.experiment_id])] for row in rows]


def _get_or_create_experiment(session, record, signatures) -> ExperimentRow:
    """Matched on matrix, treatment and the ingredients with their amounts: two rungs of
    a dose ladder differ only in `concentration`, and a control is the arm whose set is
    empty. `weight_g` is excluded — a re-read mass would split one arm in two."""
    key = (record.meat_matrix, record.treatment)
    stored = signatures.get(key)
    if stored is None:
        stored = signatures[key] = _stored_signatures(session, *key)
    want = _link_signature(record.experiment_ingredients)
    for row, signature in stored:
        if signature == want:
            return row
    row = ExperimentRow(meat_matrix=record.meat_matrix, treatment=record.treatment,
                        weight_g=record.weight_g)
    session.add(row)
    session.flush()
    stored.append([row, want])
    return row


def persist_gold_bundle(session, paper_id, bundle, ids) -> dict:
    counts = {"experiments": 0, "new_terms": 0, "measurements": 0,
              "measurement_collisions": 0, "evidence": 0}
    counts["new_terms"] = sum(register_new_terms(session, bundle, ids).values())

    seen, signatures = set(), {}
    links, measurements, evidence = [], [], []
    for record in bundle.experiments:
        experiment = _get_or_create_experiment(session, record, signatures)
        counts["experiments"] += 1
        for link in record.experiment_ingredients:
            ingredient_id = ids["ingredient"].get(link.ingredient_name)
            if ingredient_id is not None:
                links.append({"experiment_id": experiment.experiment_id,
                              "ingredient_id": ingredient_id,
                              "concentration": link.concentration,
                              "concentration_unit": link.concentration_unit})
        for item in record.measurements:
            indicator_id = ids["indicator"][(item.indicator_name, item.indicator_unit)]
            key = (experiment.experiment_id, item.day, indicator_id)
            if key in seen:
                counts["measurement_collisions"] += 1
                continue
            seen.add(key)
            measurements.append({"experiment_id": experiment.experiment_id, "day": item.day,
                                 "indicator_id": indicator_id,
                                 "indicator_value": item.indicator_value,
                                 "replicates": item.replicates})
        for item in record.evidence:
            evidence.append({
                "paper_id": paper_id, "entity_type": "experiment",
                "entity_key": _json_text({"experiment_id": experiment.experiment_id}),
                "field_name": item.field_name,
                "page_number": item.page_number, "source_type": item.source_type,
                "source_label": item.source_label, "exact_text": item.exact_text,
                "method": item.method, "rationale": item.rationale,
                "confidence": item.confidence,
                "value_is_approximate": item.value_is_approximate,
                "docling_item_ref": item.docling_item_ref})

    upsert(session, ExperimentIngredientRow, links, ["experiment_id", "ingredient_id"],
           ["concentration", "concentration_unit"])
    upsert(session, MeasurementRow, measurements,
           ["experiment_id", "day", "indicator_id"], ["indicator_value", "replicates"])
    if evidence:
        session.execute(insert(EvidenceRow), evidence)
    counts["measurements"] = len(measurements)
    counts["evidence"] = len(evidence)
    return counts


init_local_database()
print(f"SQLite schema ready at {LOCAL_DB_PATH}")

## Runner

Bronze → Silver → normalise → Gold → SQLite, in shards, with Gold overlapping the next shard's vision work. Every stage is content-addressed and cached, so a restarted run pays for a paper once. `show_stage_totals()` says where the time went, `show_paper_status()` where a killed run stopped. Start Ollama, set `PAPER_INPUTS`, and run.

In [ ]:
def build_ollama_client() -> OllamaJSONClient:
    """Check the server is up and the model is pulled, then hand back the client. Here
    rather than at the first Gold call, so a stopped server costs nothing instead of
    surfacing after Docling has spent minutes on the first paper."""
    tags_url = f"{OLLAMA_HOST.rstrip('/')}/api/tags"
    try:
        with urllib.request.urlopen(tags_url, timeout=OLLAMA_PROBE_TIMEOUT) as response:
            entries = json.load(response).get("models", [])
    except (urllib.error.URLError, TimeoutError, ValueError) as exc:
        raise RuntimeError(
            f"No Ollama server at {OLLAMA_HOST} ({exc}). Gold needs one to read "
            "`treatment` and `weight_g` out of the methods prose — start it with "
            "`ollama serve`, or point OLLAMA_HOST at another host.") from exc

    available = {name for name in (entry.get("model") or entry.get("name")
                                   for entry in entries) if name}
    if OLLAMA_MODEL not in available:
        raise RuntimeError(
            f"{OLLAMA_MODEL!r} is not pulled on {OLLAMA_HOST}. Run "
            f"`ollama pull {OLLAMA_MODEL}`. Available: {', '.join(sorted(available)) or 'none'}")
    return OllamaJSONClient()


def persist_document(session, manifest, bundle, ids) -> dict:
    paper = get_or_create_paper(session, manifest, bundle)
    persist_sections(session, paper.id, bundle.sections)
    persist_tables(session, paper.id, bundle.tables)
    persist_figures(session, paper.id, bundle.figures)
    counts = persist_gold_bundle(session, paper.id, bundle, ids)
    session.commit()
    return {"paper_id": paper.id, **counts, "sections": len(bundle.sections),
            "tables": len(bundle.tables), "figures": len(bundle.figures)}


NORMALISE_LOCK = threading.Lock()


def run_language_phase(manifest, package, *, llm_client, ids, show_decisions=False) -> tuple:
    """One paper from Silver to Gold. Returns (summary row, violation frame, bundle)."""
    slug = manifest["paper_slug"]
    adjudicate_review(package, llm_client)
    if show_decisions:
        show_gate_decisions(package)
    with stage_timer("normalise", slug), NORMALISE_LOCK:
        normalise_silver(package)

    index = reference_text_index(package)
    bundle = build_gold_from_silver(package, client=llm_client, index=index)

    with stage_timer("validate", slug):
        violations = validate_bundle_semantics(bundle, index)
    if not violations.empty:
        print(f"        {len(violations)} semantic violations "
              "— see show_normalization_report()")

    with stage_timer("database", slug), DB_WRITE_LOCK, SessionLocal() as session:
        counts = persist_document(session, manifest, bundle, ids)

    report = package["gate_report"]
    return {"pdf_path": manifest["source_pdf"], "paper_slug": slug,
            "native_pdf": manifest["native_pdf"], "bronze_seconds": manifest["elapsed_seconds"],
            "bronze_tables": report["tables_in"], "bronze_figures": report["figures_in"],
            "gated_tables": report["tables_accepted"], "gated_figures": report["figures_accepted"],
            "reference_assets": report["references"], "observations": report["observations"],
            "review_open": report.get("review", 0),
            "review_keyed": report.get("review_promoted", 0),
            "gold_experiments": len(bundle.experiments), "violations": len(violations),
            **counts}, violations, bundle


def _shards(items: Sequence, size: int) -> list:
    return [list(items[start:start + size]) for start in range(0, len(items), max(1, size))]


def _resolve_inputs(pdf_paths: Sequence) -> list:
    """Accept paths relative to the notebook, or bare names under PAPER_INPUT_DIR."""
    resolved, missing = [], []
    for item in pdf_paths:
        path = Path(item)
        if not path.exists():
            candidate = PAPER_INPUT_DIR / path.name
            path = candidate if candidate.exists() else path
        (resolved if path.exists() else missing).append(path)
    for path in missing:
        print(f"{path} does not exist — skipped")
    return resolved


def _refuse_slug_collisions(papers: Sequence) -> None:
    """Every stage keys a paper on its file name: Bronze and Silver write under it and
    the summary is ordered by it. Two PDFs called 11.pdf in different folders therefore
    overwrite each other's artifacts and one of them vanishes from the report. Invisible
    at three papers, silent data loss at three hundred, so it stops the run instead."""
    seen: dict = {}
    for path in papers:
        seen.setdefault(path.stem, []).append(path)
    clashes = {stem: paths for stem, paths in seen.items() if len(paths) > 1}
    if clashes:
        detail = "; ".join(f"{stem}: " + ", ".join(str(path) for path in paths)
                           for stem, paths in sorted(clashes.items()))
        raise ValueError(
            f"{len(clashes)} file name(s) appear more than once, and a paper is keyed on "
            f"its file name — rename them before running: {detail}")


def _run_semantics() -> dict:
    """Settings that change what is extracted rather than how fast it happens. Written
    into every run's manifest, so a consumer can tell one run's data from another's —
    REQUIRE_FIGURE_DATA=0 in particular is a legitimate kill switch that silently admits
    every probed figure on geometry alone."""
    return {
        "REQUIRE_FIGURE_DATA": REQUIRE_FIGURE_DATA,
        "FIGURE_EDGE_FILTER": FIGURE_EDGE_FILTER,
        "FIGURE_CAPTION_FILTER": FIGURE_CAPTION_FILTER,
        "TABLE_STRUCTURE": TABLE_STRUCTURE,
        "TABLE_STRUCTURE_MODE": TABLE_STRUCTURE_MODE,
        "IMAGES_SCALE": IMAGES_SCALE,
        "CHART_MAX_PIXELS": CHART_MAX_PIXELS,
        "CHART_MAX_NEW_TOKENS": CHART_MAX_NEW_TOKENS,
        "GATE_VERSION": GATE_VERSION,
        "OLLAMA_MODEL": OLLAMA_MODEL,
        "OLLAMA_THINK": OLLAMA_THINK,
        "OLLAMA_NUM_CTX": OLLAMA_NUM_CTX,
        "OLLAMA_NUM_PREDICT": OLLAMA_NUM_PREDICT,
        "OLLAMA_JSON_SCHEMA": OLLAMA_JSON_SCHEMA,
        "MAX_PROMPT_CHARS": MAX_PROMPT_CHARS,
        "MAX_METHODS_SECTION_CHARS": MAX_METHODS_SECTION_CHARS,
    }


def write_run_manifest(run_id: str, papers: Sequence) -> Path:
    path = RUN_ROOT / f"{run_id}.json"
    path.write_text(json.dumps({
        "run_id": run_id,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "papers": [str(paper) for paper in papers],
        "semantics": _run_semantics(),
        "performance": {"SHARD_SIZE": SHARD_SIZE, "GOLD_WORKERS": GOLD_WORKERS,
                        "CHART_BATCH_SIZE": CHART_BATCH_SIZE,
                        "DOCLING_PAGE_BATCH_SIZE": DOCLING_PAGE_BATCH_SIZE,
                        "CACHE_ENABLED": CACHE_ENABLED},
    }, indent=2, default=str), encoding="utf-8")
    if not REQUIRE_FIGURE_DATA:
        print("NOTE: REQUIRE_FIGURE_DATA=0 — every probed figure is admitted on geometry "
              "alone and no chart is read. That changes what the data means, not only how "
              f"long it takes; it is recorded in {path.name}.")
    return path


def _vision_producer(shards: Sequence, sink, failures: list, run_id: str) -> None:
    """Stage A and B, one shard at a time, handing each finished package straight on.
    Nothing holds more than a shard: at hundreds of papers the list of Bronze manifests
    is itself the memory wall, since each one carries every text item of its paper."""
    try:
        for shard in shards:
            manifests = list(stream_bronze(shard))
            for manifest in manifests:
                set_paper_status(manifest["file_hash"], manifest["paper_slug"],
                                 manifest["source_pdf"], "bronze", run_id=run_id)
            produced = build_silver_shard(manifests)
            gated = {manifest["paper_slug"] for manifest, _ in produced}
            for manifest in manifests:
                if manifest["paper_slug"] not in gated:
                    set_paper_status(manifest["file_hash"], manifest["paper_slug"],
                                     manifest["source_pdf"], "failed",
                                     detail="Silver failed for this paper", run_id=run_id)
            for manifest, package in produced:
                set_paper_status(manifest["file_hash"], manifest["paper_slug"],
                                 manifest["source_pdf"], "silver", run_id=run_id)
                sink.put((manifest, package))
            if RELEASE_MODELS_BETWEEN_STAGES:
                release_chart_model()
                release_docling_converters()
            free_memory()
    except BaseException as exc:
        failures.append(exc)
    finally:
        sink.put(None)


def _gold_task(manifest: dict, package: dict, *, llm_client, ids, run_id: str,
               show_decisions: bool) -> Optional[tuple]:
    """One paper, isolated. A paper that fails is recorded and skipped: without this a
    single timeout or one malformed table discards every paper behind it."""
    slug = manifest["paper_slug"]
    try:
        summary, violations, bundle = run_language_phase(
            manifest, package, llm_client=llm_client, ids=ids,
            show_decisions=show_decisions)
        fingerprint = bundle_fingerprint(bundle)
        set_paper_status(manifest["file_hash"], slug, manifest["source_pdf"], "gold",
                         run_id=run_id)
        return slug, summary, violations, fingerprint
    except Exception as exc:
        detail = f"{type(exc).__name__}: {exc}"
        record_metric("paper_failed", slug, seconds=0.0, error=detail)
        print(f"        {slug} failed in the language phase ({detail}) — the run continues")
        try:
            set_paper_status(manifest["file_hash"], slug, manifest["source_pdf"],
                             "failed", detail=detail, run_id=run_id)
        except Exception as status_exc:
            print(f"        {slug}: its failure could not be recorded ({status_exc})")
        return None


def run_pipeline(pdf_paths, *, show_decisions=None, show_report=True, baseline=None):
    """Every paper from PDF to database, in shards, with Gold overlapping the next
    shard's vision work.

    `show_decisions` defaults to whether the gate frames would be readable: several Gold
    threads printing fifty-line frames at once interleaves them into nothing. Pass True
    or False to decide it yourself.

    `baseline` is a fingerprint file a previous run wrote. Given one, the run ends with a
    diff of what Gold extracted — the only thing that shows an optimisation changed the
    speed and not the answer.
    """
    if show_decisions is None:
        show_decisions = GOLD_WORKERS == 1
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    papers = _resolve_inputs(pdf_paths)
    if not papers:
        raise FileNotFoundError(f"None of {list(pdf_paths)} exist.")

    client = build_ollama_client()
    with SessionLocal() as session:
        ids = seed_vocabulary(session, VOCABULARY)

    if RESUME_COMPLETED:
        done = completed_paper_hashes()
        before = len(papers)
        papers = [path for path in papers if file_hash(path) not in done]
        if before > len(papers):
            print(f"Resuming: {before - len(papers)} of {before} paper(s) already reached "
                  "Gold in an earlier run — set RESUME_COMPLETED=0 to redo them.")
    if not papers:
        print("Every paper is already extracted. Nothing to do.")
        return pd.DataFrame()

    _refuse_slug_collisions(papers)
    manifest_path = write_run_manifest(run_id, papers)
    order = {path.stem: position for position, path in enumerate(papers)}
    shards = _shards(papers, SHARD_SIZE)
    print(f"Run {run_id}: {len(papers)} paper(s) in {len(shards)} shard(s) of "
          f"{SHARD_SIZE}, {GOLD_WORKERS} Gold worker(s) | manifest {manifest_path.name}")

    sink: queue.Queue = queue.Queue(maxsize=VISION_QUEUE_DEPTH)
    failures: list = []
    producer = threading.Thread(target=_vision_producer,
                                args=(shards, sink, failures, run_id),
                                name="vision", daemon=True)
    capacity = GOLD_WORKERS + VISION_QUEUE_DEPTH
    results, in_flight = [], set()

    with stage_timer("run_total", run_id, papers=len(papers)):
        producer.start()
        progress = tqdm(total=len(papers), unit="paper", desc="gold")
        try:
            with ThreadPoolExecutor(max_workers=GOLD_WORKERS,
                                    thread_name_prefix="gold") as pool:
                while True:
                    while len(in_flight) >= capacity:
                        finished, in_flight = wait(in_flight, return_when=FIRST_COMPLETED)
                        results += [future.result() for future in finished]
                        progress.update(len(finished))
                    item = sink.get()
                    if item is None:
                        break
                    in_flight.add(pool.submit(
                        _gold_task, *item, llm_client=client, ids=ids, run_id=run_id,
                        show_decisions=show_decisions))
                for future in as_completed(in_flight):
                    results.append(future.result())
                    progress.update(1)
        finally:
            progress.close()
            producer.join(timeout=5)

    report_model_residency()

    if failures:
        raise RuntimeError("The vision stage failed and the run stopped early") from failures[0]

    extracted = [result for result in results if result is not None]
    if len(results) > len(extracted):
        print(f"{len(results) - len(extracted)} paper(s) failed — see show_paper_status().")
    if not extracted:
        print("No paper reached Gold — see show_paper_status().")
        return pd.DataFrame()

    extracted.sort(key=lambda result: order.get(result[0], len(order)))
    frame = pd.DataFrame([summary for _, summary, _, _ in extracted])
    display(frame)

    fingerprints = {slug: fingerprint for slug, _, _, fingerprint in extracted}
    fingerprint_path = RUN_ROOT / f"{run_id}_bundles.json"
    fingerprint_path.write_text(json.dumps(fingerprints, indent=2, default=str),
                                encoding="utf-8")
    save_timings(RUN_ROOT / f"{run_id}_timings.json")

    VOCABULARY.save_review(REVIEW_PATH)
    print(f"Vocabulary review written to {REVIEW_PATH}")
    if show_report:
        show_normalization_report(pd.concat([violations for _, _, violations, _ in extracted],
                                            ignore_index=True))
    show_stage_totals()
    if baseline:
        show_bundle_diff(baseline, fingerprints)
    else:
        print(f"Gold fingerprints written to {fingerprint_path} — pass it as `baseline=` "
              "on the next run to show that an optimisation changed nothing but the clock.")
    if OLLAMA_KEEP_ALIVE == -1:
        print(f"{OLLAMA_MODEL} stays resident by design (OLLAMA_KEEP_ALIVE=-1), so the "
              "next run does not reload it. Call `build_ollama_client().unload()` to "
              "hand the card back.")
    return frame

In [ ]:
PAPER_INPUTS = ["content/11.pdf", "content/12.pdf", "content/13.pdf"]

reset_timings()
summary_df = run_pipeline(PAPER_INPUTS)

In [ ]:
show_paper_status()